In [ ]:
from google.colab import drive
import os

# Mount Google Drive to Colab
drive.mount('/content/drive')

# Create a local working directory on the Colab disk for faster performance
work_dir = '/content/bdd100k'
os.makedirs(work_dir, exist_ok=True)

In [ ]:
#@title Extract ZIP files from Drive to Colab

# Extract image files from the 'project_BDDK100' directory in Google Drive to the local Colab directory
!unzip -q "/content/drive/MyDrive/project_BDDK100/bdd100k_images_10k.zip" -d {work_dir}

# Extract annotation/label files from the 'project_BDDK100' directory in Google Drive to the local Colab directory
!unzip -q "/content/drive/MyDrive/project_BDDK100/bdd100k_labels.zip" -d {work_dir}

print("Extraction complete!")

In [ ]:
#@title Install Required Libraries

# Install Ultralytics (for YOLO) and Albumentations (for image augmentation) silently
!pip install -q ultralytics albumentations

In [ ]:
#@title Task 1: Baseline Image Selection and YOLOv8 Evaluation

import os
import random
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# Set image directory and load pre-trained YOLOv8 model
images_dir = Path('/content/bdd100k/10k/test')
all_images = list(images_dir.glob('*.jpg'))
model = YOLO('yolov8n.pt')

# Path to store selected image paths on Google Drive for persistence
saved_paths_file = '/content/drive/MyDrive/project_BDDK100/saved_5_images.txt'

# ==============================================================================
# Control Switch:
# True  = Run search pipeline to select, save, and visualize 5 new busy images.
# False = Bypass search and load previously saved image paths.
# ==============================================================================
SEARCH_NEW_IMAGES = False

selected_data = []

if not SEARCH_NEW_IMAGES and os.path.exists(saved_paths_file):
    print("Loading previously selected images...")
    with open(saved_paths_file, 'r') as f:
        saved_paths = f.read().splitlines()

    for img_path_str in saved_paths:
        img_path = Path(img_path_str)
        if img_path.exists():
            results = model.predict(source=str(img_path), verbose=False)
            selected_data.append((img_path, results[0]))

    print(f"Successfully loaded {len(selected_data)} images!")

else:
    print("Searching for 5 new busy images...")
    random.shuffle(all_images)

    for img_path in all_images:
        results = model.predict(source=str(img_path), verbose=False)

        # Filter criterion: Retain images with 3 or more detection bounding boxes
        if len(results[0].boxes) >= 3:
            selected_data.append((img_path, results[0]))

        if len(selected_data) == 5:
            break

    # Save the optimized image paths for consistent cross-evaluation
    print("Saving new image paths...")
    with open(saved_paths_file, 'w') as f:
        for img_path, _ in selected_data:
            f.write(str(img_path) + '\n')

print("Images ready! Plotting visualization grid...")

# --- Visualization Grid ---
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Baseline Check: 5 Busy Images vs. YOLOv8 Detections', fontsize=16, fontweight='bold')

for i, (img_path, result) in enumerate(selected_data):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Plot original raw input image
    axes[0, i].imshow(img_rgb)
    axes[0, i].set_title(f'Original {i+1}')
    axes[0, i].axis('off')

    # Plot image rendered with YOLOv8 detection bounding boxes
    res_plotted = result.plot()
    res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)

    axes[1, i].imshow(res_rgb)
    axes[1, i].set_title(f'YOLOv8 {i+1}\nObjects: {len(result.boxes)}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 1: Low-Light Distortion Intensity Sweep Visualization

import cv2
import matplotlib.pyplot as plt
import albumentations as A

# Define distortion intensity levels (more negative values indicate darker outputs)
brightness_levels = [-0.1, -0.3, -0.5, -0.7, -0.8]

# Initialize 5x6 plot grid (5 images × [1 Original + 5 Distortion levels])
fig, axes = plt.subplots(5, 6, figsize=(24, 15))
fig.suptitle('Distortion Intensity Sweep: Low Light (No Model)', fontsize=20, fontweight='bold')

# Process each of the 5 cached images in selected_data
for row_idx, (img_path, _) in enumerate(selected_data):
    # Load raw image and convert to RGB format
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Plot the original reference image in the first column (index 0)
    axes[row_idx, 0].imshow(img_rgb)
    if row_idx == 0:
        axes[row_idx, 0].set_title("Original", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Apply and display the 5 levels of brightness degradation across subsequent columns
    for col_idx, b_val in enumerate(brightness_levels):
        # Configure the exact brightness degradation limit using Albumentations
        transform = A.Compose([
            A.RandomBrightnessContrast(brightness_limit=(b_val, b_val), contrast_limit=(0, 0), p=1.0)
        ])

        # Execute the distortion transformation pipeline
        dark_img = transform(image=img_rgb)["image"]

        # Map the distorted image to its corresponding cell (col_idx + 1)
        curr_ax = axes[row_idx, col_idx + 1]
        curr_ax.imshow(dark_img)

        # Append sub-titles to the first row only to maintain a clean layout
        if row_idx == 0:
            curr_ax.set_title(f"Level: {b_val}", fontsize=14)
        curr_ax.axis('off')

# Adjust sub-plot spacing and render the visual grid
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 1: YOLOv8 Object Detection under Low-Light Sweep

import cv2
import matplotlib.pyplot as plt
import albumentations as A
from ultralytics import YOLO

# Define 5 low-light distortion intensity levels
brightness_levels = [-0.1, -0.3, -0.5, -0.7, -0.8]

# Load the pre-trained YOLOv8 model
model = YOLO('yolov8n.pt')

# Initialize 5x6 plot grid (5 images × [1 Baseline + 5 Distortion levels])
fig, axes = plt.subplots(5, 6, figsize=(24, 15))
fig.suptitle('YOLOv8 Detections on Distorted Images (Low Light Sweep)', fontsize=20, fontweight='bold')

for row_idx, (img_path, baseline_result) in enumerate(selected_data):
    # --- Column 0: Baseline Reference ---
    res_plotted = baseline_result.plot()
    res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)

    axes[row_idx, 0].imshow(res_rgb)
    baseline_objs = len(baseline_result.boxes)
    if row_idx == 0:
        axes[row_idx, 0].set_title(f"Baseline\nObjects: {baseline_objs}", fontsize=14, fontweight='bold')
    else:
        axes[row_idx, 0].set_title(f"Objects: {baseline_objs}", fontsize=12)
    axes[row_idx, 0].axis('off')

    # Load raw image and convert to RGB format for augmentation pipeline
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # --- Columns 1-5: Apply distortion and run inference ---
    for col_idx, b_val in enumerate(brightness_levels):
        # Generate the degraded low-light image using exact brightness limits
        transform = A.Compose([
            A.RandomBrightnessContrast(brightness_limit=(b_val, b_val), contrast_limit=(0, 0), p=1.0)
        ])
        dark_img_rgb = transform(image=img_rgb)["image"]

        # Convert image back to BGR for YOLOv8 model compatibility
        dark_img_bgr = cv2.cvtColor(dark_img_rgb, cv2.COLOR_RGB2BGR)

        # Execute YOLOv8 object detection inference
        dark_results = model.predict(source=dark_img_bgr, verbose=False)
        dark_res_plotted = dark_results[0].plot()

        # Convert output frame back to RGB for accurate rendering in Matplotlib
        dark_res_final_rgb = cv2.cvtColor(dark_res_plotted, cv2.COLOR_BGR2RGB)

        # Render the localized result cell
        curr_ax = axes[row_idx, col_idx + 1]
        curr_ax.imshow(dark_res_final_rgb)

        num_objs = len(dark_results[0].boxes)
        if row_idx == 0:
            curr_ax.set_title(f"Level: {b_val}\nObjects: {num_objs}", fontsize=14)
        else:
            curr_ax.set_title(f"Objects: {num_objs}", fontsize=12)
        curr_ax.axis('off')

# Optimize sub-plot layout and render grid
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 1: YOLOv8 Recall vs. SNR under Extended Low-Light Degradation

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from ultralytics import YOLO

# Compute Signal-to-Noise Ratio (SNR) between pristine and distorted images
def compute_snr(clean_rgb, dark_rgb):
    clean = clean_rgb.astype(np.float64)
    noise = clean - dark_rgb.astype(np.float64)

    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)

    if noise_power == 0:
        return np.inf
    return 10.0 * np.log10(signal_power / noise_power)

# Extended distortion levels (incorporating extreme darkness at -0.9 and -1.0)
brightness_levels = [-0.1, -0.3, -0.5, -0.7, -0.8, -0.9, -1.0]

mean_snr_vals = []
mean_recall_vals = []

print("Calculating SNR and Recall for extended intensity levels...")

# Execute inference sweep across all defined brightness levels
for b_val in brightness_levels:
    # Configure Albumentations to apply the exact brightness offset
    transform = A.Compose([
        A.RandomBrightnessContrast(brightness_limit=(b_val, b_val), contrast_limit=(0, 0), p=1.0)
    ])

    level_snr = []
    level_recall = []

    for img_path, baseline_result in selected_data:
        # Load raw image and prepare RGB format
        img = cv2.imread(str(img_path))
        clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Apply low-light distortion
        dark_rgb = transform(image=clean_rgb)["image"]

        # Calculate and log the empirical SNR
        snr = compute_snr(clean_rgb, dark_rgb)
        level_snr.append(snr)

        # Convert back to BGR for YOLOv8 model inference
        dark_bgr = cv2.cvtColor(dark_rgb, cv2.COLOR_RGB2BGR)
        dark_results = model.predict(source=dark_bgr, verbose=False)

        # Quantify detection recall relative to the clean baseline
        baseline_objs = len(baseline_result.boxes)
        dark_objs = len(dark_results[0].boxes)

        if baseline_objs > 0:
            recall = min(1.0, dark_objs / baseline_objs)
        else:
            recall = 0.0

        level_recall.append(recall)

    mean_snr_vals.append(np.mean(level_snr))
    mean_recall_vals.append(np.mean(level_recall))

print("Plotting updated results...")

# Initialize plotting environment
plt.figure(figsize=(10, 6))
plt.plot(mean_snr_vals, mean_recall_vals, marker='o', linestyle='-', linewidth=2.5, markersize=8, color='#1f77b4')

# Annotate specific brightness factor levels on the plotted curve
for i, b_val in enumerate(brightness_levels):
    plt.annotate(f"b={b_val}",
                 (mean_snr_vals[i], mean_recall_vals[i]),
                 textcoords="offset points",
                 xytext=(0,10),
                 ha='center', fontsize=10, fontweight='bold')

# Invert X-axis (decreasing SNR indicates heavier distortion/darkness)
plt.gca().invert_xaxis()
plt.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, label='Clean baseline (1.00)')

# Add project credits and formal axis labels
plt.title("YOLO Detection Recall vs SNR - Low-Light Degradation Sweep\nProject by: Ayelet & Noa", fontsize=14, fontweight='bold', pad=15)
plt.xlabel(r"SNR (dB) $\rightarrow$ darker", fontsize=12)
plt.ylabel("Mean detection recall vs clean", fontsize=12)

# Set Y-axis limits to accurately accommodate edge values (-0.2 to 1.1)
plt.ylim(-0.2, 1.1)

plt.grid(True, alpha=0.4, linestyle='--')
plt.legend(loc='lower left', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 1: Low-Light Enhancement Evaluation (Gamma + CLAHE)

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from ultralytics import YOLO

# Restore low-light images using a combined Gamma and CLAHE approach
def restore_lowlight(img_rgb):
    # 1. Global Gamma Correction to boost overall brightness
    gamma = 0.35
    lut = (np.arange(256) / 255.0) ** gamma * 255
    lut = np.clip(lut, 0, 255).astype(np.uint8)
    img_gamma = cv2.LUT(img_rgb, lut)

    # 2. Local Contrast Enhancement using CLAHE on the Lightness (L) channel
    lab = cv2.cvtColor(img_gamma, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=6.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    out = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)

    return out

# Define specific low-light distortion intensity
b_val = -0.3
transform = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=(b_val, b_val), contrast_limit=(0, 0), p=1.0)
])

# Initialize a 3x5 subplot grid: [Clean, Distorted, Enhanced]
fig, axes = plt.subplots(3, 5, figsize=(22, 12))
fig.suptitle(f'Enhancement Check: Low Light (b={b_val}) -> Gamma + CLAHE', fontsize=20, fontweight='bold')

print(f"Applying enhancements and running YOLOv8 for b={b_val}...")

for i, (img_path, baseline_result) in enumerate(selected_data):
    # --- Row 1: Clean Baseline ---
    img = cv2.imread(str(img_path))
    clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Plot original clean image with baseline detections
    res_clean = baseline_result.plot()
    axes[0, i].imshow(cv2.cvtColor(res_clean, cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(f"Clean (Baseline)\nObjects: {len(baseline_result.boxes)}", fontsize=13, fontweight='bold')
    axes[0, i].axis('off')

    # --- Row 2: Distorted (Low Light) ---
    # Apply darkening transform
    dark_rgb = transform(image=clean_rgb)["image"]
    dark_bgr = cv2.cvtColor(dark_rgb, cv2.COLOR_RGB2BGR)

    # Run YOLOv8 inference on distorted image
    res_dark = model.predict(source=dark_bgr, verbose=False)[0]
    axes[1, i].imshow(cv2.cvtColor(res_dark.plot(), cv2.COLOR_BGR2RGB))
    axes[1, i].set_title(f"Distorted (b={b_val})\nObjects: {len(res_dark.boxes)}", fontsize=13, color='red')
    axes[1, i].axis('off')

    # --- Row 3: Enhanced (Restored) ---
    # Apply enhancement filter pipeline
    enh_rgb = restore_lowlight(dark_rgb)
    enh_bgr = cv2.cvtColor(enh_rgb, cv2.COLOR_RGB2BGR)

    # Run YOLOv8 inference on restored image
    res_enh = model.predict(source=enh_bgr, verbose=False)[0]
    axes[2, i].imshow(cv2.cvtColor(res_enh.plot(), cv2.COLOR_BGR2RGB))
    axes[2, i].set_title(f"Enhanced (CLAHE)\nObjects: {len(res_enh.boxes)}", fontsize=13, color='green')
    axes[2, i].axis('off')

# Optimize layout and render grid
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 1: YOLO Recall vs. SNR with Adaptive Low-Light Enhancement

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from ultralytics import YOLO

# Compute Signal-to-Noise Ratio (SNR) between clean and degraded images
def compute_snr(clean_rgb, dark_rgb):
    clean = clean_rgb.astype(np.float64)
    noise = clean - dark_rgb.astype(np.float64)
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)

    if noise_power == 0:
        return np.inf
    return 10.0 * np.log10(signal_power / noise_power)

# Adaptive low-light restoration (Dynamic Gamma + CLAHE based on distortion severity)
def restore_lowlight_adaptive(img_rgb, b_val):
    abs_b = abs(b_val)

    # Dynamically scale Gamma: darker images get stronger (lower) gamma values
    gamma = 0.95 - 0.75 * (abs_b - 0.1) / 0.9
    gamma = np.clip(gamma, 0.2, 0.95)

    # Dynamically scale CLAHE clip limit: darker images get stronger contrast limits
    clip_limit = 1.2 + 6.8 * (abs_b - 0.1) / 0.9
    clip_limit = np.clip(clip_limit, 1.2, 8.0)

    # Apply Gamma Correction
    lut = (np.arange(256) / 255.0) ** gamma * 255
    lut = np.clip(lut, 0, 255).astype(np.uint8)
    img_gamma = cv2.LUT(img_rgb, lut)

    # Apply CLAHE on the Lightness (L) channel
    lab = cv2.cvtColor(img_gamma, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
    l = clahe.apply(l)
    out = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)

    return out

# Define brightness degradation levels (including extreme darkness at -0.9, -1.0)
brightness_levels = [-0.1, -0.3, -0.5, -0.7, -0.8, -0.9, -1.0]
mean_snr_vals = []
mean_recall_dark = []
mean_recall_enh = []

print("Running Adaptive Enhancement Sweep...")

# Execute evaluation sweep across all defined intensity levels
for b_val in brightness_levels:
    transform = A.Compose([
        A.RandomBrightnessContrast(brightness_limit=(b_val, b_val), contrast_limit=(0, 0), p=1.0)
    ])

    level_snr = []
    level_recall_dark = []
    level_recall_enh = []

    for img_path, baseline_result in selected_data:
        # Load and prepare pristine image
        img = cv2.imread(str(img_path))
        clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Apply low-light distortion and calculate empirical SNR
        dark_rgb = transform(image=clean_rgb)["image"]
        snr = compute_snr(clean_rgb, dark_rgb)
        level_snr.append(snr)

        baseline_objs = len(baseline_result.boxes)

        # 1. Run inference on distorted (dark) image
        dark_bgr = cv2.cvtColor(dark_rgb, cv2.COLOR_RGB2BGR)
        dark_results = model.predict(source=dark_bgr, verbose=False)
        dark_objs = len(dark_results[0].boxes)
        recall_dark = min(1.0, dark_objs / baseline_objs) if baseline_objs > 0 else 0.0
        level_recall_dark.append(recall_dark)

        # 2. Run inference on adaptively enhanced image
        enh_rgb = restore_lowlight_adaptive(dark_rgb, b_val)
        enh_bgr = cv2.cvtColor(enh_rgb, cv2.COLOR_RGB2BGR)
        enh_results = model.predict(source=enh_bgr, verbose=False)
        enh_objs = len(enh_results[0].boxes)
        recall_enh = min(1.0, enh_objs / baseline_objs) if baseline_objs > 0 else 0.0
        level_recall_enh.append(recall_enh)

    # Aggregate mean metrics for the current intensity level
    mean_snr_vals.append(np.mean(level_snr))
    mean_recall_dark.append(np.mean(level_recall_dark))
    mean_recall_enh.append(np.mean(level_recall_enh))

# Plot comparative results (Distorted vs. Enhanced)
plt.figure(figsize=(12, 7))
plt.plot(mean_snr_vals, mean_recall_dark, marker='o', linestyle='-', linewidth=2.5, color='#d62728', label='Distorted (Low Light)')
plt.plot(mean_snr_vals, mean_recall_enh, marker='s', linestyle='-', linewidth=2.5, color='#2ca02c', label='Enhanced (Adaptive Gamma + CLAHE)')

# Annotate specific brightness factors on the enhanced curve
for i, b_val in enumerate(brightness_levels):
    plt.annotate(f"b={b_val}", (mean_snr_vals[i], mean_recall_enh[i]), textcoords="offset points", xytext=(0,12), ha='center', fontsize=9, fontweight='bold')

# Invert X-axis (decreasing SNR indicates heavier distortion)
plt.gca().invert_xaxis()

# Ideal clean baseline reference line
plt.axhline(y=1.0, color='blue', linestyle='--', label='Clean Baseline (1.00)')

plt.title("YOLO Detection Recall vs SNR (Adaptive Enhancement)\nProject by: Ayelet & Noa", fontsize=14, fontweight='bold')
plt.xlabel(r"SNR (dB) $\rightarrow$ darker", fontsize=12)
plt.ylabel("Mean Detection Recall", fontsize=12)
plt.ylim(-0.05, 1.1)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend()
plt.show()

In [ ]:
!rm -rf ~/.cache/matplotlib

In [ ]:
#@title Task 1: Large-Scale Dataset Generation & Fine-Tuning (Low Learning Rate)

import os
import yaml
import random
import cv2
import numpy as np
import torch
import albumentations as A
from pathlib import Path
from ultralytics import YOLO

# Define and create directories for the ultimate fine-tuning dataset
ft_massive_dir = Path('/content/bdd100k_ft_ultimate')
(ft_massive_dir / 'images' / 'train').mkdir(parents=True, exist_ok=True)
(ft_massive_dir / 'labels' / 'train').mkdir(parents=True, exist_ok=True)

# Helper function: Convert bounding boxes from xyxy to YOLO format (normalized cx, cy, w, h)
def save_yolo_label(txt_path, boxes_xyxy, cls_ids, w, h):
    lines = []
    for (x1, y1, x2, y2), c in zip(boxes_xyxy, cls_ids):
        cx = ((x1 + x2) / 2) / w
        cy = ((y1 + y2) / 2) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h
        lines.append(f"{int(c)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    txt_path.write_text("\n".join(lines))

print("Step 1: Creating an ULTIMATE Mixed Dataset (1000 images: 50% Clean, 50% Dark)...")

# Filter out test images to prevent data leakage and allocate 1000 images for training
test_image_paths = [str(p) for p, _ in selected_data]
train_images = [p for p in all_images if str(p) not in test_image_paths][:1000]

valid_images_count = 0
clean_count = 0
dark_count = 0

for i, img_path in enumerate(train_images):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    # Extract pseudo-labels using a lower confidence threshold (0.20) to capture smaller objects
    r = model.predict(img, conf=0.20, iou=0.5, verbose=False)[0]
    boxes = r.boxes

    # Skip images with no detections to avoid training on empty backgrounds
    if boxes is None or len(boxes) == 0:
        continue

    xyxy = boxes.xyxy.cpu().numpy()
    cls = boxes.cls.cpu().numpy()

    # Apply data augmentation: Keep 50% of images pristine, apply severe darkness to the other 50%
    if random.random() < 0.5:
        final_img_bgr = img
        clean_count += 1
    else:
        b_val = random.uniform(-0.3, -0.8)
        transform = A.Compose([
            A.RandomBrightnessContrast(brightness_limit=(b_val, b_val), contrast_limit=(0, 0), p=1.0)
        ])
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        dark_img_rgb = transform(image=img_rgb)["image"]
        final_img_bgr = cv2.cvtColor(dark_img_rgb, cv2.COLOR_RGB2BGR)
        dark_count += 1

    # Save processed images and their corresponding YOLO label files
    img_out_path = ft_massive_dir / 'images' / 'train' / f"im_{i}.jpg"
    lbl_out_path = ft_massive_dir / 'labels' / 'train' / f"im_{i}.txt"

    cv2.imwrite(str(img_out_path), final_img_bgr)
    save_yolo_label(lbl_out_path, xyxy, cls, w, h)
    valid_images_count += 1

print(f"Dataset Ready! Total: {valid_images_count} images ({clean_count} Clean, {dark_count} Dark).")

# Generate YOLO YAML configuration file for the training process
yaml_data = {
    'train': str(ft_massive_dir / 'images' / 'train'),
    'val': str(ft_massive_dir / 'images' / 'train'),
    'nc': len(model.names),
    'names': model.names
}
with open(ft_massive_dir / 'data.yaml', 'w') as f:
    yaml.dump(yaml_data, f)

print("Step 2: Training the Ultimate Model (25 Epochs) with lowered Learning Rate...")
current_device = 0 if torch.cuda.is_available() else 'cpu'

# Initialize a fresh YOLOv8 nano model for fine-tuning
ft_model_ultimate = YOLO("yolov8n.pt")

# Train the model: Reduced learning rate (lr0=0.001) ensures stable convergence and prevents "forgetting"
results = ft_model_ultimate.train(
    data=str(ft_massive_dir / 'data.yaml'),
    imgsz=640,
    epochs=25,
    batch=8,
    device=current_device,
    verbose=False,
    lr0=0.001
)

# Load the best weights generated during the training phase
best_path = Path(ft_model_ultimate.trainer.best) if hasattr(ft_model_ultimate, "trainer") else Path('/content/runs/detect/train/weights/best.pt')
ultimate_model = YOLO(str(best_path))

print("Ultimate Fine-Tuning Complete! Time to test the new gentle-learning model.")

In [ ]:
#@title Task 1: Final Evaluation Sweep - Original vs. Enhanced vs. Fine-Tuned YOLOv8

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from ultralytics import YOLO

# --- Helper Functions ---
def compute_snr(clean_rgb, dark_rgb):
    # Calculate Signal-to-Noise Ratio (SNR) to quantify degradation severity
    clean = clean_rgb.astype(np.float64)
    noise = clean - dark_rgb.astype(np.float64)
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(signal_power / noise_power)

def restore_lowlight_adaptive(img_rgb, b_val):
    # Apply adaptive Gamma correction and CLAHE based on distortion intensity
    abs_b = abs(b_val)
    gamma = np.clip(0.95 - 0.75 * (abs_b - 0.1) / 0.9, 0.2, 0.95)
    clip_limit = np.clip(1.2 + 6.8 * (abs_b - 0.1) / 0.9, 1.2, 8.0)

    lut = np.clip((np.arange(256) / 255.0) ** gamma * 255, 0, 255).astype(np.uint8)
    img_gamma = cv2.LUT(img_rgb, lut)

    lab = cv2.cvtColor(img_gamma, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
    out = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2RGB)
    return out

# --- Model Initialization ---
# CRITICAL FIX: Load a fresh, unmodified YOLOv8 instance to ensure a fair baseline comparison
fresh_original_model = YOLO("yolov8n.pt")

# Define the sweep range for low-light distortion levels
brightness_levels = [-0.1, -0.3, -0.5, -0.7, -0.8, -0.9, -1.0]
m_snr, m_dark, m_enh, m_ft = [], [], [], []

print("Running Evaluation Sweep with fresh models...")

# --- Evaluation Pipeline ---
for b_val in brightness_levels:
    # Initialize the specific darkness transformation for this sweep iteration
    transform = A.Compose([A.RandomBrightnessContrast(brightness_limit=(b_val, b_val), contrast_limit=(0, 0), p=1.0)])
    l_snr, l_dark, l_enh, l_ft = [], [], [], []

    for img_path, baseline_result in selected_data:
        # Prepare clean base image
        img = cv2.imread(str(img_path))
        clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        baseline_objs = len(baseline_result.boxes)

        # Generate distorted image and calculate empirical SNR
        dark_rgb = transform(image=clean_rgb)["image"]
        dark_bgr = cv2.cvtColor(dark_rgb, cv2.COLOR_RGB2BGR)
        l_snr.append(compute_snr(clean_rgb, dark_rgb))

        # 1. Distorted: Inference using the pristine original model on dark images
        dark_res = fresh_original_model.predict(source=dark_bgr, verbose=False)
        l_dark.append(min(1.0, len(dark_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0)

        # 2. Enhanced: Inference using the pristine original model on CLAHE-restored images
        enh_bgr = cv2.cvtColor(restore_lowlight_adaptive(dark_rgb, b_val), cv2.COLOR_RGB2BGR)
        enh_res = fresh_original_model.predict(source=enh_bgr, verbose=False)
        l_enh.append(min(1.0, len(enh_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0)

        # 3. Fine-Tuned: Inference using your custom trained 'Ultimate' model on dark images
        ft_res = ultimate_model.predict(source=dark_bgr, verbose=False)
        l_ft.append(min(1.0, len(ft_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0)

    # Aggregate mean performance metrics for the current distortion level
    m_snr.append(np.mean(l_snr))
    m_dark.append(np.mean(l_dark))
    m_enh.append(np.mean(l_enh))
    m_ft.append(np.mean(l_ft))

# --- Plotting the Results ---
plt.figure(figsize=(12, 7))
plt.plot(m_snr, m_dark, marker='o', linestyle='-', linewidth=2, color='#d62728', label='Distorted (Original YOLO)')
plt.plot(m_snr, m_enh, marker='s', linestyle='-', linewidth=2, color='#2ca02c', label='Enhanced (Adaptive CLAHE)')
plt.plot(m_snr, m_ft, marker='^', linestyle='-', linewidth=3, color='#9467bd', label='Ultimate Mixed YOLO (Our Best!)')

# Annotate the Fine-Tuned curve with the exact brightness limits
for i, b_val in enumerate(brightness_levels):
    plt.annotate(f"b={b_val}", (m_snr[i], m_ft[i]), textcoords="offset points", xytext=(0,12), ha='center', fontsize=9, fontweight='bold', color='#1f77b4')

plt.gca().invert_xaxis()
plt.axhline(y=1.0, color='blue', linestyle='--', label='Clean Baseline (1.00)')
plt.title("YOLO Detection Recall vs SNR\nDistorted vs. Enhanced vs. ULTIMATE Fine-Tuned | Project by: Ayelet & Noa", fontsize=14, fontweight='bold')
plt.xlabel(r"SNR (dB) $\rightarrow$ darker", fontsize=12)
plt.ylabel("Mean Detection Recall vs Clean", fontsize=12)
plt.ylim(-0.05, 1.1)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend()
plt.show()

In [ ]:
#@title Task 1: Grand Finale - YOLOv8 Performance Bar Chart Comparison

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from ultralytics import YOLO

# --- Final Evaluation Setup ---
b_val_hard = -0.5  # Select a severe darkness level as the test case
fresh_original_model = YOLO("yolov8n.pt") # Load a fresh base model to ensure a fair evaluation

# Classical image restoration function (Adaptive Gamma + CLAHE)
def restore_lowlight_adaptive(img_rgb, b_val):
    abs_b = abs(b_val)
    gamma = np.clip(0.95 - 0.75 * (abs_b - 0.1) / 0.9, 0.2, 0.95)
    clip_limit = np.clip(1.2 + 6.8 * (abs_b - 0.1) / 0.9, 1.2, 8.0)
    lut = np.clip((np.arange(256) / 255.0) ** gamma * 255, 0, 255).astype(np.uint8)
    img_gamma = cv2.LUT(img_rgb, lut)
    lab = cv2.cvtColor(img_gamma, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
    out = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2RGB)
    return out

# Arrays to store recall metrics
recall_distorted = []
recall_enhanced = []
recall_finetuned = []

print(f"Running Final Stand-off at darkness level b={b_val_hard}...")

transform = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=(b_val_hard, b_val_hard), contrast_limit=(0, 0), p=1.0)
])

# Iterate through the 5 control images
for img_path, baseline_result in selected_data:
    img = cv2.imread(str(img_path))
    clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    baseline_objs = len(baseline_result.boxes)

    # Generate the distorted low-light image
    dark_rgb = transform(image=clean_rgb)["image"]
    dark_bgr = cv2.cvtColor(dark_rgb, cv2.COLOR_RGB2BGR)

    # 1. Original model on Distorted image
    dark_res = fresh_original_model.predict(source=dark_bgr, verbose=False)
    r_dist = min(1.0, len(dark_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
    recall_distorted.append(r_dist)

    # 2. Original model on Enhanced image (CLAHE applied)
    enh_rgb = restore_lowlight_adaptive(dark_rgb, b_val_hard)
    enh_bgr = cv2.cvtColor(enh_rgb, cv2.COLOR_RGB2BGR)
    enh_res = fresh_original_model.predict(source=enh_bgr, verbose=False)
    r_enh = min(1.0, len(enh_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
    recall_enhanced.append(r_enh)

    # 3. Fine-Tuned 'Ultimate' model on Distorted image
    ft_res = ultimate_model.predict(source=dark_bgr, verbose=False)
    r_ft = min(1.0, len(ft_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
    recall_finetuned.append(r_ft)

# Calculate mean recall for the bar chart
mean_dist = np.mean(recall_distorted)
mean_enh = np.mean(recall_enhanced)
mean_ft = np.mean(recall_finetuned)

print("Plotting the Grand Finale Bar Chart...")

# --- Generate Bar Chart ---
labels = ['Original YOLOv8\n(Distorted Image)', 'Original YOLOv8\n(Enhanced Image)', 'Fine-Tuned YOLOv8\n(Distorted Image)']
values = [mean_dist, mean_enh, mean_ft]
colors = ['#d62728', '#2ca02c', '#9467bd'] # Red, Green, Purple (matching previous line plots)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(labels, values, color=colors, width=0.5, edgecolor='black', linewidth=1.5)

# Add reference line for the clean baseline (1.00)
ax.axhline(y=1.0, color='blue', linestyle='--', linewidth=2, label='Clean Baseline (1.00)')

# Annotate the exact value above each bar
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=14, fontweight='bold')

# Presentation-ready formatting
ax.set_ylabel('Mean Detection Recall vs Clean', fontsize=12, fontweight='bold')
ax.set_title(f'YOLOv8 Performance Under Low Light (b={b_val_hard})\nProject by: Ayelet & Noa', fontsize=16, fontweight='bold', pad=20)
ax.set_ylim(0, 1.15)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(loc='upper left', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 1: Gaussian Noise Distortion Intensity Sweep Visualization

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Custom function to generate Gaussian noise with complete mathematical control
def add_gaussian_noise(image, std):
    # Convert to float32 to prevent underflow/overflow clipping during calculation
    img_float = image.astype(np.float32)

    # Generate Gaussian noise matrix with mean=0 and specified standard deviation (std)
    noise = np.random.normal(0, std, img_float.shape)

    # Superimpose noise matrix onto the original image
    noisy_image = img_float + noise

    # Clip pixel values to valid [0, 255] range and cast back to uint8
    noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)
    return noisy_image

# Define 5 gradual intensity levels using standard deviation (STD)
# Range scales from light degradation to severe noise corruption
noise_stds = [10, 30, 60, 100, 150]

# Initialize 5x6 grid layout (5 reference images × [1 Original + 5 Noise levels])
fig, axes = plt.subplots(5, 6, figsize=(24, 15))
fig.suptitle('Distortion Intensity Sweep: Gaussian Noise (No Model)', fontsize=20, fontweight='bold')

# Iterate through the cached control images in selected_data
for row_idx, (img_path, _) in enumerate(selected_data):
    # Load original image and convert to RGB format
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Plot original reference frame in the first column (index 0)
    axes[row_idx, 0].imshow(img_rgb)
    if row_idx == 0:
        axes[row_idx, 0].set_title("Original", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Loop across the 5 defined noise intensity values
    for col_idx, std_val in enumerate(noise_stds, start=1):
        # Apply Gaussian noise distortion via the custom pipeline
        noisy_img_rgb = add_gaussian_noise(img_rgb, std_val)

        # Render the distorted frame
        axes[row_idx, col_idx].imshow(noisy_img_rgb)

        # Append dynamic sub-titles only to the first row to preserve layout neatness
        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"Noise STD: {std_val}", fontsize=14)
        axes[row_idx, col_idx].axis('off')

# Optimize padding parameters and render grid plot
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 1: YOLOv8 Detections Under Gaussian Noise Degradation Sweep

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Custom function to generate Gaussian noise with complete mathematical control
def add_gaussian_noise(image, std):
    # Convert to float32 to prevent underflow/overflow during calculations
    img_float = image.astype(np.float32)

    # Generate Gaussian noise matrix (mean=0, specified std)
    noise = np.random.normal(0, std, img_float.shape)

    # Superimpose noise onto the original image
    noisy_image = img_float + noise

    # Clip values to valid pixel range [0, 255] and cast to uint8
    noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)
    return noisy_image

# Load a fresh YOLOv8 model to ensure an unbiased baseline evaluation
fresh_original_model = YOLO("yolov8n.pt")

# Define the same robust noise standard deviation (STD) levels used previously
noise_stds = [10, 30, 60, 100, 150]

# Initialize a 5x6 plot grid (5 images × [1 Original + 5 Noise levels])
fig, axes = plt.subplots(5, 6, figsize=(24, 15))
fig.suptitle('YOLOv8 Detections Under Gaussian Noise', fontsize=20, fontweight='bold')

print("Running YOLO on noisy images... This will take a few seconds :)")

# Iterate through the 5 cached control images
for row_idx, (img_path, _) in enumerate(selected_data):
    # Load original image and convert to RGB
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # --- 1. Run inference on the original (clean) image ---
    res_baseline = fresh_original_model.predict(source=img_rgb, verbose=False)[0]
    res_baseline_plotted = res_baseline.plot() # Render detection bounding boxes

    axes[row_idx, 0].imshow(res_baseline_plotted)
    if row_idx == 0:
        axes[row_idx, 0].set_title(f"Original\nObjects: {len(res_baseline.boxes)}", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # --- 2. Run inference on the noisy images ---
    for col_idx, std_val in enumerate(noise_stds, start=1):
        # Generate the noisy image
        noisy_img_rgb = add_gaussian_noise(img_rgb, std_val)

        # Execute YOLOv8 on the noisy image
        res_noisy = fresh_original_model.predict(source=noisy_img_rgb, verbose=False)[0]
        res_noisy_plotted = res_noisy.plot() # Render bounding boxes on the noisy image

        axes[row_idx, col_idx].imshow(res_noisy_plotted)
        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"Noise STD: {std_val}\nObjects: {len(res_noisy.boxes)}", fontsize=14)
        axes[row_idx, col_idx].axis('off')

# Optimize sub-plot layout and display the visual grid
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 1: YOLOv8 Recall vs. SNR under Gaussian Noise

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Custom function to inject mathematically controlled Gaussian noise
def add_gaussian_noise(image, std):
    # Convert to float32 to prevent underflow/overflow clipping artifacts
    img_float = image.astype(np.float32)
    # Generate Gaussian noise matrix (mean=0, target std)
    noise = np.random.normal(0, std, img_float.shape)
    noisy_image = img_float + noise
    # Clip to valid [0, 255] pixel range and cast back to uint8
    noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)
    return noisy_image

# 2. Signal-to-Noise Ratio (SNR) calculation
def compute_snr(clean_rgb, noisy_rgb):
    clean = clean_rgb.astype(np.float64)
    noise = clean - noisy_rgb.astype(np.float64)
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(signal_power / noise_power)

# Load a fresh YOLOv8 model for an unbiased baseline
fresh_original_model = YOLO("yolov8n.pt")

# Define standard deviation (STD) levels for the noise sweep
noise_stds = [10, 30, 60, 100, 150]

# Arrays to store aggregated metrics per noise level
m_snr = []
m_recall_distorted = []

print("Calculating SNR and Detections across noise levels...")

for std_val in noise_stds:
    l_snr = []
    l_recall = []

    # Iterate through the 5 cached control images
    for img_path, baseline_result in selected_data:
        img = cv2.imread(str(img_path))
        clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        baseline_objs = len(baseline_result.boxes)

        # Generate the noisy image
        noisy_rgb = add_gaussian_noise(clean_rgb, std_val)
        noisy_bgr = cv2.cvtColor(noisy_rgb, cv2.COLOR_RGB2BGR) # YOLO expects BGR format

        # Calculate empirical SNR between clean and noisy images
        l_snr.append(compute_snr(clean_rgb, noisy_rgb))

        # Execute YOLOv8 inference on the distorted image
        noisy_res = fresh_original_model.predict(source=noisy_bgr, verbose=False)

        # Calculate detection recall relative to the clean baseline
        r_dist = min(1.0, len(noisy_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
        l_recall.append(r_dist)

    # Aggregate mean metrics for the current noise level
    m_snr.append(np.mean(l_snr))
    m_recall_distorted.append(np.mean(l_recall))

# --- Plot the Evaluation Curve ---
plt.figure(figsize=(10, 6))
plt.plot(m_snr, m_recall_distorted, marker='o', linestyle='-', linewidth=2.5, color='#d62728', label='Distorted (Original YOLO)')

# Annotate the specific STD value above each data point
for i, std_val in enumerate(noise_stds):
    plt.annotate(f"STD={std_val}", (m_snr[i], m_recall_distorted[i]),
                 textcoords="offset points", xytext=(0,12), ha='center', fontsize=10, fontweight='bold', color='#1f77b4')

# Invert X-axis (decreasing SNR indicates heavier distortion/noise)
plt.gca().invert_xaxis()

plt.axhline(y=1.0, color='blue', linestyle='--', label='Clean Baseline (1.00)')
plt.title("YOLO Detection Recall vs SNR\nGaussian Noise Distortion | Project by: Ayelet & Noa", fontsize=14, fontweight='bold', pad=15)
plt.xlabel(r"SNR (dB) $\rightarrow$ More Noise", fontsize=12, fontweight='bold')
plt.ylabel("Mean Detection Recall vs Clean", fontsize=12, fontweight='bold')
plt.ylim(-0.05, 1.15)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#@title Task 1: Gaussian Noise Evaluation (Distorted vs. Enhanced)

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Custom function to mathematically inject Gaussian noise
def add_gaussian_noise(image, std):
    img_float = image.astype(np.float32)
    noise = np.random.normal(0, std, img_float.shape)
    noisy_image = img_float + noise
    noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)
    return noisy_image

# 2. Compute Signal-to-Noise Ratio (SNR)
def compute_snr(clean_rgb, noisy_rgb):
    clean = clean_rgb.astype(np.float64)
    noise = clean - noisy_rgb.astype(np.float64)
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(signal_power / noise_power)

# Load a fresh YOLOv8 model for an unbiased baseline
fresh_original_model = YOLO("yolov8n.pt")

# Define standard deviation (STD) levels for the noise sweep
noise_stds = [10, 30, 60, 100, 150]

# Arrays to store aggregated metrics per noise level
m_snr = []
m_recall_distorted = []
m_recall_enhanced = [] # Added array for the enhanced images

print("Calculating SNR, applying distortion, running bilateral filter, and evaluating... (Please wait)")

for std_val in noise_stds:
    l_snr = []
    l_recall = []
    l_recall_enh = [] # Internal list for the enhanced images

    # Iterate through the 5 cached control images
    for img_path, baseline_result in selected_data:
        img = cv2.imread(str(img_path))
        clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        baseline_objs = len(baseline_result.boxes)

        # Generate the noisy image
        noisy_rgb = add_gaussian_noise(clean_rgb, std_val)
        noisy_bgr = cv2.cvtColor(noisy_rgb, cv2.COLOR_RGB2BGR) # YOLO expects BGR format

        # Calculate empirical SNR between clean and noisy images
        l_snr.append(compute_snr(clean_rgb, noisy_rgb))

        # --- Test 1: Distorted Image ---
        noisy_res = fresh_original_model.predict(source=noisy_bgr, verbose=False)

        # Calculate detection recall relative to the clean baseline
        r_dist = min(1.0, len(noisy_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
        l_recall.append(r_dist)

        # --- Test 2: Classically Enhanced Image ---
        # Apply Bilateral Filter to reduce noise while preserving edges
        enh_bgr = cv2.bilateralFilter(noisy_bgr, d=9, sigmaColor=75, sigmaSpace=75)
        enh_res = fresh_original_model.predict(source=enh_bgr, verbose=False)

        r_enh = min(1.0, len(enh_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
        l_recall_enh.append(r_enh)

    # Aggregate mean metrics for the current noise level
    m_snr.append(np.mean(l_snr))
    m_recall_distorted.append(np.mean(l_recall))
    m_recall_enhanced.append(np.mean(l_recall_enh))

# --- Plot the Combined Evaluation Curve ---
plt.figure(figsize=(10, 6))

# Red line - Distorted
plt.plot(m_snr, m_recall_distorted, marker='o', linestyle='-', linewidth=2.5, color='#d62728', label='Distorted (Original YOLO)')
# Green line - Enhanced with Bilateral Filter
plt.plot(m_snr, m_recall_enhanced, marker='s', linestyle='-', linewidth=2.5, color='#2ca02c', label='Enhanced (Bilateral Filter)')

# Annotate the specific STD value above each data point
for i, std_val in enumerate(noise_stds):
    # Place annotation above the higher of the two points to avoid overlap
    plt.annotate(f"STD={std_val}", (m_snr[i], max(m_recall_distorted[i], m_recall_enhanced[i])),
                 textcoords="offset points", xytext=(0,12), ha='center', fontsize=10, fontweight='bold', color='#1f77b4')

# Invert X-axis (decreasing SNR indicates heavier distortion/noise)
plt.gca().invert_xaxis()

plt.axhline(y=1.0, color='blue', linestyle='--', label='Clean Baseline (1.00)')
plt.title("YOLO Detection Recall vs SNR\nGaussian Noise: Distorted vs Enhanced | Project by: Ayelet & Noa", fontsize=14, fontweight='bold', pad=15)
plt.xlabel(r"SNR (dB) $\rightarrow$ More Noise", fontsize=12, fontweight='bold')
plt.ylabel("Mean Detection Recall vs Clean", fontsize=12, fontweight='bold')
plt.ylim(-0.05, 1.15)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#@title Task 1: Dataset Generation & Fine-Tuning for Gaussian Noise Resilience

import os
import yaml
import random
import cv2
import numpy as np
import torch
import albumentations as A
from pathlib import Path
from ultralytics import YOLO

# Define workspace directories for Gaussian noise fine-tuning
ft_noise_dir = Path('/content/bdd100k_ft_noise')
(ft_noise_dir / 'images' / 'train').mkdir(parents=True, exist_ok=True)
(ft_noise_dir / 'labels' / 'train').mkdir(parents=True, exist_ok=True)

# Helper function to export bounding boxes in YOLO format (normalized cx, cy, w, h)
def save_yolo_label(txt_path, boxes_xyxy, cls_ids, w, h):
    lines = []
    for (x1, y1, x2, y2), c in zip(boxes_xyxy, cls_ids):
        cx = ((x1 + x2) / 2) / w
        cy = ((y1 + y2) / 2) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h
        lines.append(f"{int(c)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    txt_path.write_text("\n".join(lines))

print("Step 1: Creating a Mixed Dataset (1000 images: 50% Clean, 50% Noisy)...")

# Exclude control/test images to avoid data leakage
test_image_paths = [str(p) for p, _ in selected_data]
train_images = [p for p in all_images if str(p) not in test_image_paths][:1000]

valid_images_count = 0
clean_count = 0
noisy_count = 0

# Initialize pristine base model for pseudo-labeling
model = YOLO('yolov8n.pt')

for i, img_path in enumerate(train_images):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    # Perform inference on pristine image to extract reference ground-truth boxes
    r = model.predict(img, conf=0.20, iou=0.5, verbose=False)[0]
    boxes = r.boxes

    if boxes is None or len(boxes) == 0:
        continue

    xyxy = boxes.xyxy.cpu().numpy()
    cls = boxes.cls.cpu().numpy()

    # 50/50 balanced dataset split: half pristine, half augmented with Gaussian noise
    if random.random() < 0.5:
        final_img_bgr = img
        clean_count += 1
    else:
        # Randomize high variance limits to maximize noise distortion diversity during training
        v_val = random.uniform(2000, 10000)
        transform = A.Compose([
            A.GaussNoise(var_limit=(v_val, v_val), mean=0, p=1.0)
        ])

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        noisy_img_rgb = transform(image=img_rgb)["image"]
        final_img_bgr = cv2.cvtColor(noisy_img_rgb, cv2.COLOR_RGB2BGR)
        noisy_count += 1

    # Save processed training image and its corresponding target label file
    img_out_path = ft_noise_dir / 'images' / 'train' / f"im_{i}.jpg"
    lbl_out_path = ft_noise_dir / 'labels' / 'train' / f"im_{i}.txt"

    cv2.imwrite(str(img_out_path), final_img_bgr)
    save_yolo_label(lbl_out_path, xyxy, cls, w, h)
    valid_images_count += 1

print(f"Dataset Ready! Total: {valid_images_count} images ({clean_count} Clean, {noisy_count} Noisy).")

# Generate YOLO dataset configuration YAML file
yaml_data = {
    'train': str(ft_noise_dir / 'images' / 'train'),
    'val': str(ft_noise_dir / 'images' / 'train'),
    'nc': len(model.names),
    'names': model.names
}
with open(ft_noise_dir / 'data.yaml', 'w') as f:
    yaml.dump(yaml_data, f)

print("Step 2: Training the Noise-Resilient Model (25 Epochs) with lowered Learning Rate...")
current_device = 0 if torch.cuda.is_available() else 'cpu'

# Load a fresh YOLOv8 nano model instance for fine-tuning
ft_model_noise = YOLO("yolov8n.pt")

# Train model: Reduced learning rate (lr0=0.001) prevents catastrophic forgetting of standard features
results = ft_model_noise.train(
    data=str(ft_noise_dir / 'data.yaml'),
    imgsz=640,
    epochs=25,
    batch=8,
    device=current_device,
    verbose=False,
    lr0=0.001
)

# Cache and load the optimal trained weights for subsequent evaluation graphs
best_path = Path(ft_model_noise.trainer.best) if hasattr(ft_model_noise, "trainer") else Path('/content/runs/detect/train/weights/best.pt')
ultimate_noise_model = YOLO(str(best_path))

print("Fine-Tuning Complete! Model is ready for the ultimate test.")

In [ ]:
#@title Task 1: Comparative Evaluation Curve - Original vs. Enhanced vs. Fine-Tuned YOLOv8

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from ultralytics import YOLO

# --- Helper Functions ---
def add_gaussian_noise(image, std):
    # Mathematically apply Gaussian noise safely by casting to float32
    img_float = image.astype(np.float32)
    noise = np.random.normal(0, std, img_float.shape)
    noisy_image = img_float + noise
    return np.clip(noisy_image, 0, 255).astype(np.uint8)

def compute_snr(clean_rgb, noisy_rgb):
    # Quantify degradation using empirical Signal-to-Noise Ratio (SNR)
    clean = clean_rgb.astype(np.float64)
    noise = clean - noisy_rgb.astype(np.float64)
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(signal_power / noise_power)

# --- Model Initialization ---
# Load a pristine base model to ensure clean baseline metrics
fresh_original_model = YOLO("yolov8n.pt")
# Note: ultimate_noise_model is assumed to be pre-loaded from the fine-tuning stage

# Define standard deviation (STD) levels for the comprehensive noise sweep
noise_stds = [10, 30, 60, 100, 150]

m_snr = []
m_recall_distorted = []
m_recall_enhanced = []
m_recall_finetuned = []

print("Calculating SNR and detection recall for all three experimental configurations...")

# --- Main Evaluation Loop ---
for std_val in noise_stds:
    l_snr = []
    l_recall_dist = []
    l_recall_enh = []
    l_recall_ft = []

    for img_path, baseline_result in selected_data:
        # Prepare validation image frames
        img = cv2.imread(str(img_path))
        clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        baseline_objs = len(baseline_result.boxes)

        # Inject noise and compute real-time SNR values
        noisy_rgb = add_gaussian_noise(clean_rgb, std_val)
        noisy_bgr = cv2.cvtColor(noisy_rgb, cv2.COLOR_RGB2BGR)
        l_snr.append(compute_snr(clean_rgb, noisy_rgb))

        # --- Configuration 1: Distorted Raw Input on Baseline Model ---
        noisy_res = fresh_original_model.predict(source=noisy_bgr, verbose=False)
        l_recall_dist.append(min(1.0, len(noisy_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0)

        # --- Configuration 2: Classical Edge-Preserving Filtering (Bilateral) ---
        enh_bgr = cv2.bilateralFilter(noisy_bgr, d=9, sigmaColor=75, sigmaSpace=75)
        enh_res = fresh_original_model.predict(source=enh_bgr, verbose=False)
        l_recall_enh.append(min(1.0, len(enh_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0)

        # --- Configuration 3: Custom Fine-Tuned Model (No Filtering Applied) ---
        ft_res = ultimate_noise_model.predict(source=noisy_bgr, verbose=False)
        l_recall_ft.append(min(1.0, len(ft_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0)

    # Compute mean metrics over the control image subset for current noise intensity
    m_snr.append(np.mean(l_snr))
    m_recall_distorted.append(np.mean(l_recall_dist))
    m_recall_enhanced.append(np.mean(l_recall_enh))
    m_recall_finetuned.append(np.mean(l_recall_ft))

# --- Plotting the Multi-Curve Evaluation Grid ---
plt.figure(figsize=(12, 7))

plt.plot(m_snr, m_recall_distorted, marker='o', linestyle='-', linewidth=2, color='#d62728', label='Distorted (Original YOLO)')
plt.plot(m_snr, m_recall_enhanced, marker='s', linestyle='-', linewidth=2, color='#2ca02c', label='Enhanced (Bilateral Filter)')
plt.plot(m_snr, m_recall_finetuned, marker='^', linestyle='-', linewidth=3, color='#9467bd', label='Ultimate Noise Model (Fine-Tuned)')

# Annotate specific standard deviation steps alongside the target fine-tuned curve
for i, std_val in enumerate(noise_stds):
    plt.annotate(f"STD={std_val}", (m_snr[i], m_recall_finetuned[i]),
                 textcoords="offset points", xytext=(0,12), ha='center', fontsize=10, fontweight='bold', color='#1f77b4')

# Invert X-axis: decreasing SNR represents increasing degradation levels moving rightward
plt.gca().invert_xaxis()

# Ideal baseline target indicator
plt.axhline(y=1.0, color='blue', linestyle='--', label='Clean Baseline (1.00)')

# Graph aesthetics and structural elements for reporting
plt.title("YOLO Detection Recall vs SNR\nGaussian Noise: Distorted vs Enhanced vs Fine-Tuned | Project by: Ayelet & Noa", fontsize=14, fontweight='bold', pad=15)
plt.xlabel(r"SNR (dB) $\rightarrow$ More Noise", fontsize=12, fontweight='bold')
plt.ylabel("Mean Detection Recall vs Clean", fontsize=12, fontweight='bold')
plt.ylim(-0.05, 1.15)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#@title Task 1: Grand Finale - Gaussian Noise Performance Bar Chart Comparison

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Define the target noise standard deviation for the final evaluation (Medium-High difficulty)
std_hard = 60

# Helper function to inject mathematically controlled Gaussian noise
def add_gaussian_noise(image, std):
    img_float = image.astype(np.float32)
    noise = np.random.normal(0, std, img_float.shape)
    noisy_image = img_float + noise
    return np.clip(noisy_image, 0, 255).astype(np.uint8)

# Load a pristine YOLOv8 base model to ensure an unbiased baseline comparison
fresh_original_model = YOLO("yolov8n.pt")

print(f"Running Final Stand-off at Gaussian Noise level STD={std_hard}...")

# Arrays to aggregate recall metrics across the 5 control images
recall_distorted = []
recall_enhanced = []
recall_finetuned = []

for img_path, baseline_result in selected_data:
    img = cv2.imread(str(img_path))
    clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    baseline_objs = len(baseline_result.boxes)

    # 1. Generate the distorted noisy image
    noisy_rgb = add_gaussian_noise(clean_rgb, std_hard)
    noisy_bgr = cv2.cvtColor(noisy_rgb, cv2.COLOR_RGB2BGR)

    # Evaluate Original YOLOv8 on Distorted image
    noisy_res = fresh_original_model.predict(source=noisy_bgr, verbose=False)
    r_dist = min(1.0, len(noisy_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
    recall_distorted.append(r_dist)

    # 2. Apply Bilateral Filter to reduce noise while preserving edges (Enhanced)
    enh_bgr = cv2.bilateralFilter(noisy_bgr, d=9, sigmaColor=75, sigmaSpace=75)

    # Evaluate Original YOLOv8 on the Enhanced image
    enh_res = fresh_original_model.predict(source=enh_bgr, verbose=False)
    r_enh = min(1.0, len(enh_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
    recall_enhanced.append(r_enh)

    # 3. Evaluate the Fine-Tuned 'Ultimate' YOLOv8 model directly on the Distorted image
    # Note: Assumes `ultimate_noise_model` is cached from the previous fine-tuning cell
    ft_res = ultimate_noise_model.predict(source=noisy_bgr, verbose=False)
    r_ft = min(1.0, len(ft_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
    recall_finetuned.append(r_ft)

# Calculate mean recall across the dataset for the final visualization
mean_dist = np.mean(recall_distorted)
mean_enh = np.mean(recall_enhanced)
mean_ft = np.mean(recall_finetuned)

print("Plotting the Grand Finale Bar Chart...")

# --- Generate the Final Bar Chart ---
labels = ['Original YOLOv8\n(Distorted Image)', 'Original YOLOv8\n(Enhanced Image)', 'Fine-Tuned YOLOv8\n(Distorted Image)']
values = [mean_dist, mean_enh, mean_ft]
colors = ['#d62728', '#2ca02c', '#9467bd'] # Red, Green, Purple (maintaining consistent presentation colors)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(labels, values, color=colors, width=0.5, edgecolor='black', linewidth=1.5)

# Reference line representing the clean baseline performance (1.00)
ax.axhline(y=1.0, color='blue', linestyle='--', linewidth=2, label='Clean Baseline (1.00)')

# Annotate exact numerical values on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=14, fontweight='bold')

# Presentation-ready aesthetic formatting
ax.set_ylabel('Mean Detection Recall vs Clean', fontsize=12, fontweight='bold')
ax.set_title(f'YOLOv8 Performance Under Gaussian Noise (STD={std_hard})\nProject by: Ayelet & Noa', fontsize=16, fontweight='bold', pad=20)
ax.set_ylim(0, 1.15)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(loc='upper left', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 1: Distortion Intensity Sweep - Severe JPEG Compression

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Simulate aggressive video/image compression via JPEG encoding artifacts
def apply_jpeg_compression(image, quality):
    # Encode and immediately decode the image via OpenCV to force compression artifacts
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encimg = cv2.imencode('.jpg', image, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return decimg

# Define 5 JPEG quality levels (1-100 scale)
# Lower values result in heavier compression, data loss, and severe macroblocking
quality_levels = [30, 15, 8, 4, 1]

# Initialize a 5x6 plot grid (5 reference images × [1 Original + 5 Quality levels])
fig, axes = plt.subplots(5, 6, figsize=(24, 15))
fig.suptitle('Distortion Intensity Sweep: Severe JPEG Compression (No Model)', fontsize=20, fontweight='bold')

print("Generating JPEG compressed images...")

# Iterate through the cached control images
for row_idx, (img_path, _) in enumerate(selected_data):
    # Load original image and convert to RGB format
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Plot original reference frame in the first column (index 0)
    axes[row_idx, 0].imshow(img_rgb)
    if row_idx == 0:
        axes[row_idx, 0].set_title("Original", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Loop across the 5 defined compression quality levels
    for col_idx, q_val in enumerate(quality_levels, start=1):
        # Apply compression degradation (OpenCV expects BGR input)
        compressed_img_bgr = apply_jpeg_compression(img, q_val)

        # Convert back to RGB for accurate rendering in Matplotlib
        compressed_img_rgb = cv2.cvtColor(compressed_img_bgr, cv2.COLOR_BGR2RGB)

        # Render the degraded frame
        axes[row_idx, col_idx].imshow(compressed_img_rgb)

        # Append dynamic sub-titles only to the top row to preserve layout neatness
        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"Quality: {q_val}", fontsize=14)
        axes[row_idx, col_idx].axis('off')

# Optimize layout padding and render the visual grid
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 1: YOLOv8 Detections Under Severe JPEG Compression Sweep

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Simulate JPEG compression by encoding and immediately decoding the image
def apply_jpeg_compression(image, quality):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encimg = cv2.imencode('.jpg', image, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return decimg

# Load a pristine YOLOv8 model for an unbiased baseline
fresh_original_model = YOLO("yolov8n.pt")

# Predefined JPEG quality levels (1-100 scale, lower indicates heavier compression)
quality_levels = [30, 15, 8, 4, 1]

# Initialize a 5x6 plot grid (5 images × [1 Baseline + 5 Compression levels])
fig, axes = plt.subplots(5, 6, figsize=(24, 15))
fig.suptitle('YOLOv8 Detections Under Severe JPEG Compression', fontsize=20, fontweight='bold')

print("Running YOLO inference on compressed images... Please wait :)")

# Iterate through the 5 cached control images
for row_idx, (img_path, _) in enumerate(selected_data):
    # Load original image in default BGR format
    img = cv2.imread(str(img_path))

    # --- 1. Run baseline inference on the original uncompressed image ---
    res_baseline = fresh_original_model.predict(source=img, verbose=False)[0]
    res_baseline_plotted = res_baseline.plot()
    res_baseline_rgb = cv2.cvtColor(res_baseline_plotted, cv2.COLOR_BGR2RGB) # Convert BGR to RGB for correct Matplotlib rendering

    axes[row_idx, 0].imshow(res_baseline_rgb)
    if row_idx == 0:
        axes[row_idx, 0].set_title(f"Original\nObjects: {len(res_baseline.boxes)}", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # --- 2. Run inference on the degraded (compressed) images ---
    for col_idx, q_val in enumerate(quality_levels, start=1):
        # Generate the JPEG compressed image
        compressed_img_bgr = apply_jpeg_compression(img, q_val)

        # Execute YOLOv8 on the compressed image
        res_compressed = fresh_original_model.predict(source=compressed_img_bgr, verbose=False)[0]

        # Render detection bounding boxes and convert to RGB
        res_compressed_plotted = res_compressed.plot()
        res_compressed_rgb = cv2.cvtColor(res_compressed_plotted, cv2.COLOR_BGR2RGB)

        axes[row_idx, col_idx].imshow(res_compressed_rgb)
        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"Quality: {q_val}\nObjects: {len(res_compressed.boxes)}", fontsize=14)
        axes[row_idx, col_idx].axis('off')

# Optimize sub-plot layout and render the visual grid
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 1: YOLOv8 Recall vs. SNR under Severe JPEG Compression

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Simulate JPEG compression via OpenCV encoding/decoding
def apply_jpeg_compression(image, quality):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encimg = cv2.imencode('.jpg', image, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return decimg

# Compute empirical Signal-to-Noise Ratio (SNR)
def compute_snr(clean_rgb, distorted_rgb):
    clean = clean_rgb.astype(np.float64)
    noise = clean - distorted_rgb.astype(np.float64)
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(signal_power / noise_power)

# Load a pristine YOLOv8 base model to ensure clean baseline metrics
fresh_original_model = YOLO("yolov8n.pt")

# Define JPEG quality levels (1-100 scale, lower indicates heavier compression)
quality_levels = [30, 15, 8, 4, 1]

m_snr = []
m_recall_distorted = []

print("Calculating SNR and detections for compressed images... (Please wait)")

for q_val in quality_levels:
    l_snr = []
    l_recall = []

    # Iterate through the 5 cached control images
    for img_path, baseline_result in selected_data:
        img = cv2.imread(str(img_path))
        clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        baseline_objs = len(baseline_result.boxes)

        # Generate the JPEG compressed image
        compressed_bgr = apply_jpeg_compression(img, q_val)
        compressed_rgb = cv2.cvtColor(compressed_bgr, cv2.COLOR_BGR2RGB)

        # Calculate empirical SNR between clean and compressed images
        l_snr.append(compute_snr(clean_rgb, compressed_rgb))

        # Execute YOLOv8 inference on the compressed image
        compressed_res = fresh_original_model.predict(source=compressed_bgr, verbose=False)

        # Calculate detection recall relative to the clean baseline
        r_dist = min(1.0, len(compressed_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
        l_recall.append(r_dist)

    # Aggregate mean metrics for the current quality level
    m_snr.append(np.mean(l_snr))
    m_recall_distorted.append(np.mean(l_recall))

# --- Plot the Evaluation Curve ---
plt.figure(figsize=(10, 6))

plt.plot(m_snr, m_recall_distorted, marker='o', linestyle='-', linewidth=2.5, color='#d62728', label='Distorted (Original YOLO)')

# Annotate specific JPEG quality value above each data point
for i, q_val in enumerate(quality_levels):
    plt.annotate(f"Q={q_val}", (m_snr[i], m_recall_distorted[i]),
                 textcoords="offset points", xytext=(0,12), ha='center', fontsize=10, fontweight='bold', color='#1f77b4')

# Invert X-axis (decreasing SNR indicates heavier compression moving rightward)
plt.gca().invert_xaxis()

plt.axhline(y=1.0, color='blue', linestyle='--', label='Clean Baseline (1.00)')
plt.title("YOLO Detection Recall vs SNR\nSevere JPEG Compression | Project by: Ayelet & Noa", fontsize=14, fontweight='bold', pad=15)
plt.xlabel(r"SNR (dB) $\rightarrow$ More Compression Artifacts", fontsize=12, fontweight='bold')
plt.ylabel("Mean Detection Recall vs Clean", fontsize=12, fontweight='bold')
plt.ylim(-0.05, 1.15)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#@title Task 1: Severe JPEG Compression Evaluation (Distorted vs. Enhanced)

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Simulate JPEG compression (induces data loss and macroblocking artifacts)
def apply_jpeg_compression(image, quality):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encimg = cv2.imencode('.jpg', image, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return decimg

# Compute empirical Signal-to-Noise Ratio (SNR)
def compute_snr(clean_rgb, distorted_rgb):
    clean = clean_rgb.astype(np.float64)
    noise = clean - distorted_rgb.astype(np.float64)
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(signal_power / noise_power)

# Load a pristine YOLOv8 model for an unbiased baseline
fresh_original_model = YOLO("yolov8n.pt")

# JPEG quality levels (1-100 scale; lower indicates heavier compression)
quality_levels = [30, 15, 8, 4, 1]

m_snr = []
m_recall_distorted = []
m_recall_enhanced = [] # Array to store recall metrics for enhanced (de-blocked) images

print("Calculating SNR, applying JPEG distortion, running de-blocking filter, and evaluating... (Please wait)")

for q_val in quality_levels:
    l_snr = []
    l_recall_dist = []
    l_recall_enh = [] # Internal list for enhanced image recall

    # Iterate through the 5 cached control images
    for img_path, baseline_result in selected_data:
        img = cv2.imread(str(img_path))
        clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        baseline_objs = len(baseline_result.boxes)

        # 1. Generate the JPEG compressed (distorted) image
        compressed_bgr = apply_jpeg_compression(img, q_val)
        compressed_rgb = cv2.cvtColor(compressed_bgr, cv2.COLOR_BGR2RGB)

        # Calculate empirical SNR
        l_snr.append(compute_snr(clean_rgb, compressed_rgb))

        # --- Test 1: Distorted Image ---
        compressed_res = fresh_original_model.predict(source=compressed_bgr, verbose=False)
        r_dist = min(1.0, len(compressed_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
        l_recall_dist.append(r_dist)

        # --- Test 2: Enhanced Image ---
        # Apply a mild bilateral filter to smooth out JPEG blocks while preserving sharp edges
        enh_bgr = cv2.bilateralFilter(compressed_bgr, d=7, sigmaColor=50, sigmaSpace=50)
        enh_res = fresh_original_model.predict(source=enh_bgr, verbose=False)

        r_enh = min(1.0, len(enh_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
        l_recall_enh.append(r_enh)

    # Aggregate mean metrics for the current quality level
    m_snr.append(np.mean(l_snr))
    m_recall_distorted.append(np.mean(l_recall_dist))
    m_recall_enhanced.append(np.mean(l_recall_enh))

# --- Plot the Combined Evaluation Curve ---
plt.figure(figsize=(10, 6))

# Red line - Distorted image evaluation
plt.plot(m_snr, m_recall_distorted, marker='o', linestyle='-', linewidth=2.5, color='#d62728', label='Distorted (Original YOLO)')
# Green line - Enhanced image evaluation
plt.plot(m_snr, m_recall_enhanced, marker='s', linestyle='-', linewidth=2.5, color='#2ca02c', label='Enhanced (De-blocking Filter)')

# Annotate JPEG quality above the higher data point to prevent overlap
for i, q_val in enumerate(quality_levels):
    plt.annotate(f"Q={q_val}", (m_snr[i], max(m_recall_distorted[i], m_recall_enhanced[i])),
                 textcoords="offset points", xytext=(0,12), ha='center', fontsize=10, fontweight='bold', color='#1f77b4')

# Invert X-axis (decreasing SNR indicates heavier compression moving rightward)
plt.gca().invert_xaxis()

plt.axhline(y=1.0, color='blue', linestyle='--', label='Clean Baseline (1.00)')
plt.title("YOLO Detection Recall vs SNR\nSevere JPEG Compression: Distorted vs Enhanced | Project by: Ayelet & Noa", fontsize=14, fontweight='bold', pad=15)
plt.xlabel(r"SNR (dB) $\rightarrow$ More Compression Artifacts", fontsize=12, fontweight='bold')
plt.ylabel("Mean Detection Recall vs Clean", fontsize=12, fontweight='bold')
plt.ylim(-0.05, 1.15)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#@title Task 1: Dataset Generation & Fine-Tuning for Severe JPEG Resilience

import os
import yaml
import random
import cv2
import numpy as np
import torch
from pathlib import Path
from ultralytics import YOLO

# Simulate JPEG compression to induce macroblocking and high-frequency detail loss
def apply_jpeg_compression(image, quality):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encimg = cv2.imencode('.jpg', image, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return decimg

# Define and create workspace directories for JPEG fine-tuning dataset
ft_jpeg_dir = Path('/content/bdd100k_ft_jpeg')
(ft_jpeg_dir / 'images' / 'train').mkdir(parents=True, exist_ok=True)
(ft_jpeg_dir / 'labels' / 'train').mkdir(parents=True, exist_ok=True)

# Helper function to export bounding boxes in YOLO format (normalized cx, cy, w, h)
def save_yolo_label(txt_path, boxes_xyxy, cls_ids, w, h):
    lines = []
    for (x1, y1, x2, y2), c in zip(boxes_xyxy, cls_ids):
        cx = ((x1 + x2) / 2) / w
        cy = ((y1 + y2) / 2) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h
        lines.append(f"{int(c)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    txt_path.write_text("\n".join(lines))

print("Step 1: Creating a Mixed Dataset (1000 images: 50% Clean, 50% JPEG Compressed)...")

# Exclude the 5 test images to strictly prevent data leakage into the training set
test_image_paths = [str(p) for p, _ in selected_data]
train_images = [p for p in all_images if str(p) not in test_image_paths][:1000]

valid_images_count = 0
clean_count = 0
jpeg_count = 0

# Initialize pristine base model for pseudo-labeling (ground truth generation)
model = YOLO('yolov8n.pt')

for i, img_path in enumerate(train_images):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    # Perform inference on pristine image to extract reference bounding boxes
    r = model.predict(img, conf=0.20, iou=0.5, verbose=False)[0]
    boxes = r.boxes

    # Skip images with no detections to avoid training on empty backgrounds
    if boxes is None or len(boxes) == 0:
        continue

    xyxy = boxes.xyxy.cpu().numpy()
    cls = boxes.cls.cpu().numpy()

    # 50/50 balanced dataset split: half pristine, half aggressively compressed
    if random.random() < 0.5:
        final_img_bgr = img
        clean_count += 1
    else:
        # Randomize severe compression levels (Quality 1-15) to maximize robustness
        q_val = random.randint(1, 15)
        final_img_bgr = apply_jpeg_compression(img, q_val)
        jpeg_count += 1

    # Save processed training image and its corresponding target label file
    img_out_path = ft_jpeg_dir / 'images' / 'train' / f"im_{i}.jpg"
    lbl_out_path = ft_jpeg_dir / 'labels' / 'train' / f"im_{i}.txt"

    cv2.imwrite(str(img_out_path), final_img_bgr)
    save_yolo_label(lbl_out_path, xyxy, cls, w, h)
    valid_images_count += 1

print(f"Dataset Ready! Total: {valid_images_count} images ({clean_count} Clean, {jpeg_count} Compressed).")

# Generate YOLO dataset configuration YAML file
yaml_data = {
    'train': str(ft_jpeg_dir / 'images' / 'train'),
    'val': str(ft_jpeg_dir / 'images' / 'train'),
    'nc': len(model.names),
    'names': model.names
}
with open(ft_jpeg_dir / 'data.yaml', 'w') as f:
    yaml.dump(yaml_data, f)

print("Step 2: Training the JPEG-Resilient Model (25 Epochs) with lowered Learning Rate...")
current_device = 0 if torch.cuda.is_available() else 'cpu'

# Load a fresh YOLOv8 nano model instance for fine-tuning
ft_model_jpeg = YOLO("yolov8n.pt")

# Train model: Reduced learning rate (lr0=0.001) mitigates catastrophic forgetting of baseline features
results = ft_model_jpeg.train(
    data=str(ft_jpeg_dir / 'data.yaml'),
    imgsz=640,
    epochs=25,
    batch=8,
    device=current_device,
    verbose=False,
    lr0=0.001
)

# Cache and load the optimal trained weights into ultimate_jpeg_model for downstream evaluation
best_path = Path(ft_model_jpeg.trainer.best) if hasattr(ft_model_jpeg, "trainer") else Path('/content/runs/detect/train/weights/best.pt')
ultimate_jpeg_model = YOLO(str(best_path))

print("Fine-Tuning Complete! Model is ready for the ultimate JPEG test.")

In [ ]:
#@title Task 1: Comparative Evaluation Curve - Original vs. Enhanced vs. Fine-Tuned (JPEG)

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Simulate JPEG compression artifacts (macroblocking and data loss)
def apply_jpeg_compression(image, quality):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encimg = cv2.imencode('.jpg', image, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return decimg

# Compute empirical Signal-to-Noise Ratio (SNR)
def compute_snr(clean_rgb, distorted_rgb):
    clean = clean_rgb.astype(np.float64)
    noise = clean - distorted_rgb.astype(np.float64)
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(signal_power / noise_power)

# Load a pristine YOLOv8 base model to ensure clean baseline metrics
fresh_original_model = YOLO("yolov8n.pt")
# Note: 'ultimate_jpeg_model' is assumed pre-loaded from the fine-tuning stage

# JPEG quality levels (1-100 scale; lower indicates heavier compression)
quality_levels = [30, 15, 8, 4, 1]

m_snr = []
m_recall_distorted = []
m_recall_enhanced = []
m_recall_finetuned = []

print("Calculating SNR and detection recall for all three experimental configurations...")

for q_val in quality_levels:
    l_snr = []
    l_recall_dist = []
    l_recall_enh = []
    l_recall_ft = []

    # Iterate through the 5 cached control images
    for img_path, baseline_result in selected_data:
        img = cv2.imread(str(img_path))
        clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        baseline_objs = len(baseline_result.boxes)

        # Generate the JPEG compressed image
        compressed_bgr = apply_jpeg_compression(img, q_val)
        compressed_rgb = cv2.cvtColor(compressed_bgr, cv2.COLOR_BGR2RGB)

        # Calculate empirical SNR
        l_snr.append(compute_snr(clean_rgb, compressed_rgb))

        # --- Configuration 1: Distorted (Original Model) ---
        compressed_res = fresh_original_model.predict(source=compressed_bgr, verbose=False)
        r_dist = min(1.0, len(compressed_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
        l_recall_dist.append(r_dist)

        # --- Configuration 2: Enhanced (Bilateral Filter + Original Model) ---
        # Apply bilateral filter to smooth compression blocks while preserving edges
        enh_bgr = cv2.bilateralFilter(compressed_bgr, d=7, sigmaColor=50, sigmaSpace=50)
        enh_res = fresh_original_model.predict(source=enh_bgr, verbose=False)
        r_enh = min(1.0, len(enh_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
        l_recall_enh.append(r_enh)

        # --- Configuration 3: Fine-Tuned (Ultimate JPEG Model) ---
        # Execute the custom fine-tuned model directly on the compressed image
        ft_res = ultimate_jpeg_model.predict(source=compressed_bgr, verbose=False)
        r_ft = min(1.0, len(ft_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0
        l_recall_ft.append(r_ft)

    # Aggregate mean metrics for the current compression level
    m_snr.append(np.mean(l_snr))
    m_recall_distorted.append(np.mean(l_recall_dist))
    m_recall_enhanced.append(np.mean(l_recall_enh))
    m_recall_finetuned.append(np.mean(l_recall_ft))

# --- Plotting the Multi-Curve Evaluation Grid ---
plt.figure(figsize=(12, 7))

# Plot curves using consistent project color palette
plt.plot(m_snr, m_recall_distorted, marker='o', linestyle='-', linewidth=2, color='#d62728', label='Distorted (Original YOLO)')
plt.plot(m_snr, m_recall_enhanced, marker='s', linestyle='-', linewidth=2, color='#2ca02c', label='Enhanced (De-blocking Filter)')
plt.plot(m_snr, m_recall_finetuned, marker='^', linestyle='-', linewidth=3, color='#9467bd', label='Ultimate JPEG Model (Fine-Tuned)')

# Annotate JPEG quality above the fine-tuned model curve
for i, q_val in enumerate(quality_levels):
    plt.annotate(f"Q={q_val}", (m_snr[i], m_recall_finetuned[i]),
                 textcoords="offset points", xytext=(0,12), ha='center', fontsize=10, fontweight='bold', color='#1f77b4')

# Invert X-axis: decreasing SNR represents increasing degradation moving rightward
plt.gca().invert_xaxis()

# Ideal baseline target indicator
plt.axhline(y=1.0, color='blue', linestyle='--', label='Clean Baseline (1.00)')

# Graph aesthetics and structural elements for reporting
plt.title("YOLO Detection Recall vs SNR\nSevere JPEG Compression: Distorted vs Enhanced vs Fine-Tuned\nProject by: Ayelet & Noa", fontsize=14, fontweight='bold', pad=15)
plt.xlabel(r"SNR (dB) $\rightarrow$ More Compression Artifacts", fontsize=12, fontweight='bold')
plt.ylabel("Mean Detection Recall vs Clean", fontsize=12, fontweight='bold')
plt.ylim(-0.05, 1.15)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#@title Task 1: Grand Finale - JPEG Compression Grouped Bar Chart Summary

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Simulate JPEG compression (induces data loss and macroblocking)
def apply_jpeg_compression(image, quality):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encimg = cv2.imencode('.jpg', image, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return decimg

# Compute empirical Signal-to-Noise Ratio (SNR)
def compute_snr(clean_rgb, distorted_rgb):
    clean = clean_rgb.astype(np.float64)
    noise = clean - distorted_rgb.astype(np.float64)
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(signal_power / noise_power)

# Load a pristine YOLOv8 model for an unbiased baseline
fresh_original_model = YOLO("yolov8n.pt")
# Note: 'ultimate_jpeg_model' is assumed to be in memory from the fine-tuning cell

# Define JPEG quality levels (lower indicates heavier compression)
quality_levels = [30, 15, 8, 4, 1]

m_snr = []
m_recall_distorted = []
m_recall_enhanced = []
m_recall_finetuned = []

print("Calculating SNR and detection recall for all three configurations... (Please wait)")

for q_val in quality_levels:
    l_snr = []
    l_recall_dist = []
    l_recall_enh = []
    l_recall_ft = []

    # Iterate through the 5 cached control images
    for img_path, baseline_result in selected_data:
        img = cv2.imread(str(img_path))
        clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        baseline_objs = len(baseline_result.boxes)

        # Generate the JPEG compressed image
        compressed_bgr = apply_jpeg_compression(img, q_val)
        compressed_rgb = cv2.cvtColor(compressed_bgr, cv2.COLOR_BGR2RGB)

        # Calculate empirical SNR
        l_snr.append(compute_snr(clean_rgb, compressed_rgb))

        # --- Configuration 1: Distorted (Original Model) ---
        compressed_res = fresh_original_model.predict(source=compressed_bgr, verbose=False)
        l_recall_dist.append(min(1.0, len(compressed_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0)

        # --- Configuration 2: Enhanced (De-blocking Filter + Original Model) ---
        # Apply bilateral filter to smooth macroblocks while preserving edges
        enh_bgr = cv2.bilateralFilter(compressed_bgr, d=7, sigmaColor=50, sigmaSpace=50)
        enh_res = fresh_original_model.predict(source=enh_bgr, verbose=False)
        l_recall_enh.append(min(1.0, len(enh_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0)

        # --- Configuration 3: Fine-Tuned (Ultimate JPEG Model) ---
        # Execute the custom fine-tuned model directly on the compressed image
        try:
            ft_res = ultimate_jpeg_model.predict(source=compressed_bgr, verbose=False)
            l_recall_ft.append(min(1.0, len(ft_res[0].boxes) / baseline_objs) if baseline_objs > 0 else 0.0)
        except NameError:
            print("Warning: ultimate_jpeg_model not found! Assigning 0.0 to Fine-Tuned results.")
            l_recall_ft.append(0.0)

    # Aggregate mean metrics across the control images
    m_snr.append(np.mean(l_snr))
    m_recall_distorted.append(np.mean(l_recall_dist))
    m_recall_enhanced.append(np.mean(l_recall_enh))
    m_recall_finetuned.append(np.mean(l_recall_ft))

# --- Plotting the Grouped Bar Chart Summary ---
plt.figure(figsize=(14, 7))

# Define bar positions and width
x = np.arange(len(quality_levels))
width = 0.25

# Plot grouped bars
bars1 = plt.bar(x - width, m_recall_distorted, width, color='#d62728', edgecolor='black', label='Distorted (Original YOLO)')
bars2 = plt.bar(x, m_recall_enhanced, width, color='#2ca02c', edgecolor='black', label='Enhanced (De-blocking Filter)')
bars3 = plt.bar(x + width, m_recall_finetuned, width, color='#9467bd', edgecolor='black', label='Ultimate JPEG Model (Fine-Tuned)')

# Clean baseline reference line
plt.axhline(y=1.0, color='blue', linestyle='--', linewidth=2, label='Clean Baseline (1.00)')

# Helper function to annotate values above bars
def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        plt.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),  # Slight vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

add_labels(bars1)
add_labels(bars2)
add_labels(bars3)

# Format X-axis with Quality (Q) and average SNR
labels_x = [f"Q={q}\n(SNR: {snr:.1f}dB)" for q, snr in zip(quality_levels, m_snr)]
plt.xticks(x, labels_x, fontsize=11, fontweight='bold')

# Final graph aesthetics and formatting
plt.title("YOLO Detection Recall Under Severe JPEG Compression\nProject by: Ayelet & Noa", fontsize=16, fontweight='bold', pad=20)
plt.xlabel("JPEG Quality (Lower is worse)", fontsize=13, fontweight='bold')
plt.ylabel("Mean Detection Recall vs Clean", fontsize=13, fontweight='bold')
plt.ylim(0, 1.2) # Extended Y-axis to prevent label cropping
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 2: Low-Light Distortion Intensity Sweep Visualization

import cv2
import matplotlib.pyplot as plt
import albumentations as A

# Define 5 distinct intensity levels for low-light degradation
brightness_levels = [-0.1, -0.3, -0.5, -0.7, -0.8]

# Initialize a plot grid dynamically based on the number of selected images
# Columns: 1 Original baseline + 5 distortion levels
fig, axes = plt.subplots(len(selected_data), 6, figsize=(24, 15))
fig.suptitle('Low Light Distortion Sweep (No Model)\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.98)

# Iterate through the cached control images
for row_idx, (img_path, _) in enumerate(selected_data):
    # Load the original image and convert from BGR to RGB format
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Plot original reference frame in the first column (index 0)
    axes[row_idx, 0].imshow(img_rgb)
    if row_idx == 0:
        axes[row_idx, 0].set_title("Original", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Loop across the 5 defined low-light intensity values
    for col_idx, b_val in enumerate(brightness_levels):
        # Configure Albumentations to apply the exact brightness offset
        transform = A.Compose([
            A.RandomBrightnessContrast(brightness_limit=(b_val, b_val), contrast_limit=(0, 0), p=1.0)
        ])

        # Apply the darkening transformation to the original image
        dark_img = transform(image=img_rgb)["image"]

        # Render the degraded image in the corresponding subplot
        curr_ax = axes[row_idx, col_idx + 1]
        curr_ax.imshow(dark_img)

        # Append dynamic sub-titles only to the top row to maintain a clean layout
        if row_idx == 0:
            curr_ax.set_title(f"Level: {b_val}", fontsize=14)
        curr_ax.axis('off')

# Optimize layout padding and render the visual grid
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 2: Mask R-CNN Baseline Generation (NMS & Ego-Vehicle Filter)

import torch
import torchvision
import torchvision.ops as ops  # Import specific computer vision operators (e.g., NMS)
from torchvision.transforms import functional as F
import numpy as np
import matplotlib.pyplot as plt
import cv2

print("Loading Mask R-CNN model (Strict NMS + Ego-Vehicle Filter)...")
# Dynamically assign hardware accelerator
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Ensure the model is loaded once and moved to the correct compute device
if 'model_maskrcnn' not in locals():
    model_maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
model_maskrcnn.to(device)
model_maskrcnn.eval()

def get_raw_masks_filtered(img_rgb, threshold=0.5, iou_nms_threshold=0.3):
    # Convert image to tensor, add batch dimension, and push to compute device
    img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(device)
    h, w = img_rgb.shape[:2]

    with torch.no_grad():
        prediction = model_maskrcnn(img_tensor)[0]

    # 1. Primary spatial filtering based on minimum confidence threshold
    valid_indices = prediction['scores'] > threshold
    masks = prediction['masks'][valid_indices]
    boxes = prediction['boxes'][valid_indices]
    scores = prediction['scores'][valid_indices]

    if len(boxes) == 0:
        return torch.empty((0, 1, h, w))

    # 2. Inject Non-Maximum Suppression (NMS) mechanism
    # Eliminates duplicate overlapping boxes; retains the highest-scoring instance
    # An IoU threshold of 0.3 suppresses lower-confidence boxes overlapping by >30%
    keep_indices = ops.nms(boxes, scores, iou_threshold=iou_nms_threshold)

    masks = masks[keep_indices]
    boxes = boxes[keep_indices]

    # 3. Post-processing Ego-Vehicle filter
    # Removes false positives originating from the camera's host vehicle (e.g., car hood)
    filtered_masks = []
    for i in range(masks.shape[0]):
        box = boxes[i]
        y_center = (box[1].item() + box[3].item()) / 2

        # Filter out predictions whose geometric center falls in the bottom 15% of the frame
        if y_center > h * 0.85:
            continue

        filtered_masks.append(masks[i:i+1])

    if len(filtered_masks) > 0:
        return torch.cat(filtered_masks, dim=0)
    else:
        return torch.empty((0, 1, h, w))

# Upgraded visualization function to overlay instance segmentation masks
def get_clean_visual_segmentation(img_path, threshold=0.5):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Extract clean masks using the custom filtering pipeline
    masks = get_raw_masks_filtered(img_rgb, threshold)

    res_img = img_rgb.copy()
    for i in range(masks.shape[0]):
        # Offload tensor back to CPU for NumPy conversion and Matplotlib rendering
        mask = masks[i, 0].cpu().numpy()
        color = np.random.randint(0, 255, 3, dtype=np.uint8)

        # Apply alpha blending for mask visualization
        res_img[mask > 0.5] = res_img[mask > 0.5] * 0.5 + color * 0.5

    return res_img, len(masks)

print("Generating PERFECT Mask R-CNN Baselines (No Ego, No Double Detections)...")

# Initialize plotting grid
fig, axes = plt.subplots(len(selected_data), 2, figsize=(14, 5 * len(selected_data)))
fig.suptitle(' Final Mask R-CNN Baseline (NMS Enabled)\nProject by: Ayelet & Noa', fontsize=16, y=1.02, fontweight='bold')

for i, (img_path, _) in enumerate(selected_data):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Render raw input
    axes[i, 0].imshow(img_rgb)
    axes[i, 0].set_title(f"Original Image {i+1}")
    axes[i, 0].axis('off')

    # Execute segmentation pipeline and render output
    res_img, obj_count = get_clean_visual_segmentation(img_path)

    axes[i, 1].imshow(res_img.astype(np.uint8))
    axes[i, 1].set_title(f"Mask R-CNN with NMS (Found {obj_count} instances)")
    axes[i, 1].axis('off')

# Optimize layout and render visual outputs
plt.tight_layout()
plt.show()

In [ ]:
#@title Task 2: Mask R-CNN Segmentation Under Low-Light Distortion Sweep

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
import torch
from torchvision.transforms import functional as F

print("Generating Task 2 Visual Sweep: Mask R-CNN Detections under Low Light...")

# Define 5 distinct intensity levels for low-light degradation
brightness_levels = [-0.1, -0.3, -0.5, -0.7, -0.8]

# Initialize a 5x6 plot grid (5 images × [1 Baseline + 5 Distortion levels])
fig, axes = plt.subplots(5, 6, figsize=(24, 18))
fig.suptitle('Mask R-CNN Instance Segmentation under Low Light Sweep\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.99)

# Iterate through the 5 original control images
for row_idx, (img_path, _) in enumerate(selected_data):
    # Load image and convert to RGB format
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # --- Column 0: Clean baseline with NMS and ego-vehicle filtering ---
    base_masks = get_raw_masks_filtered(img_rgb, threshold=0.5)
    res_img_base = img_rgb.copy()

    # Overlay baseline masks
    for i in range(base_masks.shape[0]):
        mask = base_masks[i, 0].cpu().numpy()
        color = np.random.randint(0, 255, 3, dtype=np.uint8)
        res_img_base[mask > 0.5] = res_img_base[mask > 0.5] * 0.5 + color * 0.5

    axes[row_idx, 0].imshow(res_img_base.astype(np.uint8))
    if row_idx == 0:
        axes[row_idx, 0].set_title(f"Baseline\nInstances: {len(base_masks)}", fontsize=14, fontweight='bold')
    else:
        axes[row_idx, 0].set_title(f"Instances: {len(base_masks)}", fontsize=12)
    axes[row_idx, 0].axis('off')

    # --- Columns 1-5: Apply low-light distortion and run the optimized model ---
    for col_idx, b_val in enumerate(brightness_levels):
        # Configure the specific darkening transformation
        transform = A.Compose([
            A.RandomBrightnessContrast(brightness_limit=(b_val, b_val), contrast_limit=(0, 0), p=1.0)
        ])
        dark_img_rgb = transform(image=img_rgb)["image"]

        # Execute our optimized pipeline (includes built-in NMS and ego-vehicle filtering)
        dark_masks = get_raw_masks_filtered(dark_img_rgb, threshold=0.5)

        # Overlay the segmentation masks onto the darkened image
        res_img_dark = dark_img_rgb.copy()
        for i in range(dark_masks.shape[0]):
            mask = dark_masks[i, 0].cpu().numpy()
            color = np.random.randint(0, 255, 3, dtype=np.uint8)
            res_img_dark[mask > 0.5] = res_img_dark[mask > 0.5] * 0.5 + color * 0.5

        # Render the result in the corresponding grid cell
        curr_ax = axes[row_idx, col_idx + 1]
        curr_ax.imshow(res_img_dark.astype(np.uint8))

        num_objs = len(dark_masks)
        if row_idx == 0:
            curr_ax.set_title(f"Level: {b_val}\nInstances: {num_objs}", fontsize=14)
        else:
            curr_ax.set_title(f"Instances: {num_objs}", fontsize=12)
        curr_ax.axis('off')

# Optimize layout and render the visual grid
plt.tight_layout()
plt.show()

In [ ]:
#@title Task 2: Mask R-CNN Evaluation Curve - IoU-Matched Recall vs. SNR

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A

print("Task 2: Plotting Scientific Graph - Mask R-CNN Degradation (Filtered)...")

# --- Helper Functions ---
def compute_snr(clean_rgb, dist_rgb):
    # Quantify distortion severity using empirical Signal-to-Noise Ratio (SNR)
    clean = clean_rgb.astype(np.float64)
    noise = (clean - dist_rgb.astype(np.float64))
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(signal_power / noise_power)

def compute_mask_iou(mask1, mask2):
    # Calculate Intersection over Union (IoU) between two binary segmentation masks
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return intersection / union if union > 0 else 0.0

def calculate_matched_recall(base_masks, dist_masks, iou_thresh=0.5):
    # Calculate recall by matching degraded masks to the clean baseline masks via IoU
    if len(base_masks) == 0: return np.nan
    if len(dist_masks) == 0: return 0.0

    matched = 0
    for b_m in base_masks:
        b_bin = b_m[0].cpu().numpy() > 0.5
        best_iou = 0

        # Find the best matching mask from the distorted predictions
        for d_m in dist_masks:
            d_bin = d_m[0].cpu().numpy() > 0.5
            iou = compute_mask_iou(b_bin, d_bin)
            if iou > best_iou: best_iou = iou

        # Count as a valid detection if IoU exceeds the defined threshold
        if best_iou > iou_thresh: matched += 1

    return matched / len(base_masks)

# --- Scientific Evaluation Sweep ---
brightness_levels = [-0.1, -0.3, -0.5, -0.7, -0.8]
m_snr = []
m_recall = []

for b_val in brightness_levels:
    lvl_snr, lvl_rec = [], []

    # Configure the specific low-light intensity offset
    transform = A.Compose([
        A.RandomBrightnessContrast(brightness_limit=(b_val, b_val), contrast_limit=(0, 0), p=1.0)
    ])

    for img_path, _ in selected_data:
        img = cv2.imread(str(img_path))
        clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        dark_rgb = transform(image=clean_rgb)["image"]

        # Extract instance masks using our optimized pipeline (NMS + Ego-Vehicle filter)
        base_masks = get_raw_masks_filtered(clean_rgb, threshold=0.5)
        dist_masks = get_raw_masks_filtered(dark_rgb, threshold=0.5)

        # Calculate metrics for the current image iteration
        snr = compute_snr(clean_rgb, dark_rgb)
        recall = calculate_matched_recall(base_masks, dist_masks)

        if not np.isnan(recall):
            lvl_snr.append(snr)
            lvl_rec.append(recall)

    # Aggregate mean metrics for the current distortion level
    m_snr.append(np.mean(lvl_snr))
    m_recall.append(np.mean(lvl_rec))
    print(f"Level {b_val}: SNR = {m_snr[-1]:.2f} dB, Recall = {m_recall[-1]:.2f}")

# --- Render the Evaluation Curve ---
fig, ax = plt.subplots(figsize=(10, 6))

# Plot degraded recall curve
ax.plot(m_snr, m_recall, marker='o', markersize=8, color='tab:red', linewidth=2.5, label='Degraded Recall')

# Ideal baseline target indicator
ax.axhline(y=1.0, color='mediumblue', linestyle='--', label='Clean Baseline (Perfect Match)')

# Graph aesthetics and structural formatting
ax.set_title('Mask R-CNN IoU-Matched Recall vs SNR\nFiltered Baseline | Project by: Ayelet & Noa', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel(r'SNR (dB) $\rightarrow$ Worse Quality (Darker)', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Matched Recall (IoU > 0.5)', fontsize=12, fontweight='bold')

# Invert X-axis (decreasing SNR indicates heavier degradation moving rightward)
ax.invert_xaxis()
ax.set_ylim(-0.05, 1.1)

# Annotate specific brightness factors above each plotted point
for i, b_val in enumerate(brightness_levels):
    ax.annotate(f"b={b_val}", (m_snr[i], m_recall[i]), textcoords="offset points", xytext=(0, 10), ha='center', fontsize=10, fontweight='bold')

ax.grid(True, linestyle=':', alpha=0.7)
ax.legend(loc='lower left', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 2: Mask R-CNN Low Light Evaluation (Distorted vs. Enhanced)

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A

print("Running Step 5: Mask R-CNN Enhancement Sweep (Filtered Distorted vs. Enhanced)...")

# 1. Classical restoration function: Adaptive Gamma + CLAHE
def restore_lowlight_adaptive(img_rgb, b_val):
    abs_b = abs(b_val)
    gamma = np.clip(0.95 - 0.75 * (abs_b - 0.1) / 0.9, 0.2, 0.95)
    clip_limit = np.clip(1.2 + 6.8 * (abs_b - 0.1) / 0.9, 1.2, 8.0)

    lut = np.clip((np.arange(256) / 255.0) ** gamma * 255, 0, 255).astype(np.uint8)
    img_gamma = cv2.LUT(img_rgb, lut)

    lab = cv2.cvtColor(img_gamma, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
    out = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2RGB)

    return out

brightness_levels = [-0.1, -0.3, -0.5, -0.7, -0.8]
m_snr = []
m_recall_distorted = []
m_recall_enhanced = []

# Phase A: Compute precise baseline (with NMS and ego-vehicle filtering) for each control image
baseline_data = []
for img_path, _ in selected_data:
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Extract pristine baseline masks using our optimized filtering pipeline
    base_masks = get_raw_masks_filtered(img_rgb, threshold=0.5)
    baseline_data.append((img_rgb, base_masks))

# Phase B: Comprehensive evaluation sweep across all intensity levels
for b_val in brightness_levels:
    l_snr, l_recall_dist, l_recall_enh = [], [], []

    transform = A.Compose([A.RandomBrightnessContrast(brightness_limit=(b_val, b_val), contrast_limit=(0, 0), p=1.0)])

    for img_rgb, base_masks in baseline_data:
        # 1. Induce low-light distortion and compute empirical SNR
        dark_img = transform(image=img_rgb)["image"]
        snr = compute_snr(img_rgb, dark_img)

        # 2. Run model inference on the distorted (unfiltered) image
        dist_masks = get_raw_masks_filtered(dark_img, threshold=0.5)
        recall_dist = calculate_matched_recall(base_masks, dist_masks)

        # 3. Apply mathematical enhancement filter (Adaptive Gamma + CLAHE)
        enh_img = restore_lowlight_adaptive(dark_img, b_val)

        # 4. Run model inference on the restored image
        enh_masks = get_raw_masks_filtered(enh_img, threshold=0.5)
        recall_enh = calculate_matched_recall(base_masks, enh_masks)

        if not np.isnan(recall_dist):
            l_snr.append(snr)
            l_recall_dist.append(recall_dist)
            l_recall_enh.append(recall_enh)

    # Aggregate mean metrics for the current intensity level
    m_snr.append(np.mean(l_snr))
    m_recall_distorted.append(np.mean(l_recall_dist))
    m_recall_enhanced.append(np.mean(l_recall_enh))
    print(f"Level {b_val}: SNR={m_snr[-1]:.1f}dB | Distorted Recall={m_recall_distorted[-1]:.2f} | Enhanced Recall={m_recall_enhanced[-1]:.2f}")

# --- Phase C: Plotting the Combined Evaluation Graph ---
fig, ax = plt.subplots(figsize=(11, 6))

# Plot failing distorted curve (Red) vs. recovering enhanced curve (Green)
ax.plot(m_snr, m_recall_distorted, marker='o', linestyle='-', linewidth=2.5, color='#d62728', label='Distorted (No Filter)')
ax.plot(m_snr, m_recall_enhanced, marker='s', linestyle='-', linewidth=2.5, color='#2ca02c', label='Enhanced (Gamma + CLAHE)')

# Annotate brightness factor above the higher data point to prevent overlap
for i, b_val in enumerate(brightness_levels):
    ax.annotate(f"b={b_val}", (m_snr[i], max(m_recall_distorted[i], m_recall_enhanced[i])),
                textcoords="offset points", xytext=(0,12), ha='center', fontsize=10, fontweight='bold', color='#1f77b4')

ax.invert_xaxis()
ax.axhline(y=1.0, color='blue', linestyle='--', label='Clean Baseline (1.00)')

# Graph aesthetics and structural formatting
ax.set_title("Mask R-CNN Recall: Distorted vs. Enhanced (Low Light)\nFiltered Baseline (NMS & No Ego-Vehicle)", fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel(r"SNR (dB) $\rightarrow$ Worse Quality (Darker)", fontsize=12, fontweight='bold')
ax.set_ylabel("Mean Matched Recall (IoU > 0.5)", fontsize=12, fontweight='bold')
ax.set_ylim(-0.05, 1.15)
ax.grid(True, alpha=0.4, linestyle='--')
ax.legend(loc='lower left', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 2: Alpha Blending Precision & Recall Metrics (YOLOv8)

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
import torch
from ultralytics import YOLO

print("Executing Lecturer's Challenge: Alpha Blending & Precision Metrics...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

print("Loading pristine model...")
# Initialize YOLOv8 model for bounding box extraction (red boxes in visualization)
model_yolo = YOLO('yolov8n.pt')

# Classical low-light restoration filter (Adaptive Gamma + CLAHE)
def restore_lowlight_adaptive(img_rgb, b_val):
    abs_b = abs(b_val)
    gamma = np.clip(0.95 - 0.75 * (abs_b - 0.1) / 0.9, 0.2, 0.95)
    clip_limit = np.clip(1.2 + 6.8 * (abs_b - 0.1) / 0.9, 1.2, 8.0)

    lut = np.clip((np.arange(256) / 255.0) ** gamma * 255, 0, 255).astype(np.uint8)
    img_gamma = cv2.LUT(img_rgb, lut)

    lab = cv2.cvtColor(img_gamma, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
    out = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2RGB)
    return out

# Compute Intersection over Union (IoU) between two bounding boxes
def box_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    interArea = max(0, x2 - x1) * max(0, y2 - y1)
    box1Area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2Area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    iou = interArea / float(box1Area + box2Area - interArea) if (box1Area + box2Area - interArea) > 0 else 0
    return iou

print("Calculating Baseline...")
baseline_boxes_dict = {}
for img_path, _ in selected_data:
    img = cv2.imread(str(img_path))
    results = model_yolo.predict(img, verbose=False)[0]
    baseline_boxes_dict[str(img_path)] = results.boxes.xyxy.cpu().numpy()

print("Running Alpha Blending sweep...")
b_val = -0.5
alphas = [0.0, 0.3, 0.5, 0.7, 1.0]

m_recall = []
m_precision = []

worst_precision = 1.0
worst_img = None
worst_preds = None

transform = A.Compose([A.RandomBrightnessContrast(brightness_limit=(b_val, b_val), contrast_limit=(0, 0), p=1.0)])

print("\n--- Data ---")
for alpha in alphas:
    l_recall = []
    l_precision = []

    for img_path, _ in selected_data:
        img = cv2.imread(str(img_path))
        clean_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 1. Apply low-light distortion and generate the restored image
        dark_img = transform(image=clean_rgb)["image"]
        enhanced_img = restore_lowlight_adaptive(dark_img, b_val)

        # 2. Perform linear Alpha Blending between the dark and restored images
        blended_img = cv2.addWeighted(enhanced_img, alpha, dark_img, 1.0 - alpha, 0)

        # 3. Run YOLO inference on the blended image
        results = model_yolo.predict(blended_img, verbose=False)[0]
        pred_boxes = results.boxes.xyxy.cpu().numpy()
        base_boxes = baseline_boxes_dict[str(img_path)]

        # 4. Calculate True Positives (TP), False Positives (FP), and False Negatives (FN)
        tp = 0
        matched_base = set()

        for p_box in pred_boxes:
            best_iou = 0
            best_base_idx = -1
            for i, b_box in enumerate(base_boxes):
                if i in matched_base: continue
                iou = box_iou(p_box, b_box)
                if iou > best_iou:
                    best_iou = iou
                    best_base_idx = i

            if best_iou > 0.5:
                tp += 1
                matched_base.add(best_base_idx)

        fp = len(pred_boxes) - tp
        fn = len(base_boxes) - tp

        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

        if not np.isnan(recall):
            l_recall.append(recall)
            l_precision.append(precision)

            # Track and cache the worst-case precision image for visual inspection
            if precision < worst_precision and precision > 0:
                worst_precision = precision
                worst_img = blended_img.copy()
                worst_preds = pred_boxes

    mean_rec = np.mean(l_recall)
    mean_prec = np.mean(l_precision)
    m_recall.append(mean_rec)
    m_precision.append(mean_prec)
    print(f"Alpha {alpha:.1f} | Recall: {mean_rec:.3f} | Precision: {mean_prec:.3f}")

# --- Plotting Dual-Axis Evaluation Graph ---
fig, ax1 = plt.subplots(figsize=(10, 5))

color = 'tab:red'
ax1.set_xlabel('Alpha (0 = 100% Dark, 1 = 100% Enhanced Filter)', fontweight='bold')
ax1.set_ylabel('Mean Recall', color=color, fontweight='bold')
ax1.plot(alphas, m_recall, color=color, marker='o', linewidth=2, label='Recall')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(-0.05, 1.05)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Mean Precision', color=color, fontweight='bold')
ax2.plot(alphas, m_precision, color=color, marker='s', linestyle='--', linewidth=2, label='Precision')
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(-0.05, 1.05)

plt.title(f"Degradation Tracking: Alpha Blending at b={b_val}\nProject by: Noa & Ayelet", fontweight='bold', pad=15)
plt.show()

# --- Visual Inspection of the Worst-Case Scenario ---
if worst_img is not None:
    print(f"\n--- Visual Inspection: The Worst Precision Case ---")
    fig, ax = plt.subplots(figsize=(10, 6))

    # Render prediction bounding boxes (in red)
    for box in worst_preds:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(worst_img, (x1, y1), (x2, y2), (255, 0, 0), 3) # RGB format for red

    ax.imshow(worst_img)
    ax.set_title(f"Visual Error Inspection (Precision = {worst_precision:.2f})\nRed boxes are model predictions. Do you see a pattern?", fontweight='bold')
    ax.axis('off')
    plt.show()
else:
    print("No worst case found (Precision was perfect).")

In [ ]:
#@title Task 2: Mask R-CNN Fine-Tuning - Data Prep (50/50 Mix, Full Range)

import random
import cv2
import torch
import numpy as np
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F

print("Preparing Mask R-CNN Dataset with strict 50/50 Mix and Full Darkness Range...")

class MaskRCNNDarkDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths
        # Albumentations transform covering a full sweep of low-light conditions
        self.darken_transform = A.Compose([
            A.RandomBrightnessContrast(brightness_limit=(-0.85, -0.05), contrast_limit=(0, 0), p=1.0)
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = str(self.image_paths[idx])
        img_bgr = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Standardize resolution to preserve dash-cam aspect ratio (16:9)
        img_rgb = cv2.resize(img_rgb, (512, 288))

        # The clean image acts as the baseline for Teacher model pseudo-labeling
        clean_tensor = F.to_tensor(img_rgb)

        # Strict 50/50 split: Half of the batch forces the model to adapt to darkness,
        # while the other half prevents catastrophic forgetting of well-lit scenarios.
        if random.random() < 0.50:
            dark_rgb = self.darken_transform(image=img_rgb)["image"]
        else:
            dark_rgb = img_rgb

        dark_tensor = F.to_tensor(dark_rgb)

        return dark_tensor, clean_tensor

# Filter out test images to maintain strict isolation of the evaluation set
test_image_paths = [str(p) for p, _ in selected_data]
train_images = [p for p in all_images if str(p) not in test_image_paths][:3000]

# Initialize Dataset and DataLoader (batch_size=4 is standard for Mask R-CNN to avoid OOM on GPU)
train_dataset = MaskRCNNDarkDataset(train_images)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)

print(f"Mask R-CNN Dataset ready! Total training images: {len(train_dataset)}")

In [ ]:
#@title Task 2: Mask R-CNN Fine-Tuning - Smart Low-Light Adaptation

import copy
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights

print("Initializing Mask R-CNN Teacher-Student Architecture...")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Explicitly load the standard pre-trained Mask R-CNN model
print("Loading base PyTorch model...")
weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
mask_rcnn_model = maskrcnn_resnet50_fpn(weights=weights)

# Initialize Teacher model and freeze it (eval mode) to generate stable pseudo-labels
teacher_model = mask_rcnn_model.to(device)
teacher_model.eval()

# Initialize Student model via deep copy and set to training mode to enable loss computation
student_model = copy.deepcopy(mask_rcnn_model).to(device)
student_model.train()

# Unfreeze the entire network to allow deep feature adaptation under low SNR environments
for param in student_model.parameters():
    param.requires_grad = True

# Mask R-CNN is highly sensitive; use a lower learning rate (1e-5) with weight decay for stability
optimizer = optim.AdamW(student_model.parameters(), lr=1e-5, weight_decay=1e-4)

# Cosine Annealing scheduler ensures smooth convergence over the training cycle
epochs = 10
scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)

print(f"Starting Fine-Tuning for {epochs} epochs...")

for epoch in range(epochs):
    epoch_loss = 0.0

    for batch_idx, (dark_inputs, clean_inputs) in enumerate(train_loader):
        dark_inputs = list(img.to(device) for img in dark_inputs)
        clean_inputs = list(img.to(device) for img in clean_inputs)

        # 1. Generate Pseudo-Labels (Ground Truth) using the frozen Teacher on CLEAN images
        with torch.no_grad():
            teacher_outputs = teacher_model(clean_inputs)

        targets = []
        for output in teacher_outputs:
            # Filter high-confidence detections (score > 0.6) to ensure high-quality ground truth
            keep = output['scores'] > 0.6
            target = {
                'boxes': output['boxes'][keep],
                'labels': output['labels'][keep],
                'masks': output['masks'][keep]
            }
            targets.append(target)

        # Edge Case Safety: Skip batch if no confident objects were detected by the Teacher
        if any(len(t['boxes']) == 0 for t in targets):
            continue

        # 2. Forward pass through the Student model using DARK images and the generated targets
        optimizer.zero_grad()

        # PyTorch Mask R-CNN automatically computes a dictionary of losses during .train()
        loss_dict = student_model(dark_inputs, targets)

        # Sum all multi-task losses (classification, bounding box regression, mask generation, RPN)
        losses = sum(loss for loss in loss_dict.values())

        # 3. Backpropagation and optimization step
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

        if batch_idx % 25 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(train_loader)}] | Total Loss: {losses.item():.4f}")

    # Step the learning rate scheduler per epoch
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"===> Epoch {epoch+1} Average Loss: {epoch_loss / len(train_loader):.4f} | Current LR: {current_lr:.7f}")

# Persist the optimized weights for evaluation
torch.save(student_model.state_dict(), '/content/mask_rcnn_finetuned_darkness.pth')
print("Training Complete! Model saved to '/content/mask_rcnn_finetuned_darkness.pth'")

In [ ]:
#@title Task 2 - Step 8: The Grand Finale Sweep (Original vs. Enhanced vs. Fine-Tuned)



import torch

import torchvision

import torchvision.ops as ops

from torchvision.transforms import functional as F

import cv2

import numpy as np

import matplotlib.pyplot as plt

import albumentations as A



print("\nStep 8: Running Final Evaluation Sweep across all Low-Light levels...")


# --- 1. Model Initialization ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Ensure pristine original model is loaded for baseline and standard comparisons
weights = torchvision.models.detection.MaskRCNN_ResNet50_FPN_Weights.DEFAULT
fresh_original_model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=weights).to(device)
fresh_original_model.eval()

# Load the NEW Smart Fine-Tuned model from the saved .pth file
print("Loading the Smart Fine-Tuned Mask R-CNN model...")
# Initialize the base architecture
finetuned_maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None)
# Load our newly trained weights
finetuned_maskrcnn.load_state_dict(torch.load('/content/mask_rcnn_finetuned_darkness.pth', map_location=device))
finetuned_maskrcnn.to(device)
finetuned_maskrcnn.eval()


# --- 2. Ultimate Evaluation Pipeline (NMS + Ego-Vehicle Filter) ---

def get_clean_boxes(model_obj, img_rgb, threshold=0.25, iou_nms=0.3):

    h, w = img_rgb.shape[:2]

    img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(device)



    with torch.no_grad():

        prediction = model_obj(img_tensor)[0]



    # Primary confidence thresholding

    valid = prediction['scores'] > threshold

    boxes = prediction['boxes'][valid]

    scores = prediction['scores'][valid]



    if len(boxes) == 0: return np.array([])



    # Apply Non-Maximum Suppression (NMS)

    keep_nms = ops.nms(boxes, scores, iou_threshold=iou_nms)

    boxes = boxes[keep_nms]



    # Post-processing Ego-Vehicle Filter (Exclude bottom 15%)

    final_boxes = []

    for box in boxes:

        y_center = (box[1].item() + box[3].item()) / 2

        if y_center <= h * 0.85:

            final_boxes.append(box.cpu().numpy())



    return np.array(final_boxes) if len(final_boxes) > 0 else np.array([])



# --- 3. Signal Processing & Metric Utilities ---

def restore_lowlight_adaptive(img_rgb, b_val):

    abs_b = abs(b_val)

    gamma = np.clip(0.95 - 0.75 * (abs_b - 0.1) / 0.9, 0.2, 0.95)

    clip_limit = np.clip(1.2 + 6.8 * (abs_b - 0.1) / 0.9, 1.2, 8.0)

    lut = np.clip((np.arange(256) / 255.0) ** gamma * 255, 0, 255).astype(np.uint8)

    img_gamma = cv2.LUT(img_rgb, lut)

    lab = cv2.cvtColor(img_gamma, cv2.COLOR_RGB2LAB)

    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))

    out = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2RGB)

    return out



def compute_iou(box1, box2):

    x_left, y_top = max(box1[0], box2[0]), max(box1[1], box2[1])

    x_right, y_bottom = min(box1[2], box2[2]), min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top: return 0.0

    intersection = (x_right - x_left) * (y_bottom - y_top)

    b1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])

    b2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    return intersection / float(b1_area + b2_area - intersection)



def calculate_matched_recall(base_boxes, dist_boxes, iou_threshold=0.5):

    if len(base_boxes) == 0: return np.nan

    if len(dist_boxes) == 0: return 0.0

    matched = 0

    for b_box in base_boxes:

        best_iou = 0

        for d_box in dist_boxes:

            iou = compute_iou(b_box, d_box)

            if iou > best_iou: best_iou = iou

        if best_iou > iou_threshold: matched += 1

    return matched / len(base_boxes)



def compute_snr(clean_rgb, dist_rgb):

    clean, dist_img = clean_rgb.astype(np.float64), dist_rgb.astype(np.float64)

    noise_power = np.mean((clean - dist_img) ** 2)

    if noise_power == 0: return np.inf

    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)





# --- Phase A: Compute Precise Baseline for each Control Image ---

print("Extracting pristine baseline data...")

baseline_data = []

for img_path, _ in selected_data:

    img = cv2.imread(str(img_path))

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    base_boxes = get_clean_boxes(fresh_original_model, img_rgb)

    baseline_data.append((img_rgb, base_boxes))





# --- Phase B: Comprehensive Evaluation Sweep ---

brightness_levels = [-0.1, -0.3, -0.5, -0.7, -0.8]

m_snr = []

m_recall_distorted = []

m_recall_enhanced = []

m_recall_finetuned = []



print("Running head-to-head evaluation across all distortion levels...")

for b_val in brightness_levels:

    l_snr, l_r_dist, l_r_enh, l_r_ft = [], [], [], []

    transform = A.Compose([A.RandomBrightnessContrast(brightness_limit=(b_val, b_val), contrast_limit=(0, 0), p=1.0)])



    for img_rgb, base_boxes in baseline_data:

        # 1. Induce distortion and compute SNR

        dark_img = transform(image=img_rgb)["image"]

        snr = compute_snr(img_rgb, dark_img)



        # 2. Config 1: Original Model on Distorted Image

        dark_boxes_orig = get_clean_boxes(fresh_original_model, dark_img)

        r_dist = calculate_matched_recall(base_boxes, dark_boxes_orig)



        # 3. Config 2: Original Model on Enhanced Image (Gamma + CLAHE)

        enh_img = restore_lowlight_adaptive(dark_img, b_val)

        enh_boxes_orig = get_clean_boxes(fresh_original_model, enh_img)

        r_enh = calculate_matched_recall(base_boxes, enh_boxes_orig)



        # 4. Config 3: Fine-Tuned Model on Distorted Image

        dark_boxes_ft = get_clean_boxes(finetuned_maskrcnn, dark_img)

        r_ft = calculate_matched_recall(base_boxes, dark_boxes_ft)



        if not np.isnan(r_dist):

            l_snr.append(snr)

            l_r_dist.append(r_dist)

            l_r_enh.append(r_enh)

            l_r_ft.append(r_ft)



    # Aggregate means

    m_snr.append(np.mean(l_snr))

    m_recall_distorted.append(np.mean(l_r_dist))

    m_recall_enhanced.append(np.mean(l_r_enh))

    m_recall_finetuned.append(np.mean(l_r_ft))

    print(f"Level {b_val}: SNR={m_snr[-1]:.1f}dB | Orig={m_recall_distorted[-1]:.2f} | Enh={m_recall_enhanced[-1]:.2f} | FT={m_recall_finetuned[-1]:.2f}")





# --- Phase C: Plotting the Grand Finale Curve ---

fig, ax = plt.subplots(figsize=(11, 6))



# Plot all three configurations

ax.plot(m_snr, m_recall_distorted, marker='o', linestyle='-', linewidth=2.5, color='#d62728', label='Original Model (Distorted Image)')

ax.plot(m_snr, m_recall_enhanced, marker='s', linestyle='-', linewidth=2.5, color='#2ca02c', label='Original Model (Enhanced Image)')

ax.plot(m_snr, m_recall_finetuned, marker='^', linestyle='-', linewidth=2.5, color='#9467bd', label='Fine-Tuned Model (Distorted Image)')



# Clean baseline reference

ax.axhline(y=1.0, color='blue', linestyle='--', label='Clean Baseline (1.00)')



# Annotate brightness values at the top of the curve cluster

for i, b_val in enumerate(brightness_levels):

    highest_val = max(m_recall_distorted[i], m_recall_enhanced[i], m_recall_finetuned[i])

    ax.annotate(f"b={b_val}", (m_snr[i], highest_val),

                textcoords="offset points", xytext=(0,12), ha='center', fontsize=10, fontweight='bold', color='#1f77b4')



ax.invert_xaxis() # Decreasing SNR flows rightwards



# Graph aesthetics and structural formatting

ax.set_title("The Grand Finale: Low Light Recall Comparison\n(Original vs. Mathematical Enhancement vs. Deep Fine-Tuning)", fontsize=15, fontweight='bold', pad=15)

ax.set_xlabel(r"SNR (dB) $\rightarrow$ Worse Quality (Darker)", fontsize=12, fontweight='bold')

ax.set_ylabel("Mean Matched Recall (IoU > 0.5)", fontsize=12, fontweight='bold')

ax.set_ylim(-0.05, 1.25)

ax.grid(True, alpha=0.4, linestyle='--')

ax.legend(loc='lower left', fontsize=11)



plt.tight_layout()

plt.show()

In [ ]:
#@title Task 2 - Distortion: Gaussian noise

In [ ]:
#@title Task 2: Distortion Intensity Sweep - Gaussian Noise (No Model)

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Custom function to generate Gaussian noise with complete mathematical control
def add_gaussian_noise(image, std):
    # Convert to float32 to prevent underflow/overflow clipping artifacts during calculation
    img_float = image.astype(np.float32)

    # Generate Gaussian noise matrix with mean=0 and target standard deviation (std)
    noise = np.random.normal(0, std, img_float.shape)

    # Superimpose the noise matrix onto the original image
    noisy_image = img_float + noise

    # Clip pixel values to the valid [0, 255] range and cast back to uint8
    noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)
    return noisy_image

# Define 5 gradual intensity levels using standard deviation (STD) - from light to severe noise
noise_stds = [10, 30, 60, 100, 150]

# Initialize plot grid: dynamic rows (images) and 6 columns (1 Original + 5 Noise levels)
fig, axes = plt.subplots(len(selected_data), 6, figsize=(24, 15))
fig.suptitle('Task 2 - Step 2: Distortion Intensity Sweep: Gaussian Noise (No Model)\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.98)

# Iterate through the cached control images
for row_idx, (img_path, _) in enumerate(selected_data):
    # Load original image and convert from BGR to RGB format
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Plot original reference frame in the first column (index 0)
    axes[row_idx, 0].imshow(img_rgb)
    if row_idx == 0:
        axes[row_idx, 0].set_title("Original", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Loop across the 5 defined noise intensity values and render them
    for col_idx, std_val in enumerate(noise_stds, start=1):
        # Apply Gaussian noise distortion via the custom pipeline
        noisy_img_rgb = add_gaussian_noise(img_rgb, std_val)

        # Render the distorted frame in the corresponding grid cell
        axes[row_idx, col_idx].imshow(noisy_img_rgb)

        # Append dynamic sub-titles only to the top row to maintain layout neatness
        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"Noise STD: {std_val}", fontsize=14)
        axes[row_idx, col_idx].axis('off')

# Optimize layout padding and display the visual grid
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 2: Mask R-CNN Gaussian Noise Degradation Analysis (Precision & Recall)

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision.transforms import functional as F

print("Step 3 & 4: Applying Mask R-CNN to Manual Gaussian Noise & Calculating Metrics...")

# --- 1. Environment & Model Setup ---
# Dynamically allocate hardware accelerator
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Working on device: {device}")

if 'model_maskrcnn' not in locals():
    print("Loading Mask R-CNN model...")
    model_maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
    model_maskrcnn.eval()

# Force the model to the active compute device for each execution
model_maskrcnn.to(device)

# --- 2. Custom Gaussian Noise Generator ---
def add_gaussian_noise(image, std):
    # Cast to float32 to prevent overflow/underflow clipping artifacts
    img_float = image.astype(np.float32)
    noise = np.random.normal(0, std, img_float.shape)
    noisy_image = img_float + noise
    # Clip to valid [0, 255] pixel range and cast back to uint8
    noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)
    return noisy_image

# --- 3. Academic Metric Computations ---
def compute_iou(box1, box2):
    # Calculate Intersection over Union (IoU) between two bounding boxes
    x_left, y_top = max(box1[0], box2[0]), max(box1[1], box2[1])
    x_right, y_bottom = min(box1[2], box2[2]), min(box1[3], box2[3])
    if x_right < x_left or y_bottom < y_top: return 0.0

    intersection = (x_right - x_left) * (y_bottom - y_top)
    return intersection / float(((box1[2]-box1[0])*(box1[3]-box1[1])) + ((box2[2]-box2[0])*(box2[3]-box2[1])) - intersection)

def calculate_precision_recall(base_boxes, dist_boxes, iou_threshold=0.5):
    # Compute precision and recall by matching predicted boxes to ground-truth baseline
    if len(base_boxes) == 0 and len(dist_boxes) == 0: return 1.0, 1.0
    if len(base_boxes) == 0 or len(dist_boxes) == 0: return 0.0, 0.0

    matched_gt, matched_pred = set(), set()
    for p_idx, p_box in enumerate(dist_boxes):
        best_iou, best_g_idx = 0, -1
        for g_idx, g_box in enumerate(base_boxes):
            if g_idx in matched_gt: continue
            iou = compute_iou(p_box, g_box)
            if iou > best_iou: best_iou, best_g_idx = iou, g_idx

        # Register a valid detection if the IoU exceeds the defined threshold
        if best_iou > iou_threshold:
            matched_gt.add(best_g_idx)
            matched_pred.add(p_idx)

    tp = len(matched_gt)
    fp = len(dist_boxes) - tp
    fn = len(base_boxes) - tp
    return (tp / (tp + fn) if (tp + fn) > 0 else 0.0), (tp / (tp + fp) if (tp + fp) > 0 else 0.0)

def compute_snr(clean_rgb, noisy_rgb):
    # Quantify distortion severity using empirical Signal-to-Noise Ratio (SNR)
    clean, noise_img = clean_rgb.astype(np.float64), noisy_rgb.astype(np.float64)
    noise = clean - noise_img
    noise_power = np.mean(noise ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)

# --- Optimized Inference Pipeline (NMS + Ego-Vehicle Filter) ---
def get_strict_model_outputs(img_rgb, threshold=0.5, iou_nms_threshold=0.3):
    h, w = img_rgb.shape[:2]
    img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model_maskrcnn(img_tensor)[0]

    # Primary spatial filtering based on minimum confidence threshold
    keep = prediction['scores'] > threshold
    boxes = prediction['boxes'][keep]
    scores = prediction['scores'][keep]
    masks = prediction['masks'][keep]

    if len(boxes) == 0:
        return img_rgb.copy(), []

    # Inject Non-Maximum Suppression (NMS) to prevent duplicate overlapping boxes
    keep_nms = torchvision.ops.nms(boxes, scores, iou_threshold=iou_nms_threshold)
    boxes = boxes[keep_nms]
    masks = masks[keep_nms]

    # Spatial Ego-Vehicle filtering via bounding box coordinates
    final_boxes = []
    final_masks = []

    for i in range(len(boxes)):
        box = boxes[i]
        y_center = (box[1].item() + box[3].item()) / 2

        # Exclude objects whose geometric center falls in the bottom 15% of the frame
        if y_center < h * 0.85:
            final_boxes.append(box.cpu().numpy())
            final_masks.append(masks[i].cpu().numpy())

    # Overlay predicted masks for visualization
    res_img = img_rgb.copy()
    for m in final_masks:
        color = np.random.randint(0, 255, 3, dtype=np.uint8)
        res_img[m[0] > 0.5] = res_img[m[0] > 0.5] * 0.5 + color * 0.5

    return res_img.astype(np.uint8), final_boxes


# --- 4. Main Execution: Visual Grid & Metric Calculation ---
noise_stds = [10, 30, 60, 100, 150]

m_snr, m_recall, m_precision = [], [], []

fig_vis, axes = plt.subplots(len(selected_data), 6, figsize=(24, 15))
fig_vis.suptitle('Mask R-CNN Detections under Manual Gaussian Noise\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.98)

# Establish pristine baseline metrics
baseline_data = []
for row_idx, (img_path, _) in enumerate(selected_data):
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    base_res, base_boxes = get_strict_model_outputs(img_rgb)
    baseline_data.append((img_rgb, base_boxes))

    axes[row_idx, 0].imshow(base_res)
    if row_idx == 0: axes[row_idx, 0].set_title(f"Baseline\nObjects: {len(base_boxes)}", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

print("Applying model on noisy images...")
for col_idx, std_val in enumerate(noise_stds):
    l_snr, l_recall, l_precision = [], [], []

    for row_idx, (img_rgb, base_boxes) in enumerate(baseline_data):
        noisy_img = add_gaussian_noise(img_rgb, std_val)
        noisy_res, noisy_boxes = get_strict_model_outputs(noisy_img)

        # Render degraded inference outputs
        axes[row_idx, col_idx + 1].imshow(noisy_res)
        if row_idx == 0: axes[row_idx, col_idx + 1].set_title(f"Noise STD: {std_val}\nObjects: {len(noisy_boxes)}", fontsize=14)
        axes[row_idx, col_idx + 1].axis('off')

        # Compute empirical performance metrics
        l_snr.append(compute_snr(img_rgb, noisy_img))
        rec, prec = calculate_precision_recall(base_boxes, noisy_boxes)
        l_recall.append(rec)
        l_precision.append(prec)

    # Aggregate mean metrics for current noise level
    m_snr.append(np.mean(l_snr))
    m_recall.append(np.mean(l_recall))
    m_precision.append(np.mean(l_precision))

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# --- 5. Plot Degradation Curve (Dual-Axis) ---
print("Plotting Precision/Recall Degradation Graph...")
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Recall on primary Y-axis
color = 'tab:red'
ax1.set_xlabel(r'SNR (dB) $\rightarrow$ Worse Quality (More Noise)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean Recall', color=color, fontsize=12, fontweight='bold')
ax1.plot(m_snr, m_recall, marker='o', color=color, linewidth=2.5, label='Recall')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(-0.05, 1.1)
ax1.invert_xaxis() # Decreasing SNR maps to increasing noise moving rightward

# Plot Precision on secondary Y-axis
ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Mean Precision', color=color, fontsize=12, fontweight='bold')
ax2.plot(m_snr, m_precision, marker='s', color=color, linewidth=2.5, linestyle='--', label='Precision')
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(-0.05, 1.1)

# Annotate specific standard deviation steps
for i, std_val in enumerate(noise_stds):
    ax1.annotate(f"STD={std_val}", (m_snr[i], m_recall[i]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, fontweight='bold')

# Graph aesthetics
plt.title("Mask R-CNN Metrics vs SNR under Custom Gaussian Noise\nProject by: Ayelet & Noa", fontsize=14, fontweight='bold', pad=15)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#@title Task 2: Mask R-CNN Recovery Attempt - Low-Pass Filter vs. Gaussian Noise

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision.transforms import functional as F

print("Step 5: Applying Low-Pass Filter (Gaussian Blur) Enhancement...")

# --- 1. Environment & Model Setup ---
# Dynamically allocate hardware compute device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Working on device: {device}")

if 'model_maskrcnn' not in locals():
    print("Loading Mask R-CNN model...")
    model_maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
    model_maskrcnn.eval()

model_maskrcnn.to(device)

# --- 2. Image Processing & Noise Functions ---
def add_gaussian_noise(image, std):
    # Mathematically inject noise using float32 to prevent clipping artifacts
    img_float = image.astype(np.float32)
    noise = np.random.normal(0, std, img_float.shape)
    noisy_image = np.clip(img_float + noise, 0, 255).astype(np.uint8)
    return noisy_image

def apply_low_pass_filter(noisy_img):
    # Apply a 5x5 Gaussian Blur kernel to smooth out high-frequency noise
    return cv2.GaussianBlur(noisy_img, (5, 5), 0)

# --- 3. Metric Evaluation Functions ---
def compute_iou(box1, box2):
    # Calculate Intersection over Union (IoU) between bounding boxes
    x_left, y_top = max(box1[0], box2[0]), max(box1[1], box2[1])
    x_right, y_bottom = min(box1[2], box2[2]), min(box1[3], box2[3])
    if x_right < x_left or y_bottom < y_top: return 0.0

    intersection = (x_right - x_left) * (y_bottom - y_top)
    return intersection / float(((box1[2]-box1[0])*(box1[3]-box1[1])) + ((box2[2]-box2[0])*(box2[3]-box2[1])) - intersection)

def calculate_precision_recall(base_boxes, dist_boxes, iou_threshold=0.5):
    # Calculate performance metrics by matching predicted boxes to the baseline
    if len(base_boxes) == 0 and len(dist_boxes) == 0: return 1.0, 1.0
    if len(base_boxes) == 0 or len(dist_boxes) == 0: return 0.0, 0.0

    matched_gt, matched_pred = set(), set()
    for p_idx, p_box in enumerate(dist_boxes):
        best_iou, best_g_idx = 0, -1
        for g_idx, g_box in enumerate(base_boxes):
            if g_idx in matched_gt: continue
            iou = compute_iou(p_box, g_box)
            if iou > best_iou: best_iou, best_g_idx = iou, g_idx

        if best_iou > iou_threshold:
            matched_gt.add(best_g_idx)
            matched_pred.add(p_idx)

    tp = len(matched_gt)
    fp = len(dist_boxes) - tp
    fn = len(base_boxes) - tp
    return (tp / (tp + fn) if (tp + fn) > 0 else 0.0), (tp / (tp + fp) if (tp + fp) > 0 else 0.0)

def compute_snr(clean_rgb, dist_rgb):
    # Calculate empirical Signal-to-Noise Ratio (SNR)
    clean, dist_img = clean_rgb.astype(np.float64), dist_rgb.astype(np.float64)
    noise_power = np.mean((clean - dist_img) ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)

# --- Optimized Inference Pipeline (NMS + Ego-Vehicle Filter) ---
def get_strict_boxes(img_rgb, threshold=0.5, iou_nms_threshold=0.3):
    h, w = img_rgb.shape[:2]
    img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model_maskrcnn(img_tensor)[0]

    # Primary confidence thresholding
    keep = prediction['scores'] > threshold
    boxes = prediction['boxes'][keep]
    scores = prediction['scores'][keep]

    if len(boxes) == 0:
        return []

    # Non-Maximum Suppression (NMS) to eliminate duplicate overlapping boxes
    keep_nms = torchvision.ops.nms(boxes, scores, iou_threshold=iou_nms_threshold)
    boxes = boxes[keep_nms]

    # Spatial Ego-Vehicle filter (removes host vehicle hood from bottom of frame)
    final_boxes = []
    for i in range(len(boxes)):
        box = boxes[i]
        y_center = (box[1].item() + box[3].item()) / 2
        if y_center < h * 0.85:
            final_boxes.append(box.cpu().numpy())

    return np.array(final_boxes) if len(final_boxes) > 0 else []

# --- 4. Main Evaluation Sweep (Distorted vs Enhanced) ---
noise_stds = [10, 30, 60, 100, 150]
m_snr, m_rec_dist, m_prec_dist = [], [], []
m_rec_enh, m_prec_enh = [], []

print("Extracting baselines and processing sweeps...")
baseline_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    base_boxes = get_strict_boxes(img_rgb)
    baseline_data.append((img_rgb, base_boxes))

for std_val in noise_stds:
    l_snr, l_rec_d, l_prec_d, l_rec_e, l_prec_e = [], [], [], [], []
    for img_rgb, base_boxes in baseline_data:
        # Generate distortion (Gaussian Noise)
        noisy_img = add_gaussian_noise(img_rgb, std_val)
        noisy_boxes = get_strict_boxes(noisy_img)
        rec_d, prec_d = calculate_precision_recall(base_boxes, noisy_boxes)

        # Apply enhancement (Low-Pass Filter)
        enhanced_img = apply_low_pass_filter(noisy_img)
        enh_boxes = get_strict_boxes(enhanced_img)
        rec_e, prec_e = calculate_precision_recall(base_boxes, enh_boxes)

        l_snr.append(compute_snr(img_rgb, noisy_img))
        l_rec_d.append(rec_d)
        l_prec_d.append(prec_d)
        l_rec_e.append(rec_e)
        l_prec_e.append(prec_e)

    # Aggregate mean metrics
    m_snr.append(np.mean(l_snr))
    m_rec_dist.append(np.mean(l_rec_d))
    m_prec_dist.append(np.mean(l_prec_d))
    m_rec_enh.append(np.mean(l_rec_e))
    m_prec_enh.append(np.mean(l_prec_e))

# --- 5. Plotting Side-by-Side Comparative Graphs ---
print("Plotting comparative graphs...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Mask R-CNN Recovery Attempt: Gaussian Noise vs Low-Pass Filter\nProject by: Ayelet & Noa', fontsize=16, fontweight='bold')

# Left Graph: Recall Tracking
ax1.plot(m_snr, m_rec_dist, marker='o', color='#d62728', linewidth=2.5, label='Distorted (Noise Only)')
ax1.plot(m_snr, m_rec_enh, marker='s', color='#2ca02c', linewidth=2.5, label='Enhanced (Gaussian Blur)')
ax1.set_xlabel(r'SNR (dB) $\rightarrow$ Worse Quality', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean Recall', fontsize=12, fontweight='bold')
ax1.set_title('Recall Comparison', fontsize=14)
ax1.axhline(y=1.0, color='blue', linestyle='--', alpha=0.5, label='Baseline')
ax1.set_ylim(-0.05, 1.1)
ax1.invert_xaxis()
ax1.grid(True, alpha=0.3)
ax1.legend(loc='lower left')

# Annotate STD limits
for i, std_val in enumerate(noise_stds):
    ax1.annotate(f"STD={std_val}", (m_snr[i], max(m_rec_dist[i], m_rec_enh[i])), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9)

# Right Graph: Precision Tracking
ax2.plot(m_snr, m_prec_dist, marker='o', color='#1f77b4', linewidth=2.5, linestyle='--', label='Distorted Precision')
ax2.plot(m_snr, m_prec_enh, marker='s', color='#9467bd', linewidth=2.5, linestyle='--', label='Enhanced Precision')
ax2.set_xlabel(r'SNR (dB) $\rightarrow$ Worse Quality', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean Precision', fontsize=12, fontweight='bold')
ax2.set_title('Precision Comparison', fontsize=14)
ax2.axhline(y=1.0, color='blue', linestyle='--', alpha=0.5)
ax2.set_ylim(-0.05, 1.1)
ax2.invert_xaxis()
ax2.grid(True, alpha=0.3)
ax2.legend(loc='lower left')

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 2: Alpha Blending Analysis (Gaussian Noise vs Blur)

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision.transforms import functional as F

print("Step 5.5: Executing Lecturer's Challenge (Alpha Blending) for Gaussian Noise...")

# --- 1. Environment & Model Setup ---
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

if 'model_maskrcnn' not in locals():
    print("Loading Mask R-CNN model...")
    model_maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
    model_maskrcnn.eval()

model_maskrcnn.to(device)

# --- 2. Image Processing & Metric Evaluation Utilities ---
def add_gaussian_noise(image, std):
    # Mathematically inject noise using float32 to prevent clipping artifacts
    img_float = image.astype(np.float32)
    noise = np.random.normal(0, std, img_float.shape)
    return np.clip(img_float + noise, 0, 255).astype(np.uint8)

def apply_low_pass_filter(noisy_img):
    # Apply a 5x5 Gaussian Blur kernel to suppress high-frequency noise elements
    return cv2.GaussianBlur(noisy_img, (5, 5), 0)

def compute_iou(box1, box2):
    # Calculate Intersection over Union (IoU) between bounding boxes
    x_left, y_top = max(box1[0], box2[0]), max(box1[1], box2[1])
    x_right, y_bottom = min(box1[2], box2[2]), min(box1[3], box2[3])
    if x_right < x_left or y_bottom < y_top: return 0.0

    intersection = (x_right - x_left) * (y_bottom - y_top)
    return intersection / float(((box1[2]-box1[0])*(box1[3]-box1[1])) + ((box2[2]-box2[0])*(box2[3]-box2[1])) - intersection)

def calculate_precision_recall(base_boxes, dist_boxes, iou_threshold=0.5):
    # Calculate Precision and Recall by matching inference predictions to the control baseline
    if len(base_boxes) == 0 and len(dist_boxes) == 0: return 1.0, 1.0
    if len(base_boxes) == 0 or len(dist_boxes) == 0: return 0.0, 0.0

    matched_gt, matched_pred = set(), set()
    for p_idx, p_box in enumerate(dist_boxes):
        best_iou, best_g_idx = 0, -1
        for g_idx, g_box in enumerate(base_boxes):
            if g_idx in matched_gt: continue
            iou = compute_iou(p_box, g_box)
            if iou > best_iou: best_iou, best_g_idx = iou, g_idx
        if best_iou > iou_threshold:
            matched_gt.add(best_g_idx)
            matched_pred.add(p_idx)

    tp = len(matched_gt)
    fp = len(dist_boxes) - tp
    fn = len(base_boxes) - tp
    return (tp / (tp + fn) if (tp + fn) > 0 else 0.0), (tp / (tp + fp) if (tp + fp) > 0 else 0.0)

# --- Optimized Inference Pipeline (Filtered Baseline) ---
def get_strict_boxes(img_rgb, threshold=0.5, iou_nms_threshold=0.3):
    h, w = img_rgb.shape[:2]
    img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model_maskrcnn(img_tensor)[0]

    # Primary spatial filtering based on minimum confidence threshold
    keep = prediction['scores'] > threshold
    boxes = prediction['boxes'][keep]
    scores = prediction['scores'][keep]

    if len(boxes) == 0:
        return []

    # Inject Non-Maximum Suppression (NMS) to prevent duplicate overlapping boxes
    keep_nms = torchvision.ops.nms(boxes, scores, iou_threshold=iou_nms_threshold)
    boxes = boxes[keep_nms]

    # Post-processing Ego-Vehicle filter (eliminates host vehicle hood detections)
    final_boxes = []
    for i in range(len(boxes)):
        box = boxes[i]
        y_center = (box[1].item() + box[3].item()) / 2
        if y_center < h * 0.85:
            final_boxes.append(box.cpu().numpy())

    return np.array(final_boxes) if len(final_boxes) > 0 else []

# --- 3. Experimental Parameters ---
alphas = [0.0, 0.3, 0.5, 0.7, 1.0]
# Select the most extreme noise level to stress-test the filter's performance
std_hard = 150

results_recall = []
results_precision = []

print("Calculating Baseline...")
base_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    base_boxes = get_strict_boxes(img_rgb)
    base_data.append((img_rgb, base_boxes))

# --- 4. Alpha Blending Execution Sweep ---
print(f"Running Alpha Blending sweep on extreme noise (STD={std_hard})...")
best_alpha_img_data = None

for alpha in alphas:
    l_recall = []
    l_precision = []

    for img_rgb, base_boxes in base_data:
        # 1. Induce maximum noise distortion
        noisy_img = add_gaussian_noise(img_rgb, std_hard)
        # 2. Apply full low-pass filter restoration
        enh_img = apply_low_pass_filter(noisy_img)

        # 3. Perform linear Alpha Blending interpolation
        blended_img = cv2.addWeighted(enh_img, alpha, noisy_img, 1.0 - alpha, 0)

        # 4. Execute Mask R-CNN inference on the blended image
        boxes = get_strict_boxes(blended_img)

        rec, prec = calculate_precision_recall(base_boxes, boxes)
        l_recall.append(rec)
        l_precision.append(prec)

        # Cache the fully enhanced image state for downstream visual inspection
        if alpha == 1.0:
            best_alpha_img_data = (blended_img, boxes)

    results_recall.append(np.mean(l_recall))
    results_precision.append(np.mean(l_precision))

print("\n--- Alpha Blending Results (Noise STD=150) ---")
for i, a in enumerate(alphas):
    print(f"Alpha {a:.1f} | Recall: {results_recall[i]:.3f} | Precision: {results_precision[i]:.3f}")

# --- 5. Plotting Dual-Axis Alpha Blending Graph ---
fig, ax1 = plt.subplots(figsize=(9, 5))

# Plot Recall on primary Y-axis
color = 'tab:red'
ax1.set_xlabel('Alpha (0 = 100% Noisy, 1 = 100% Blurred/Enhanced)', fontweight='bold')
ax1.set_ylabel('Mean Recall', color=color, fontweight='bold')
ax1.plot(alphas, results_recall, marker='o', color=color, linewidth=2.5, label='Recall')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(-0.05, 1.05)

# Plot Precision on secondary Y-axis
ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Mean Precision', color=color, fontweight='bold')
ax2.plot(alphas, results_precision, marker='s', color=color, linewidth=2.5, linestyle='--', label='Precision')
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(-0.05, 1.05)

fig.suptitle('Lecturer Challenge: Alpha Blending at STD=150 (Gaussian Noise)\nProject by: Ayelet & Noa', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.grid(True, alpha=0.3)
plt.show()

# --- 6. Visual Inspection of the Filtered State ---
print("\n--- Visual Inspection: 100% Enhanced (Alpha=1.0) ---")
blended_img, pred_boxes = best_alpha_img_data
fig_v, ax_v = plt.subplots(figsize=(10, 6))
ax_v.imshow(blended_img)

# Render prediction bounding boxes
for box in pred_boxes:
    rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], fill=False, color='green', linewidth=2)
    ax_v.add_patch(rect)

ax_v.set_title("Visual Inspection (Alpha=1.0 - Full Gaussian Blur)\nNotice how the 'snow' is smoothed out, preventing false positive detections.", fontweight='bold')
ax_v.axis('off')
plt.show()

In [ ]:
#@title Task 2: Mixed Dataset Generation for Gaussian Noise Fine-Tuning

import os
import torch
import torchvision
from torchvision.transforms import functional as F
import cv2
import random
import numpy as np
from pathlib import Path
import torchvision.ops as ops

print("Step 6: Creating the Ultimate Mixed Dataset for Gaussian Noise...")

# --- 1. Environment & Model Initialization ---
# Dynamically allocate hardware compute device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Load the pristine baseline model to generate ground-truth pseudo-labels
if 'model_maskrcnn' not in locals():
    print("Loading pristine Mask R-CNN model for pseudo-labeling...")
    model_maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
    model_maskrcnn.eval()

model_maskrcnn.to(device)

# Ensure control images are properly cached to prevent data leakage
if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing! Please re-run the clean image selection block.")

# Fallback recovery in case the global image list was cleared from memory
if 'all_images' not in locals():
    print("Recovering all_images list...")
    images_dir = Path('/content/bdd100k/10k/test')
    all_images = list(images_dir.glob('*.jpg'))
    random.shuffle(all_images)

# --- 2. Target Directory Preparation ---
dataset_dir = Path('/content/gaussian_dataset')
dataset_dir.mkdir(parents=True, exist_ok=True)
(dataset_dir / 'images').mkdir(exist_ok=True)
(dataset_dir / 'targets').mkdir(exist_ok=True)

# Select 1,000 exclusive images for the training set (strictly excluding the 5 control images)
test_image_paths = [str(p) for p, _ in selected_data]
train_images = [p for p in all_images if str(p) not in test_image_paths][:1000]

valid_count = 0
clean_count = 0
noisy_count = 0

print("Extracting Pseudo-labels and injecting Gaussian Noise. Please wait...")

for i, img_path in enumerate(train_images):
    img = cv2.imread(str(img_path))
    if img is None: continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    # --- A. Extract Pseudo-labels using our optimized pipeline (NMS + Ego-Vehicle Filter) ---
    img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model_maskrcnn(img_tensor)[0]

    # Primary confidence thresholding
    keep = prediction['scores'] > 0.5
    boxes = prediction['boxes'][keep]
    scores = prediction['scores'][keep]
    masks = prediction['masks'][keep]
    labels = prediction['labels'][keep]

    if len(boxes) == 0: continue

    # Apply Non-Maximum Suppression (NMS) to eliminate duplicate bounding boxes
    keep_nms = ops.nms(boxes, scores, iou_threshold=0.3)
    boxes = boxes[keep_nms]
    masks = masks[keep_nms]
    labels = labels[keep_nms]

    # Apply spatial Ego-Vehicle filter (exclude objects in the bottom 15% of the frame)
    final_keep = []
    for j in range(len(boxes)):
        y_center = (boxes[j][1].item() + boxes[j][3].item()) / 2
        if y_center < h * 0.85:
            final_keep.append(j)

    if len(final_keep) == 0: continue

    boxes = boxes[final_keep]
    masks = masks[final_keep]
    labels = labels[final_keep]

    # --- B. Generate Mixed Dataset (50% Pristine, 50% Gaussian Noise) ---
    if random.random() < 0.5:
        final_img_rgb = img_rgb
        clean_count += 1
    else:
        # Randomize noise standard deviation between 50 (Medium) and 150 (Extreme)
        std_val = random.uniform(50, 150)
        img_float = img_rgb.astype(np.float32)
        noise = np.random.normal(0, std_val, img_float.shape)
        final_img_rgb = np.clip(img_float + noise, 0, 255).astype(np.uint8)
        noisy_count += 1

    # --- C. Export Image and Target Tensors ---
    img_out_path = dataset_dir / 'images' / f"img_{i}.jpg"
    target_out_path = dataset_dir / 'targets' / f"tgt_{i}.pt"

    cv2.imwrite(str(img_out_path), cv2.cvtColor(final_img_rgb, cv2.COLOR_RGB2BGR))

    # Package ground-truth tensors into a dictionary for PyTorch training
    target = {
        "boxes": boxes.cpu(),
        "masks": masks.cpu(),
        "labels": labels.cpu()
    }
    torch.save(target, target_out_path)

    valid_count += 1

    if valid_count % 100 == 0:
        print(f"Processed {valid_count} valid images...")

print(f"\nGaussian Dataset Ready! Total: {valid_count} images ({clean_count} Clean, {noisy_count} Noisy).")

In [ ]:
#@title Task 2: Mask R-CNN Fine-Tuning & Grand Finale Evaluation

import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from torchvision.transforms import functional as F
import torchvision
import torchvision.ops as ops

# ==========================================
# Environment & Variable Security
# ==========================================
# Dynamically allocate hardware compute device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Working on device: {device}")

# Ensure control images are properly cached to prevent data leakage
if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing! Please re-run the clean image selection block.")

# --- Standalone Utility Functions ---
def compute_iou(box1, box2):
    # Calculate Intersection over Union (IoU) between two bounding boxes
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])
    if x_right < x_left or y_bottom < y_top: return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return intersection_area / float(box1_area + box2_area - intersection_area)

def calculate_precision_recall(base_boxes, dist_boxes, iou_threshold=0.5):
    # Calculate precision and recall by matching predicted boxes to ground-truth baseline
    if len(base_boxes) == 0 and len(dist_boxes) == 0: return 1.0, 1.0
    if len(base_boxes) == 0 or len(dist_boxes) == 0: return 0.0, 0.0

    matched_gt = set()
    for p_box in dist_boxes:
        best_iou = 0
        best_g_idx = -1
        for g_idx, g_box in enumerate(base_boxes):
            if g_idx in matched_gt: continue
            iou = compute_iou(p_box, g_box)
            if iou > best_iou:
                best_iou = iou
                best_g_idx = g_idx
        if best_iou > iou_threshold:
            matched_gt.add(best_g_idx)

    tp = len(matched_gt)
    fp = len(dist_boxes) - tp
    fn = len(base_boxes) - tp
    return (tp / (tp + fn) if (tp + fn) > 0 else 0.0), (tp / (tp + fp) if (tp + fp) > 0 else 0.0)

def add_gaussian_noise(image, std):
    # Mathematically inject noise using float32 to prevent clipping artifacts
    img_float = image.astype(np.float32)
    noise = np.random.normal(0, std, img_float.shape)
    return np.clip(img_float + noise, 0, 255).astype(np.uint8)

def apply_low_pass_filter(noisy_img):
    # Apply a 5x5 Gaussian Blur kernel to smooth out high-frequency noise
    return cv2.GaussianBlur(noisy_img, (5, 5), 0)

# --- Optimized Inference Function (NMS + Ego-Vehicle Filter) ---
def get_strict_boxes_from_model(model_obj, img_rgb, current_device, threshold=0.5, iou_nms_threshold=0.3):
    h, w = img_rgb.shape[:2]
    img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(current_device)
    model_obj.to(current_device)

    with torch.no_grad():
        prediction = model_obj(img_tensor)[0]

    # Primary confidence thresholding
    keep = prediction['scores'] > threshold
    boxes = prediction['boxes'][keep]
    scores = prediction['scores'][keep]

    if len(boxes) == 0: return np.array([])

    # Apply Non-Maximum Suppression (NMS) to eliminate duplicate bounding boxes
    keep_nms = ops.nms(boxes, scores, iou_threshold=iou_nms_threshold)
    boxes = boxes[keep_nms]

    # Apply spatial Ego-Vehicle filter (exclude objects in the bottom 15% of the frame)
    final_boxes = []
    for box in boxes:
        y_center = (box[1].item() + box[3].item()) / 2
        if y_center < h * 0.85:
            final_boxes.append(box.cpu().numpy())

    return np.array(final_boxes) if len(final_boxes) > 0 else np.array([])

# ==========================================
# Step 7: Dataset Definition and Fine-Tuning Loop
# ==========================================
class GaussianNoiseDataset(Dataset):
    def __init__(self, dataset_dir):
        self.dataset_dir = Path(dataset_dir)
        self.img_files = sorted(list((self.dataset_dir / 'images').glob('*.jpg')))

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        idx_str = img_path.stem.split('_')[1]
        tgt_path = self.dataset_dir / 'targets' / f"tgt_{idx_str}.pt"

        # Load and convert image
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_tensor = F.to_tensor(img_rgb)

        # Load ground-truth target tensors
        target = torch.load(tgt_path)
        # Binarize masks for PyTorch compatibility
        target['masks'] = (target['masks'] > 0.5).squeeze(1).to(torch.uint8)

        return img_tensor, target

print("Setting up PyTorch DataLoader...")
train_dataset = GaussianNoiseDataset('/content/gaussian_dataset')
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

print("Initializing model for fine-tuning...")
finetuned_maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
finetuned_maskrcnn.to(device)

# Configure optimizer with a conservative learning rate to prevent catastrophic forgetting
params = [p for p in finetuned_maskrcnn.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=1e-5, weight_decay=0.0005)

EPOCHS = 5
total_batches = len(train_loader)
print(f"Starting Fine-Tuning Loop ({EPOCHS} Epochs) for Gaussian Noise...")
finetuned_maskrcnn.train()

for epoch in range(EPOCHS):
    epoch_loss = 0
    print(f"\n--- Starting Epoch {epoch+1}/{EPOCHS} ---")

    for batch_idx, (images, targets) in enumerate(train_loader):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass and loss calculation
        loss_dict = finetuned_maskrcnn(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Backward pass and optimization step
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        epoch_loss += losses.item()

        # Live progress reporting
        if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == total_batches:
            avg_batch_loss = epoch_loss / (batch_idx + 1)
            print(f"  -> Batch [{batch_idx+1}/{total_batches}] completed. Current Average Loss: {avg_batch_loss:.4f}")

    print(f"=== Epoch {epoch+1} Completed! Final Average Loss: {epoch_loss/total_batches:.4f} ===")

print("\nFine-Tuning Complete! Switching model to evaluation mode.")
finetuned_maskrcnn.eval()

# ==========================================
# Step 8: The Grand Finale - Head-to-Head Evaluation
# ==========================================
print("\nStep 8: Running Final Evaluation on extreme noise STD=150...")

# Load a pristine Mask R-CNN model for an unbiased baseline comparison
fresh_original_model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
fresh_original_model.eval()

std_hard = 150
rec_distorted, prec_distorted = [], []
rec_enhanced, prec_enhanced = [], []
rec_finetuned, prec_finetuned = [], []

print("Extracting test baseline data...")
baseline_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    base_boxes = get_strict_boxes_from_model(fresh_original_model, img_rgb, device)
    baseline_data.append((img_rgb, base_boxes))

print("Evaluating all three setups...")
for img_rgb, base_boxes in baseline_data:
    # 1. Configuration 1: Distorted Image on Original Model
    noisy_img = add_gaussian_noise(img_rgb, std_hard)
    boxes_dist = get_strict_boxes_from_model(fresh_original_model, noisy_img, device)
    r_d, p_d = calculate_precision_recall(base_boxes, boxes_dist)
    rec_distorted.append(r_d)
    prec_distorted.append(p_d)

    # 2. Configuration 2: Enhanced Image (Low-Pass Filter) on Original Model
    enhanced_img = apply_low_pass_filter(noisy_img)
    boxes_enh = get_strict_boxes_from_model(fresh_original_model, enhanced_img, device)
    r_e, p_e = calculate_precision_recall(base_boxes, boxes_enh)
    rec_enhanced.append(r_e)
    prec_enhanced.append(p_e)

    # 3. Configuration 3: Distorted Image on Fine-Tuned Model
    boxes_ft = get_strict_boxes_from_model(finetuned_maskrcnn, noisy_img, device)
    r_f, p_f = calculate_precision_recall(base_boxes, boxes_ft)
    rec_finetuned.append(r_f)
    prec_finetuned.append(p_f)

# Aggregate mean metrics for final visualization
mean_rec = [np.mean(rec_distorted), np.mean(rec_enhanced), np.mean(rec_finetuned)]
mean_prec = [np.mean(prec_distorted), np.mean(prec_enhanced), np.mean(prec_finetuned)]

# --- Dual-Bar Chart Visualization ---
print("Plotting the Grand Finale Bar Chart...")
labels = ['Original Model\n(Distorted Image)', 'Original Model\n(Enhanced Image)', 'Fine-Tuned Model\n(Distorted Image)']
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, mean_rec, width, label='Recall', color='#d62728', edgecolor='black', linewidth=1.2)
rects2 = ax.bar(x + width/2, mean_prec, width, label='Precision', color='#1f77b4', edgecolor='black', linewidth=1.2)

# Reference line representing the clean baseline performance (1.00)
ax.axhline(y=1.0, color='blue', linestyle='--', linewidth=1.5, alpha=0.6, label='Clean Baseline (1.00)')

# Helper function to explicitly annotate metric values above the bars
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=11, fontweight='bold')

autolabel(rects1)
autolabel(rects2)

# Graph aesthetics and structural formatting
ax.set_ylabel('Scores', fontsize=12, fontweight='bold')
ax.set_title(f'The Grand Finale: Mask R-CNN Performance at Extreme Noise (STD={std_hard})\nProject by: Ayelet & Noa', fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.2)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
#@title Step 9: Side-by-Side Noise Evaluation Sweep (Line Charts)

import numpy as np
import matplotlib.pyplot as plt
import cv2

print("Step 9: Running Full SNR Sweep for Line Charts...")

# --- 1. Signal Processing Utility (SNR) ---
def compute_snr(clean_rgb, dist_rgb):
    clean, dist_img = clean_rgb.astype(np.float64), dist_rgb.astype(np.float64)
    noise_power = np.mean((clean - dist_img) ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)

# --- 2. Main Evaluation Sweep (Distorted vs Enhanced vs Fine-Tuned) ---
noise_stds = [10, 30, 60, 100, 150]
m_snr, m_rec_dist, m_prec_dist = [], [], []
m_rec_enh, m_prec_enh = [], []
m_rec_ft, m_prec_ft = [], []

print("Extracting baselines from original model...")
# We use the baseline_data that was already calculated in Step 8!
# If it doesn't exist, we recalculate it:
if 'baseline_data' not in locals():
    baseline_data = []
    for img_path, _ in selected_data:
        img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        base_boxes = get_strict_boxes_from_model(fresh_original_model, img_rgb, device)
        baseline_data.append((img_rgb, base_boxes))

print("Processing sweeps across all noise levels...")
for std_val in noise_stds:
    l_snr = []
    l_rec_d, l_prec_d = [], []
    l_rec_e, l_prec_e = [], []
    l_rec_f, l_prec_f = [], []

    for img_rgb, base_boxes in baseline_data:
        # Generate distorted image (Gaussian Noise)
        noisy_img = add_gaussian_noise(img_rgb, std_val)
        l_snr.append(compute_snr(img_rgb, noisy_img))

        # 1. Distorted Image on Original Model
        noisy_boxes = get_strict_boxes_from_model(fresh_original_model, noisy_img, device)
        rec_d, prec_d = calculate_precision_recall(base_boxes, noisy_boxes)
        l_rec_d.append(rec_d); l_prec_d.append(prec_d)

        # 2. Enhanced Image (Low-Pass Filter) on Original Model
        enhanced_img = apply_low_pass_filter(noisy_img)
        enh_boxes = get_strict_boxes_from_model(fresh_original_model, enhanced_img, device)
        rec_e, prec_e = calculate_precision_recall(base_boxes, enh_boxes)
        l_rec_e.append(rec_e); l_prec_e.append(prec_e)

        # 3. Distorted Image on Fine-Tuned Model
        ft_boxes = get_strict_boxes_from_model(finetuned_maskrcnn, noisy_img, device)
        rec_f, prec_f = calculate_precision_recall(base_boxes, ft_boxes)
        l_rec_f.append(rec_f); l_prec_f.append(prec_f)

    # Aggregate mean metrics
    m_snr.append(np.mean(l_snr))
    m_rec_dist.append(np.mean(l_rec_d)); m_prec_dist.append(np.mean(l_prec_d))
    m_rec_enh.append(np.mean(l_rec_e)); m_prec_enh.append(np.mean(l_prec_e))
    m_rec_ft.append(np.mean(l_rec_f)); m_prec_ft.append(np.mean(l_prec_f))
    print(f"Level STD={std_val}: SNR={m_snr[-1]:.1f}dB | FT Recall={m_rec_ft[-1]:.2f} | FT Precision={m_prec_ft[-1]:.2f}")


# --- 3. Plotting Side-by-Side Comparative Graphs ---
print("Plotting comparative side-by-side graphs...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('The Grand Finale: Gaussian Noise & Fine-Tuning Performance\nProject by: Ayelet & Noa', fontsize=16, fontweight='bold')

# --- Left Graph: Recall Tracking ---
ax1.plot(m_snr, m_rec_dist, marker='o', color='#d62728', linewidth=2.5, label='Original Model (Distorted)')
ax1.plot(m_snr, m_rec_enh, marker='s', color='#2ca02c', linewidth=2.5, label='Original Model (Enhanced)')
ax1.plot(m_snr, m_rec_ft, marker='^', color='#9467bd', linewidth=2.5, label='Fine-Tuned Model (Distorted)')

ax1.set_xlabel(r'SNR (dB) $\rightarrow$ Worse Quality', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean Recall', fontsize=12, fontweight='bold')
ax1.set_title('Recall Comparison', fontsize=14)
ax1.axhline(y=1.0, color='blue', linestyle='--', alpha=0.5, label='Baseline')
ax1.set_ylim(-0.05, 1.15)
ax1.invert_xaxis()
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.legend(loc='lower left')

# Annotate STD values on the Recall Graph
for i, std_val in enumerate(noise_stds):
    highest_val = max(m_rec_dist[i], m_rec_enh[i], m_rec_ft[i])
    ax1.annotate(f"STD={std_val}", (m_snr[i], highest_val), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, fontweight='bold', color='#1f77b4')


# --- Right Graph: Precision Tracking ---
ax2.plot(m_snr, m_prec_dist, marker='o', color='#d62728', linewidth=2.5, label='Original Model (Distorted)')
ax2.plot(m_snr, m_prec_enh, marker='s', color='#2ca02c', linewidth=2.5, label='Original Model (Enhanced)')
ax2.plot(m_snr, m_prec_ft, marker='^', color='#9467bd', linewidth=2.5, label='Fine-Tuned Model (Distorted)')

ax2.set_xlabel(r'SNR (dB) $\rightarrow$ Worse Quality', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean Precision', fontsize=12, fontweight='bold')
ax2.set_title('Precision Comparison', fontsize=14)
ax2.axhline(y=1.0, color='blue', linestyle='--', alpha=0.5)
ax2.set_ylim(-0.05, 1.15)
ax2.invert_xaxis()
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.legend(loc='lower left')

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 2- Disstortion: severe JPEG


In [ ]:
#@title Task 2: Distortion Intensity Sweep - JPEG Compression (No Model)

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("Generating JPEG Compression Intensity Sweep (No Model)...")

# --- 1. Custom Degradation Function: JPEG Compression ---
def apply_jpeg_compression(img_rgb, quality):
    # Convert from RGB to BGR as required by OpenCV
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

    # Encode the image into memory using the specified JPEG quality parameter (1-100)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encimg = cv2.imencode('.jpg', img_bgr, encode_param)

    # Decode the compressed byte array back into an image array
    decimg = cv2.imdecode(encimg, 1)

    # Convert back to RGB for correct downstream rendering
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

# Ensure control images are properly cached to prevent execution errors
if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing! Please re-run the clean image selection block.")

# --- 2. Define Compression Levels ---
# Scale: 100 is lossless/perfect.
# We sweep down to 2 to induce extreme macro-blocking and high-frequency data loss.
jpeg_qualities = [50, 30, 15, 5, 2]

# --- 3. Generate Visual Evaluation Grid ---
# Initialize dynamic plot matrix: rows = # of images, cols = 1 baseline + 5 degradation levels
fig, axes = plt.subplots(len(selected_data), 6, figsize=(24, 15))
fig.suptitle('Distortion Intensity Sweep (JPEG Compression)\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.98)

# Iterate through the cached control images
for row_idx, (img_path, _) in enumerate(selected_data):
    # Load original image and convert to RGB format
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Column 0: Render original reference frame
    axes[row_idx, 0].imshow(img_rgb)
    if row_idx == 0:
        axes[row_idx, 0].set_title("Original\n(Quality: 100)", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Columns 1-5: Apply and render increasing compression levels
    for col_idx, quality in enumerate(jpeg_qualities, start=1):
        compressed_img = apply_jpeg_compression(img_rgb, quality)

        axes[row_idx, col_idx].imshow(compressed_img)

        # Append dynamic sub-titles exclusively to the top row for a clean layout
        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"JPEG Quality: {quality}", fontsize=14)
        axes[row_idx, col_idx].axis('off')

# Optimize layout padding and render the visual grid
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 2 - Mask R-CNN Detections & Metrics on JPEG Compression

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision.transforms import functional as F

print("Task 3 - Steps 3 & 4: Mask R-CNN Detections & Metrics on JPEG Compression...")

# --- 1. Environment & Model Setup ---
# Dynamically allocate hardware compute device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

if 'model_maskrcnn' not in locals():
    print("Loading Mask R-CNN model (Pristine version)...")
    model_maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
    model_maskrcnn.eval()

# Force the model to the active compute device for each execution
model_maskrcnn.to(device)

# Ensure control images are properly cached to prevent data leakage
if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing! Please re-run the clean image selection block.")

# --- 2. Custom Degradation Function: JPEG Compression ---
def apply_jpeg_compression(img_rgb, quality):
    # Convert RGB to BGR as required natively by OpenCV
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

    # Encode the image into memory using the specified JPEG quality parameter
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    result, encimg = cv2.imencode('.jpg', img_bgr, encode_param)

    # Decode the compressed byte array back into a readable image array
    decimg = cv2.imdecode(encimg, 1)

    # Convert back to RGB for correct downstream rendering
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

# --- 3. Academic Metric Computations ---
def compute_iou(box1, box2):
    # Calculate Intersection over Union (IoU) between two bounding boxes
    x_left, y_top = max(box1[0], box2[0]), max(box1[1], box2[1])
    x_right, y_bottom = min(box1[2], box2[2]), min(box1[3], box2[3])
    if x_right < x_left or y_bottom < y_top: return 0.0

    intersection = (x_right - x_left) * (y_bottom - y_top)
    return intersection / float(((box1[2]-box1[0])*(box1[3]-box1[1])) + ((box2[2]-box2[0])*(box2[3]-box2[1])) - intersection)

def calculate_precision_recall(base_boxes, dist_boxes, iou_threshold=0.5):
    # Compute precision and recall by matching predicted boxes to ground-truth baseline
    if len(base_boxes) == 0 and len(dist_boxes) == 0: return 1.0, 1.0
    if len(base_boxes) == 0 or len(dist_boxes) == 0: return 0.0, 0.0

    matched_gt, matched_pred = set(), set()
    for p_idx, p_box in enumerate(dist_boxes):
        best_iou, best_g_idx = 0, -1
        for g_idx, g_box in enumerate(base_boxes):
            if g_idx in matched_gt: continue
            iou = compute_iou(p_box, g_box)
            if iou > best_iou: best_iou, best_g_idx = iou, g_idx

        # Register a valid detection if the IoU exceeds the defined threshold
        if best_iou > iou_threshold:
            matched_gt.add(best_g_idx)
            matched_pred.add(p_idx)

    tp = len(matched_gt)
    fp = len(dist_boxes) - tp
    fn = len(base_boxes) - tp
    return (tp / (tp + fn) if (tp + fn) > 0 else 0.0), (tp / (tp + fp) if (tp + fp) > 0 else 0.0)

def compute_snr(clean_rgb, dist_rgb):
    # Quantify distortion severity using empirical Signal-to-Noise Ratio (SNR)
    clean, dist_img = clean_rgb.astype(np.float64), dist_rgb.astype(np.float64)
    noise_power = np.mean((clean - dist_img) ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)

# --- Optimized Inference Pipeline (NMS + Ego-Vehicle Filter) ---
def get_strict_model_outputs(img_rgb, threshold=0.5, iou_nms_threshold=0.3):
    h, w = img_rgb.shape[:2]
    img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model_maskrcnn(img_tensor)[0]

    # Primary spatial filtering based on minimum confidence threshold
    keep = prediction['scores'] > threshold
    boxes = prediction['boxes'][keep]
    scores = prediction['scores'][keep]
    masks = prediction['masks'][keep]

    if len(boxes) == 0:
        return img_rgb.copy(), []

    # Inject Non-Maximum Suppression (NMS) to prevent duplicate overlapping boxes
    keep_nms = torchvision.ops.nms(boxes, scores, iou_threshold=iou_nms_threshold)
    boxes = boxes[keep_nms]
    masks = masks[keep_nms]

    # Spatial Ego-Vehicle filtering via bounding box coordinates
    final_boxes = []
    final_masks = []

    for i in range(len(boxes)):
        box = boxes[i]
        y_center = (box[1].item() + box[3].item()) / 2

        # Exclude objects whose geometric center falls in the bottom 15% of the frame
        if y_center < h * 0.85:
            final_boxes.append(box.cpu().numpy())
            final_masks.append(masks[i].cpu().numpy())

    # Overlay predicted masks for visualization
    res_img = img_rgb.copy()
    for m in final_masks:
        color = np.random.randint(0, 255, 3, dtype=np.uint8)
        res_img[m[0] > 0.5] = res_img[m[0] > 0.5] * 0.5 + color * 0.5

    return res_img.astype(np.uint8), final_boxes

# --- 4. Main Execution: Visual Grid & Metric Calculation ---
jpeg_qualities = [50, 30, 15, 5, 2]
m_snr, m_recall, m_precision = [], [], []

fig_vis, axes = plt.subplots(len(selected_data), 6, figsize=(24, 15))
fig_vis.suptitle('Mask R-CNN Detections under JPEG Compression Artifacts\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.98)

print("Calculating Baseline Detections...")
baseline_data = []
for row_idx, (img_path, _) in enumerate(selected_data):
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    base_res, base_boxes = get_strict_model_outputs(img_rgb)
    baseline_data.append((img_rgb, base_boxes))

    axes[row_idx, 0].imshow(base_res)
    if row_idx == 0: axes[row_idx, 0].set_title(f"Baseline\nObjects: {len(base_boxes)}", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

print("Applying model on compressed images...")
for col_idx, quality in enumerate(jpeg_qualities):
    l_snr, l_recall, l_precision = [], [], []

    for row_idx, (img_rgb, base_boxes) in enumerate(baseline_data):
        # Apply JPEG compression distortion
        jpeg_img = apply_jpeg_compression(img_rgb, quality)
        jpeg_res, jpeg_boxes = get_strict_model_outputs(jpeg_img)

        # Render degraded inference outputs
        axes[row_idx, col_idx + 1].imshow(jpeg_res)
        if row_idx == 0: axes[row_idx, col_idx + 1].set_title(f"Quality: {quality}%\nObjects: {len(jpeg_boxes)}", fontsize=14)
        axes[row_idx, col_idx + 1].axis('off')

        # Compute empirical performance metrics
        l_snr.append(compute_snr(img_rgb, jpeg_img))
        rec, prec = calculate_precision_recall(base_boxes, jpeg_boxes)
        l_recall.append(rec)
        l_precision.append(prec)

    # Aggregate mean metrics for current compression level
    m_snr.append(np.mean(l_snr))
    m_recall.append(np.mean(l_recall))
    m_precision.append(np.mean(l_precision))

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# --- 5. Plot Degradation Curve (Dual-Axis) ---
print("Plotting Precision/Recall Degradation Graph for JPEG...")
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Recall on primary Y-axis
color = 'tab:red'
ax1.set_xlabel(r'SNR (dB) $\rightarrow$ Worse Quality (More Compression)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean Recall', color=color, fontsize=12, fontweight='bold')
ax1.plot(m_snr, m_recall, marker='o', color=color, linewidth=2.5, label='Recall')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(-0.05, 1.1)
ax1.invert_xaxis() # Decreasing SNR maps to increasing noise moving rightward

# Plot Precision on secondary Y-axis
ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Mean Precision', color=color, fontsize=12, fontweight='bold')
ax2.plot(m_snr, m_precision, marker='s', color=color, linewidth=2.5, linestyle='--', label='Precision')
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(-0.05, 1.1)

# Annotate specific JPEG quality steps
for i, quality in enumerate(jpeg_qualities):
    ax1.annotate(f"Q={quality}%", (m_snr[i], m_recall[i]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, fontweight='bold')

# Graph aesthetics
plt.title("Mask R-CNN Metrics vs SNR under JPEG Compression\nProject by: Ayelet & Noa", fontsize=14, fontweight='bold', pad=15)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#@title Task 2 - Bilateral Filter Recovery Attempt (JPEG Artifacts)

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision.transforms import functional as F

print("Task 3 - Step 5: Applying Bilateral Filter Enhancement to JPEG Artifacts...")

# --- 1. Environment & Model Setup ---
# Dynamically allocate hardware compute device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Working on device: {device}")

if 'model_maskrcnn' not in locals():
    print("Loading Mask R-CNN model...")
    model_maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
    model_maskrcnn.eval()

model_maskrcnn.to(device)

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing! Please re-run the clean image selection block.")

# --- 2. Signal Processing Functions (Degradation & Enhancement) ---
def apply_jpeg_compression(img_rgb, quality):
    # Simulate JPEG compression data loss
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encimg = cv2.imencode('.jpg', img_bgr, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

def apply_bilateral_filter(distorted_img):
    # Apply Bilateral Filter: highly effective at smoothing JPEG macroblocking
    # while preserving structural edges (d=9 neighborhood, sigma=75)
    return cv2.bilateralFilter(distorted_img, d=9, sigmaColor=75, sigmaSpace=75)

# --- 3. Academic Metric Evaluation ---
def compute_iou(box1, box2):
    # Calculate Intersection over Union (IoU) between bounding boxes
    x_left, y_top = max(box1[0], box2[0]), max(box1[1], box2[1])
    x_right, y_bottom = min(box1[2], box2[2]), min(box1[3], box2[3])
    if x_right < x_left or y_bottom < y_top: return 0.0

    intersection = (x_right - x_left) * (y_bottom - y_top)
    return intersection / float(((box1[2]-box1[0])*(box1[3]-box1[1])) + ((box2[2]-box2[0])*(box2[3]-box2[1])) - intersection)

def calculate_precision_recall(base_boxes, dist_boxes, iou_threshold=0.5):
    # Calculate performance metrics by matching inference to baseline
    if len(base_boxes) == 0 and len(dist_boxes) == 0: return 1.0, 1.0
    if len(base_boxes) == 0 or len(dist_boxes) == 0: return 0.0, 0.0

    matched_gt, matched_pred = set(), set()
    for p_idx, p_box in enumerate(dist_boxes):
        best_iou, best_g_idx = 0, -1
        for g_idx, g_box in enumerate(base_boxes):
            if g_idx in matched_gt: continue
            iou = compute_iou(p_box, g_box)
            if iou > best_iou: best_iou, best_g_idx = iou, g_idx

        if best_iou > iou_threshold:
            matched_gt.add(best_g_idx)
            matched_pred.add(p_idx)

    tp = len(matched_gt)
    fp = len(dist_boxes) - tp
    fn = len(base_boxes) - tp
    return (tp / (tp + fn) if (tp + fn) > 0 else 0.0), (tp / (tp + fp) if (tp + fp) > 0 else 0.0)

def compute_snr(clean_rgb, dist_rgb):
    # Quantify compression severity using empirical Signal-to-Noise Ratio (SNR)
    clean, dist_img = clean_rgb.astype(np.float64), dist_rgb.astype(np.float64)
    noise_power = np.mean((clean - dist_img) ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)

# --- Optimized Inference Pipeline (NMS + Ego-Vehicle Filter) ---
def get_strict_boxes(img_rgb, threshold=0.5, iou_nms_threshold=0.3):
    h, w = img_rgb.shape[:2]
    img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model_maskrcnn(img_tensor)[0]

    # Primary confidence thresholding
    keep = prediction['scores'] > threshold
    boxes = prediction['boxes'][keep]
    scores = prediction['scores'][keep]

    if len(boxes) == 0:
        return []

    # Non-Maximum Suppression (NMS) to eliminate duplicate overlapping predictions
    keep_nms = torchvision.ops.nms(boxes, scores, iou_threshold=iou_nms_threshold)
    boxes = boxes[keep_nms]

    # Spatial Ego-Vehicle filtering via bounding box coordinates
    final_boxes = []
    for i in range(len(boxes)):
        box = boxes[i]
        y_center = (box[1].item() + box[3].item()) / 2
        if y_center < h * 0.85:
            final_boxes.append(box.cpu().numpy())

    return np.array(final_boxes) if len(final_boxes) > 0 else []

# --- 4. Main Evaluation Sweep (Distorted vs Enhanced) ---
jpeg_qualities = [50, 30, 15, 5, 2]
m_snr, m_rec_dist, m_prec_dist = [], [], []
m_rec_enh, m_prec_enh = [], []

print("Extracting baselines and processing sweeps...")
baseline_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    base_boxes = get_strict_boxes(img_rgb)
    baseline_data.append((img_rgb, base_boxes))

for quality in jpeg_qualities:
    l_snr, l_rec_d, l_prec_d, l_rec_e, l_prec_e = [], [], [], [], []
    for img_rgb, base_boxes in baseline_data:
        # Generate distortion (JPEG Compression)
        jpeg_img = apply_jpeg_compression(img_rgb, quality)
        dist_boxes = get_strict_boxes(jpeg_img)
        rec_d, prec_d = calculate_precision_recall(base_boxes, dist_boxes)

        # Apply enhancement (Bilateral Filter)
        enhanced_img = apply_bilateral_filter(jpeg_img)
        enh_boxes = get_strict_boxes(enhanced_img)
        rec_e, prec_e = calculate_precision_recall(base_boxes, enh_boxes)

        l_snr.append(compute_snr(img_rgb, jpeg_img))
        l_rec_d.append(rec_d)
        l_prec_d.append(prec_d)
        l_rec_e.append(rec_e)
        l_prec_e.append(prec_e)

    # Aggregate mean metrics
    m_snr.append(np.mean(l_snr))
    m_rec_dist.append(np.mean(l_rec_d))
    m_prec_dist.append(np.mean(l_prec_d))
    m_rec_enh.append(np.mean(l_rec_e))
    m_prec_enh.append(np.mean(l_prec_e))

# --- 5. Plotting Side-by-Side Comparative Graphs ---
print("Plotting comparative graphs for Bilateral Enhancement...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Mask R-CNN Recovery Attempt: JPEG Artifacts vs Bilateral Filter\nProject by: Ayelet & Noa', fontsize=16, fontweight='bold')

# Left Subplot: Recall Tracking
ax1.plot(m_snr, m_rec_dist, marker='o', color='#d62728', linewidth=2.5, label='Distorted (JPEG Only)')
ax1.plot(m_snr, m_rec_enh, marker='s', color='#2ca02c', linewidth=2.5, label='Enhanced (Bilateral Filter)')
ax1.set_xlabel(r'SNR (dB) $\rightarrow$ Worse Quality', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean Recall', fontsize=12, fontweight='bold')
ax1.set_title('Recall Comparison', fontsize=14)
ax1.axhline(y=1.0, color='blue', linestyle='--', alpha=0.5, label='Baseline')
ax1.set_ylim(-0.05, 1.1)
ax1.invert_xaxis()
ax1.grid(True, alpha=0.3)
ax1.legend(loc='lower left')

# Annotate JPEG Quality steps
for i, quality in enumerate(jpeg_qualities):
    ax1.annotate(f"Q={quality}%", (m_snr[i], max(m_rec_dist[i], m_rec_enh[i])), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9)

# Right Subplot: Precision Tracking
ax2.plot(m_snr, m_prec_dist, marker='o', color='#1f77b4', linewidth=2.5, linestyle='--', label='Distorted Precision')
ax2.plot(m_snr, m_prec_enh, marker='s', color='#9467bd', linewidth=2.5, linestyle='--', label='Enhanced Precision')
ax2.set_xlabel(r'SNR (dB) $\rightarrow$ Worse Quality', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean Precision', fontsize=12, fontweight='bold')
ax2.set_title('Precision Comparison', fontsize=14)
ax2.axhline(y=1.0, color='blue', linestyle='--', alpha=0.5)
ax2.set_ylim(-0.05, 1.1)
ax2.invert_xaxis()
ax2.grid(True, alpha=0.3)
ax2.legend(loc='lower left')

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 2 - Alpha Blending Analysis (JPEG vs Bilateral Filter)

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision.transforms import functional as F

print("Task 3 - Step 5.5: Executing Lecturer's Challenge (Alpha Blending) for JPEG...")

# --- 1. Environment & Model Setup ---
# Dynamically allocate hardware compute device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

if 'model_maskrcnn' not in locals():
    print("Loading Mask R-CNN model (Pristine version)...")
    model_maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
    model_maskrcnn.eval()

model_maskrcnn.to(device)

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing! Please re-run the clean image selection block.")

# --- 2. Image Processing & Metric Evaluation Utilities ---
def apply_jpeg_compression(img_rgb, quality):
    # Simulate JPEG compression data loss
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encimg = cv2.imencode('.jpg', img_bgr, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

def apply_bilateral_filter(distorted_img):
    # Apply optimal Bilateral Filter for JPEG de-blocking
    # Smooths macro-blocks while preserving high-frequency structural edges
    return cv2.bilateralFilter(distorted_img, d=9, sigmaColor=75, sigmaSpace=75)

def compute_iou(box1, box2):
    # Calculate Intersection over Union (IoU) between bounding boxes
    x_left, y_top = max(box1[0], box2[0]), max(box1[1], box2[1])
    x_right, y_bottom = min(box1[2], box2[2]), min(box1[3], box2[3])
    if x_right < x_left or y_bottom < y_top: return 0.0

    intersection = (x_right - x_left) * (y_bottom - y_top)
    return intersection / float(((box1[2]-box1[0])*(box1[3]-box1[1])) + ((box2[2]-box2[0])*(box2[3]-box2[1])) - intersection)

def calculate_precision_recall(base_boxes, dist_boxes, iou_threshold=0.5):
    # Calculate Precision and Recall by matching inference predictions to baseline
    if len(base_boxes) == 0 and len(dist_boxes) == 0: return 1.0, 1.0
    if len(base_boxes) == 0 or len(dist_boxes) == 0: return 0.0, 0.0

    matched_gt, matched_pred = set(), set()
    for p_idx, p_box in enumerate(dist_boxes):
        best_iou, best_g_idx = 0, -1
        for g_idx, g_box in enumerate(base_boxes):
            if g_idx in matched_gt: continue
            iou = compute_iou(p_box, g_box)
            if iou > best_iou: best_iou, best_g_idx = iou, g_idx
        if best_iou > iou_threshold:
            matched_gt.add(best_g_idx)
            matched_pred.add(p_idx)

    tp = len(matched_gt)
    fp = len(dist_boxes) - tp
    fn = len(base_boxes) - tp
    return (tp / (tp + fn) if (tp + fn) > 0 else 0.0), (tp / (tp + fp) if (tp + fp) > 0 else 0.0)

# --- Optimized Inference Pipeline (Filtered Baseline) ---
def get_strict_boxes(img_rgb, threshold=0.5, iou_nms_threshold=0.3):
    h, w = img_rgb.shape[:2]
    img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model_maskrcnn(img_tensor)[0]

    # Primary spatial filtering based on minimum confidence threshold
    keep = prediction['scores'] > threshold
    boxes = prediction['boxes'][keep]
    scores = prediction['scores'][keep]

    if len(boxes) == 0:
        return []

    # Inject Non-Maximum Suppression (NMS) to prevent duplicate overlapping boxes
    keep_nms = torchvision.ops.nms(boxes, scores, iou_threshold=iou_nms_threshold)
    boxes = boxes[keep_nms]

    # Post-processing Ego-Vehicle filter (eliminates host vehicle hood detections)
    final_boxes = []
    for i in range(len(boxes)):
        box = boxes[i]
        y_center = (box[1].item() + box[3].item()) / 2
        if y_center < h * 0.85:
            final_boxes.append(box.cpu().numpy())

    return np.array(final_boxes) if len(final_boxes) > 0 else []

# --- 3. Experimental Parameters ---
alphas = [0.0, 0.3, 0.5, 0.7, 1.0]
# Select extreme compression level to induce heavy macroblocking artifacts
q_hard = 5

results_recall = []
results_precision = []

print("Calculating Baseline...")
base_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    base_boxes = get_strict_boxes(img_rgb)
    base_data.append((img_rgb, base_boxes))

# --- 4. Alpha Blending Execution Sweep ---
print(f"Running Alpha Blending sweep on extreme compression (Quality={q_hard}%)...")
best_alpha_img_data = None

for alpha in alphas:
    l_recall = []
    l_precision = []

    for img_rgb, base_boxes in base_data:
        # 1. Induce extreme JPEG compression distortion
        jpeg_img = apply_jpeg_compression(img_rgb, q_hard)
        # 2. Apply full bilateral filter restoration
        enh_img = apply_bilateral_filter(jpeg_img)

        # 3. Perform linear Alpha Blending interpolation
        blended_img = cv2.addWeighted(enh_img, alpha, jpeg_img, 1.0 - alpha, 0)

        # 4. Execute Mask R-CNN inference on the blended image
        boxes = get_strict_boxes(blended_img)

        rec, prec = calculate_precision_recall(base_boxes, boxes)
        l_recall.append(rec)
        l_precision.append(prec)

        # Cache fully enhanced image state for visual inspection
        if alpha == 1.0:
            best_alpha_img_data = (blended_img, boxes)

    results_recall.append(np.mean(l_recall))
    results_precision.append(np.mean(l_precision))

print("\n--- Alpha Blending Results (JPEG Q=5%) ---")
for i, a in enumerate(alphas):
    print(f"Alpha {a:.1f} | Recall: {results_recall[i]:.3f} | Precision: {results_precision[i]:.3f}")

# --- 5. Plotting Dual-Axis Alpha Blending Graph ---
fig, ax1 = plt.subplots(figsize=(9, 5))

# Plot Recall on primary Y-axis
color = 'tab:red'
ax1.set_xlabel('Alpha (0 = 100% JPEG Artifacts, 1 = 100% Bilateral Filter)', fontweight='bold')
ax1.set_ylabel('Mean Recall', color=color, fontweight='bold')
ax1.plot(alphas, results_recall, marker='o', color=color, linewidth=2.5, label='Recall')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(-0.05, 1.05)

# Plot Precision on secondary Y-axis
ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Mean Precision', color=color, fontweight='bold')
ax2.plot(alphas, results_precision, marker='s', color=color, linewidth=2.5, linestyle='--', label='Precision')
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(-0.05, 1.05)

fig.suptitle('Lecturer Challenge: Alpha Blending at JPEG Q=5%\nProject by: Ayelet & Noa', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.grid(True, alpha=0.3)
plt.show()

# --- 6. Visual Inspection of the Filtered State ---
print("\n--- Visual Inspection: 100% Enhanced (Alpha=1.0) ---")
blended_img, pred_boxes = best_alpha_img_data
fig_v, ax_v = plt.subplots(figsize=(10, 6))
ax_v.imshow(blended_img)

# Render prediction bounding boxes
for box in pred_boxes:
    rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], fill=False, color='green', linewidth=2)
    ax_v.add_patch(rect)

ax_v.set_title("Visual Inspection (Alpha=1.0 - Full Bilateral Filter)\nNotice how the filter tries to smooth blocks while keeping edges.", fontweight='bold')
ax_v.axis('off')
plt.show()

In [ ]:
!rm -rf /content/maskrcnn_dataset /content/gaussian_dataset

In [ ]:
#@title Task 2 -  Mixed Dataset Generation for JPEG Resilience

import os
import torch
import torchvision
from torchvision.transforms import functional as F
import cv2
import random
import numpy as np
from pathlib import Path

print("Step 6: Creating the Mixed Dataset for JPEG Compression...")

# --- 1. Environment & Model Initialization ---
# Dynamically allocate hardware compute device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Load the pristine baseline model to generate ground-truth pseudo-labels
if 'model_maskrcnn' not in locals():
    print("Loading pristine Mask R-CNN model for pseudo-labeling...")
    model_maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
    model_maskrcnn.eval()

model_maskrcnn.to(device)

# Ensure control images are properly cached to prevent data leakage
if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing! Please re-run the clean image selection block.")

# Fallback recovery in case the global image list was cleared from memory
if 'all_images' not in locals():
    print("Recovering all_images list...")
    images_dir = Path('/content/bdd100k/10k/test')
    all_images = list(images_dir.glob('*.jpg'))
    random.shuffle(all_images)

# --- 2. Target Directory Preparation ---
dataset_dir = Path('/content/jpeg_dataset')
dataset_dir.mkdir(parents=True, exist_ok=True)
(dataset_dir / 'images').mkdir(exist_ok=True)
(dataset_dir / 'targets').mkdir(exist_ok=True)

# Select 1,000 exclusive images for the training set (strictly excluding the 5 control test images)
test_image_paths = [str(p) for p, _ in selected_data]
train_images = [p for p in all_images if str(p) not in test_image_paths][:1000]

valid_count = 0
clean_count = 0
jpeg_count = 0

# Utility Function: Simulate JPEG macro-blocking and detail loss
def apply_jpeg_compression(img_rgb, quality):
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encimg = cv2.imencode('.jpg', img_bgr, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

print("Extracting Pseudo-labels and applying JPEG Compression. Please wait...")

for i, img_path in enumerate(train_images):
    img = cv2.imread(str(img_path))
    if img is None: continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    # --- A. Extract Pseudo-labels using optimized pipeline ---
    img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model_maskrcnn(img_tensor)[0]

    # 1. Primary spatial filtering based on minimum confidence threshold (0.5)
    keep = prediction['scores'] > 0.5
    boxes = prediction['boxes'][keep]
    scores = prediction['scores'][keep]
    masks = prediction['masks'][keep]
    labels = prediction['labels'][keep]

    if len(boxes) == 0:
        continue

    # 2. Inject Non-Maximum Suppression (NMS) to eliminate duplicate bounding boxes
    keep_nms = torchvision.ops.nms(boxes, scores, iou_threshold=0.3)
    boxes = boxes[keep_nms]
    masks = masks[keep_nms]
    labels = labels[keep_nms]

    # 3. Apply spatial Ego-Vehicle filter (exclude host vehicle hood detections)
    final_boxes, final_masks, final_labels = [], [], []
    for j in range(len(boxes)):
        box = boxes[j]
        y_center = (box[1].item() + box[3].item()) / 2

        # Exclude objects whose geometric center falls in the bottom 15% of the frame
        if y_center < h * 0.85:
            final_boxes.append(box)
            final_masks.append(masks[j])
            final_labels.append(labels[j])

    if len(final_boxes) == 0:
        continue

    # Re-stack lists into standard PyTorch tensors
    final_boxes = torch.stack(final_boxes)
    final_masks = torch.stack(final_masks)
    final_labels = torch.stack(final_labels)

    # --- B. Generate Mixed Dataset (50% Pristine, 50% Compressed) ---
    if random.random() < 0.5:
        final_img_rgb = img_rgb
        clean_count += 1
    else:
        # Randomize JPEG Quality factor between 5% (extreme artifacts) and 50% (mild artifacts)
        quality = int(random.uniform(5, 50))
        final_img_rgb = apply_jpeg_compression(img_rgb, quality)
        jpeg_count += 1

    # --- C. Export Image and Target Tensors to Disk ---
    img_out_path = dataset_dir / 'images' / f"img_{i}.jpg"
    target_out_path = dataset_dir / 'targets' / f"tgt_{i}.pt"

    cv2.imwrite(str(img_out_path), cv2.cvtColor(final_img_rgb, cv2.COLOR_RGB2BGR))

    # Package ground-truth tensors into a dictionary for PyTorch training loop compatibility
    target = {
        "boxes": final_boxes.cpu(),
        "masks": final_masks.cpu(),
        "labels": final_labels.cpu()
    }
    torch.save(target, target_out_path)

    valid_count += 1

    if valid_count % 100 == 0:
        print(f"Processed {valid_count} valid images...")

print(f"\nJPEG Dataset Ready! Total: {valid_count} images ({clean_count} Clean, {jpeg_count} Compressed).")

In [ ]:
#@title Task 2 - Mask R-CNN Fine-Tuning & Grand Finale Evaluation (JPEG)

import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from torchvision.transforms import functional as F
import torchvision
import torchvision.ops as ops

# ==========================================
# Environment Setup & Variable Validation
# ==========================================
# Dynamically allocate hardware compute device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Working on device: {device}")

# Ensure control images are properly cached to prevent data leakage
if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing! Please re-run the clean image selection block.")

# --- Independent Utility Functions ---
def compute_iou(box1, box2):
    # Calculate Intersection over Union (IoU) between bounding boxes
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])
    if x_right < x_left or y_bottom < y_top: return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return intersection_area / float(box1_area + box2_area - intersection_area)

def calculate_precision_recall(base_boxes, dist_boxes, iou_threshold=0.5):
    # Compute Precision and Recall metrics against ground truth baseline
    if len(base_boxes) == 0 and len(dist_boxes) == 0: return 1.0, 1.0
    if len(base_boxes) == 0 or len(dist_boxes) == 0: return 0.0, 0.0

    matched_gt = set()
    for p_box in dist_boxes:
        best_iou = 0
        best_g_idx = -1
        for g_idx, g_box in enumerate(base_boxes):
            if g_idx in matched_gt: continue
            iou = compute_iou(p_box, g_box)
            if iou > best_iou:
                best_iou = iou
                best_g_idx = g_idx
        if best_iou > iou_threshold:
            matched_gt.add(best_g_idx)

    tp = len(matched_gt)
    fp = len(dist_boxes) - tp
    fn = len(base_boxes) - tp
    return (tp / (tp + fn) if (tp + fn) > 0 else 0.0), (tp / (tp + fp) if (tp + fp) > 0 else 0.0)

def apply_jpeg_compression(img_rgb, quality):
    # Simulate JPEG compression artifacts (macroblocking, high-frequency data loss)
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encimg = cv2.imencode('.jpg', img_bgr, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

def apply_bilateral_filter(distorted_img):
    # Apply Bilateral Filter for optimal JPEG de-blocking while preserving edges
    return cv2.bilateralFilter(distorted_img, d=9, sigmaColor=75, sigmaSpace=75)

# --- Optimized Inference Pipeline (Filtered Baseline for Final Evaluation) ---
def get_strict_boxes_from_model(model_obj, img_rgb, current_device, threshold=0.5, iou_nms_threshold=0.3):
    h, w = img_rgb.shape[:2]
    img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(current_device)
    model_obj.to(current_device)

    with torch.no_grad():
        prediction = model_obj(img_tensor)[0]

    # Primary confidence thresholding
    keep = prediction['scores'] > threshold
    boxes = prediction['boxes'][keep]
    scores = prediction['scores'][keep]

    if len(boxes) == 0:
        return np.array([])

    # Non-Maximum Suppression (NMS) to eliminate duplicate bounding boxes
    keep_nms = torchvision.ops.nms(boxes, scores, iou_threshold=iou_nms_threshold)
    boxes = boxes[keep_nms]

    # Spatial Ego-Vehicle filter (exclude host vehicle hood detections)
    final_boxes = []
    for i in range(len(boxes)):
        box = boxes[i]
        y_center = (box[1].item() + box[3].item()) / 2
        if y_center < h * 0.85:
            final_boxes.append(box.cpu().numpy())

    return np.array(final_boxes) if len(final_boxes) > 0 else np.array([])

# ==========================================
# Step 7: Dataset Definition & Fine-Tuning Execution
# ==========================================
class JpegDataset(Dataset):
    # Custom PyTorch Dataset for loading JPEG compression artifact pairs
    def __init__(self, dataset_dir):
        self.dataset_dir = Path(dataset_dir)
        self.img_files = sorted(list((self.dataset_dir / 'images').glob('*.jpg')))

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        idx_str = img_path.stem.split('_')[1]
        tgt_path = self.dataset_dir / 'targets' / f"tgt_{idx_str}.pt"

        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_tensor = F.to_tensor(img_rgb)

        target = torch.load(tgt_path)
        # Binarize masks for PyTorch compatibility
        target['masks'] = (target['masks'] > 0.5).squeeze(1).to(torch.uint8)
        return img_tensor, target

print("Setting up PyTorch DataLoader...")
train_dataset = JpegDataset('/content/jpeg_dataset')
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

print("Initializing model for fine-tuning on JPEG artifacts...")
finetuned_maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
finetuned_maskrcnn.to(device)

# Configure optimizer with a conservative learning rate
params = [p for p in finetuned_maskrcnn.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=1e-5, weight_decay=0.0005)

EPOCHS = 5
total_batches = len(train_loader)
print(f"Starting Fine-Tuning Loop ({EPOCHS} Epochs) for JPEG...")
finetuned_maskrcnn.train()

for epoch in range(EPOCHS):
    epoch_loss = 0
    print(f"\n--- Starting Epoch {epoch+1}/{EPOCHS} ---")

    for batch_idx, (images, targets) in enumerate(train_loader):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass and loss calculation
        loss_dict = finetuned_maskrcnn(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Backward pass and optimization step
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        epoch_loss += losses.item()

        # Live batch progress reporting
        if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == total_batches:
            avg_batch_loss = epoch_loss / (batch_idx + 1)
            print(f"  -> Batch [{batch_idx+1}/{total_batches}] completed. Current Average Loss: {avg_batch_loss:.4f}")

    print(f"=== Epoch {epoch+1} Completed! Final Average Loss: {epoch_loss/total_batches:.4f} ===")

print("\nFine-Tuning Complete! Switching model to evaluation mode.")
finetuned_maskrcnn.eval()

# ==========================================
# Step 8: The Grand Finale - Head-to-Head Comparative Evaluation
# ==========================================
print("\nStep 8: Running Final Evaluation on Extreme JPEG Compression (Quality=5%)...")

# Load a pristine Mask R-CNN model for an unbiased baseline comparison
fresh_original_model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
fresh_original_model.eval()

q_hard = 5
rec_distorted, prec_distorted = [], []
rec_enhanced, prec_enhanced = [], []
rec_finetuned, prec_finetuned = [], []

print("Extracting test baseline data...")
baseline_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    base_boxes = get_strict_boxes_from_model(fresh_original_model, img_rgb, device)
    baseline_data.append((img_rgb, base_boxes))

print("Evaluating all three setups...")
for img_rgb, base_boxes in baseline_data:
    # 1. Original Model evaluated on Distorted Image
    jpeg_img = apply_jpeg_compression(img_rgb, q_hard)
    boxes_dist = get_strict_boxes_from_model(fresh_original_model, jpeg_img, device)
    r_d, p_d = calculate_precision_recall(base_boxes, boxes_dist)
    rec_distorted.append(r_d)
    prec_distorted.append(p_d)

    # 2. Original Model evaluated on Enhanced Image (Bilateral Filter)
    enhanced_img = apply_bilateral_filter(jpeg_img)
    boxes_enh = get_strict_boxes_from_model(fresh_original_model, enhanced_img, device)
    r_e, p_e = calculate_precision_recall(base_boxes, boxes_enh)
    rec_enhanced.append(r_e)
    prec_enhanced.append(p_e)

    # 3. Fine-Tuned Model evaluated directly on Distorted Image
    boxes_ft = get_strict_boxes_from_model(finetuned_maskrcnn, jpeg_img, device)
    r_f, p_f = calculate_precision_recall(base_boxes, boxes_ft)
    rec_finetuned.append(r_f)
    prec_finetuned.append(p_f)

# Aggregate mean metrics for final plotting
mean_rec = [np.mean(rec_distorted), np.mean(rec_enhanced), np.mean(rec_finetuned)]
mean_prec = [np.mean(prec_distorted), np.mean(prec_enhanced), np.mean(prec_finetuned)]

# --- Dual-Bar Chart Visualization ---
print("Plotting the Final JPEG Bar Chart...")
labels = ['Original Model\n(JPEG Distorted)', 'Original Model\n(Enhanced - Bilateral)', 'Fine-Tuned Model\n(JPEG Distorted)']
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, mean_rec, width, label='Recall', color='#d62728', edgecolor='black', linewidth=1.2)
rects2 = ax.bar(x + width/2, mean_prec, width, label='Precision', color='#1f77b4', edgecolor='black', linewidth=1.2)

# Reference line representing the clean baseline performance (1.00)
ax.axhline(y=1.0, color='blue', linestyle='--', linewidth=1.5, alpha=0.6, label='Clean Baseline (1.00)')

# Helper function to explicitly annotate metric values above the bars
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=11, fontweight='bold')

autolabel(rects1)
autolabel(rects2)

# Graph aesthetics and structural formatting
ax.set_ylabel('Scores', fontsize=12, fontweight='bold')
ax.set_title(f'The Grand Finale: Mask R-CNN vs Extreme JPEG Compression (Quality={q_hard}%)\nProject by: Ayelet & Noa', fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.2)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 2: Side-by-Side JPEG Evaluation Sweep (Line Charts)

import numpy as np
import matplotlib.pyplot as plt
import cv2

print("Step 9: Running Full Quality Sweep for JPEG Line Charts...")

# --- 1. Signal Processing Utility (SNR) ---
def compute_snr(clean_rgb, dist_rgb):
    clean, dist_img = clean_rgb.astype(np.float64), dist_rgb.astype(np.float64)
    noise_power = np.mean((clean - dist_img) ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)

# --- 2. Main Evaluation Sweep (Distorted vs Enhanced vs Fine-Tuned) ---
# JPEG Quality levels: 100 is best, 0 is absolute blocky mess.
jpeg_qualities = [50, 30, 15, 10, 5]
m_snr, m_rec_dist, m_prec_dist = [], [], []
m_rec_enh, m_prec_enh = [], []
m_rec_ft, m_prec_ft = [], []

print("Extracting baselines from original model...")
# Reuse baseline data if it exists in memory to save time
if 'baseline_data' not in locals():
    baseline_data = []
    for img_path, _ in selected_data:
        img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        base_boxes = get_strict_boxes_from_model(fresh_original_model, img_rgb, device)
        baseline_data.append((img_rgb, base_boxes))

print("Processing sweeps across all JPEG quality levels...")
for q_val in jpeg_qualities:
    l_snr = []
    l_rec_d, l_prec_d = [], []
    l_rec_e, l_prec_e = [], []
    l_rec_f, l_prec_f = [], []

    for img_rgb, base_boxes in baseline_data:
        # Generate distorted image (JPEG Compression)
        jpeg_img = apply_jpeg_compression(img_rgb, q_val)
        l_snr.append(compute_snr(img_rgb, jpeg_img))

        # 1. Distorted Image on Original Model
        dist_boxes = get_strict_boxes_from_model(fresh_original_model, jpeg_img, device)
        rec_d, prec_d = calculate_precision_recall(base_boxes, dist_boxes)
        l_rec_d.append(rec_d); l_prec_d.append(prec_d)

        # 2. Enhanced Image (Bilateral Filter) on Original Model
        enhanced_img = apply_bilateral_filter(jpeg_img)
        enh_boxes = get_strict_boxes_from_model(fresh_original_model, enhanced_img, device)
        rec_e, prec_e = calculate_precision_recall(base_boxes, enh_boxes)
        l_rec_e.append(rec_e); l_prec_e.append(prec_e)

        # 3. Distorted Image on Fine-Tuned Model
        ft_boxes = get_strict_boxes_from_model(finetuned_maskrcnn, jpeg_img, device)
        rec_f, prec_f = calculate_precision_recall(base_boxes, ft_boxes)
        l_rec_f.append(rec_f); l_prec_f.append(prec_f)

    # Aggregate mean metrics
    m_snr.append(np.mean(l_snr))
    m_rec_dist.append(np.mean(l_rec_d)); m_prec_dist.append(np.mean(l_prec_d))
    m_rec_enh.append(np.mean(l_rec_e)); m_prec_enh.append(np.mean(l_prec_e))
    m_rec_ft.append(np.mean(l_rec_f)); m_prec_ft.append(np.mean(l_prec_f))
    print(f"Quality Q={q_val}%: SNR={m_snr[-1]:.1f}dB | FT Recall={m_rec_ft[-1]:.2f} | FT Precision={m_prec_ft[-1]:.2f}")


# --- 3. Plotting Side-by-Side Comparative Graphs ---
print("Plotting comparative side-by-side graphs...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('The Grand Finale: JPEG Compression & Fine-Tuning Performance\nProject by: Ayelet & Noa', fontsize=16, fontweight='bold')

# --- Left Graph: Recall Tracking ---
ax1.plot(m_snr, m_rec_dist, marker='o', color='#d62728', linewidth=2.5, label='Original Model (Distorted)')
ax1.plot(m_snr, m_rec_enh, marker='s', color='#2ca02c', linewidth=2.5, label='Original Model (Bilateral Filter)')
ax1.plot(m_snr, m_rec_ft, marker='^', color='#9467bd', linewidth=2.5, label='Fine-Tuned Model (Distorted)')

ax1.set_xlabel(r'SNR (dB) $\rightarrow$ Worse Quality (More Compression)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean Recall', fontsize=12, fontweight='bold')
ax1.set_title('Recall Comparison', fontsize=14)
ax1.axhline(y=1.0, color='blue', linestyle='--', alpha=0.5, label='Baseline')
ax1.set_ylim(-0.05, 1.15)
ax1.invert_xaxis()
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.legend(loc='lower left')

# Annotate JPEG Quality (Q) values on the Recall Graph
for i, q_val in enumerate(jpeg_qualities):
    highest_val = max(m_rec_dist[i], m_rec_enh[i], m_rec_ft[i])
    ax1.annotate(f"Q={q_val}%", (m_snr[i], highest_val), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, fontweight='bold', color='#1f77b4')


# --- Right Graph: Precision Tracking ---
ax2.plot(m_snr, m_prec_dist, marker='o', color='#d62728', linewidth=2.5, label='Original Model (Distorted)')
ax2.plot(m_snr, m_prec_enh, marker='s', color='#2ca02c', linewidth=2.5, label='Original Model (Bilateral Filter)')
ax2.plot(m_snr, m_prec_ft, marker='^', color='#9467bd', linewidth=2.5, label='Fine-Tuned Model (Distorted)')

ax2.set_xlabel(r'SNR (dB) $\rightarrow$ Worse Quality (More Compression)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean Precision', fontsize=12, fontweight='bold')
ax2.set_title('Precision Comparison', fontsize=14)
ax2.axhline(y=1.0, color='blue', linestyle='--', alpha=0.5)
ax2.set_ylim(-0.05, 1.15)
ax2.invert_xaxis()
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.legend(loc='lower left')

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 3: Baseline Monocular Depth Estimation using MiDaS

import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np

print("Initializing Monocular Depth Estimation with MiDaS...")

# --- 1. Device Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Working on device: {device}")

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing! Please re-run the clean image selection block.")

# --- 2. Load MiDaS Model and Transforms ---
# Utilizing the 'small' variant to ensure memory-efficient execution (e.g., in Google Colab)
midas_model_type = "MiDaS_small"
print(f"Downloading/Loading {midas_model_type}...")
midas = torch.hub.load("isl-org/MiDaS", midas_model_type)
midas.to(device)
midas.eval()

midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
transform = midas_transforms.small_transform

# --- 3. Execute Baseline Model and Visualize Results ---
fig, axes = plt.subplots(len(selected_data), 2, figsize=(16, 4 * len(selected_data)))
fig.suptitle('Baseline: Monocular Depth Estimation with MiDaS\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.98)

for idx, (img_path, _) in enumerate(selected_data):
    # Read and convert image to RGB
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocess input via MiDaS transforms and perform inference
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        prediction = midas(input_batch)

        # Interpolate the depth map to match the original image resolution
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_rgb.shape[:2],
            mode="bicubic",
            align_corners=False,
        ).squeeze()

    # Convert prediction tensor to NumPy array
    depth_map = prediction.cpu().numpy()

    # Normalize depth map values to the [0, 1] range for heatmap visualization
    depth_map_normalized = cv2.normalize(depth_map, None, 0, 1, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_32F)

    # Left column: Original RGB image
    axes[idx, 0].imshow(img_rgb)
    axes[idx, 0].set_title("Original Image", fontsize=15, fontweight='bold')
    axes[idx, 0].axis('off')

    # Right column: Predicted depth map
    # The 'magma' colormap represents closer objects with brighter colors and distant objects with darker colors
    im = axes[idx, 1].imshow(depth_map_normalized, cmap='magma')
    axes[idx, 1].set_title("Predicted Depth Map (Magma)", fontsize=15, fontweight='bold')
    axes[idx, 1].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 3 : Refined Baseline with ROI Masking

import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np

print("Task 3 -  Refined Baseline with ROI Masking...")

# --- 1. Environment and Model Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Working on device: {device}")

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing! Please re-run the clean image selection block.")

if 'midas' not in locals():
    print("Loading MiDaS_small...")
    midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small")
    midas.to(device)
    midas.eval()
    midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
    transform = midas_transforms.small_transform

# --- 2. Execute Clean Baseline ---
fig, axes = plt.subplots(len(selected_data), 2, figsize=(16, 4 * len(selected_data)))
fig.suptitle('Clean Baseline: Depth Estimation with ROI Fix\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.98)

for idx, (img_path, _) in enumerate(selected_data):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    # Define Region of Interest (ROI): Exclude the bottom 20%
    roi_h = int(h * 0.8)

    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=(h, w),
            mode="bicubic",
            align_corners=False,
        ).squeeze()

    raw_depth = prediction.cpu().numpy()

    # --- Scientific Adjustment: Dynamic calculation of depth values based solely on the ROI ---
    roi_depth = raw_depth[:roi_h, :]
    d_min = roi_depth.min()
    d_max = roi_depth.max()

    # Clip and stretch the data to ensure vehicles receive warmer colors
    depth_clipped = np.clip(raw_depth, d_min, d_max)
    depth_normalized = (depth_clipped - d_min) / (d_max - d_min + 1e-6)

    # Mask (black out) the dashboard area in both the depth map and the original image
    depth_normalized[roi_h:, :] = 0
    clean_img_rgb = img_rgb.copy()
    clean_img_rgb[roi_h:, :] = 0

    axes[idx, 0].imshow(clean_img_rgb)
    axes[idx, 0].set_title("Original Image (Masked)", fontsize=15, fontweight='bold')
    axes[idx, 0].axis('off')

    im = axes[idx, 1].imshow(depth_normalized, cmap='magma')
    axes[idx, 1].set_title("Clean Depth Map (Dynamic Range Fixed)", fontsize=15, fontweight='bold')
    axes[idx, 1].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 3:Gradual Darkness Sweep (HSV Method)

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("Generating Gradual Darkness Sweep (HSV Method)...")

def apply_realistic_darkness(img_rgb, darkness_factor):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 2] = hsv[:, :, 2] * darkness_factor
    hsv[:, :, 2] = np.clip(hsv[:, :, 2], 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing! Please re-run the clean image selection block.")

# --- Progressive Darkness Intensity Levels ---
darkness_levels = [0.9, 0.7, 0.5, 0.35, 0.15]

fig, axes = plt.subplots(len(selected_data), 6, figsize=(24, 15))
fig.suptitle('Gradual Darkness Sweep (HSV)\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.98)

for row_idx, (img_path, _) in enumerate(selected_data):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Maintain a clean Region of Interest (ROI) by masking the bottom area
    h, w = img_rgb.shape[:2]
    roi_h = int(h * 0.8)
    img_rgb[roi_h:, :] = 0

    # Reference column (Original image)
    axes[row_idx, 0].imshow(img_rgb)
    if row_idx == 0:
        axes[row_idx, 0].set_title("Original\n(Factor: 1.0)", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Apply darkness levels iteratively
    for col_idx, factor in enumerate(darkness_levels, start=1):
        dark_img = apply_realistic_darkness(img_rgb, factor)

        axes[row_idx, col_idx].imshow(dark_img)

        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"Factor: {factor}", fontsize=14)
        axes[row_idx, col_idx].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 3: Visualizing MiDaS Depth Maps under Darkness

import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np

print("Visualizing MiDaS Depth Maps under Darkness...")

# --- 1. Environment and Model Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'midas' not in locals():
    print("Loading MiDaS_small...")
    midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()
    midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
    transform = midas_transforms.small_transform

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing!")

# --- 2. Helper Functions ---
def apply_realistic_darkness(img_rgb, darkness_factor):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * darkness_factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

def get_normalized_depth(img_rgb, roi_h):
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = midas(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

# --- 3. Generate Visual Grid of Depth Maps ---
darkness_levels = [0.9, 0.7, 0.5, 0.35, 0.05]

fig, axes = plt.subplots(len(selected_data), 6, figsize=(24, 15))
fig.suptitle('MiDaS Depth Maps Degradation under Darkness\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.98)

for row_idx, (img_path, _) in enumerate(selected_data):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    roi_h = int(img_rgb.shape[0] * 0.8)

    # Column 0: Reference (Clean baseline depth map)
    base_depth = get_normalized_depth(img_rgb, roi_h)
    axes[row_idx, 0].imshow(base_depth, cmap='magma')
    if row_idx == 0:
        axes[row_idx, 0].set_title("Baseline Depth\n(Factor: 1.0)", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Columns 1-5: Depth maps under degradation
    for col_idx, factor in enumerate(darkness_levels, start=1):
        dark_img = apply_realistic_darkness(img_rgb, factor)
        dark_depth = get_normalized_depth(dark_img, roi_h)

        axes[row_idx, col_idx].imshow(dark_depth, cmap='magma')

        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"Depth at\nFactor: {factor}", fontsize=14)
        axes[row_idx, col_idx].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 3: MiDaS Inference & RMSE Evaluation on Darkness

import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np

print("MiDaS Inference & RMSE Evaluation on Darkness...")

# --- 1. Environment and Model Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'midas' not in locals():
    print("Loading MiDaS_small...")
    midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()
    midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
    transform = midas_transforms.small_transform

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing!")

# --- 2. Mathematical and Distortion Functions ---
def apply_realistic_darkness(img_rgb, darkness_factor):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * darkness_factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

def compute_snr(clean_rgb, dist_rgb):
    clean, dist = clean_rgb.astype(np.float64), dist_rgb.astype(np.float64)
    noise_power = np.mean((clean - dist) ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)

def compute_roi_rmse(depth1, depth2, roi_h):
    # Calculate the Root Mean Square Error (RMSE) exclusively on the top 80% (ROI)
    roi_d1 = depth1[:roi_h, :]
    roi_d2 = depth2[:roi_h, :]
    mse = np.mean((roi_d1 - roi_d2) ** 2)
    return np.sqrt(mse)

def get_normalized_depth(img_rgb, roi_h):
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = midas(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0  # Black out the dashboard area
    return depth_norm

# --- 3. Experiment Execution ---
darkness_levels = [0.9, 0.7, 0.5, 0.35, 0.15]
m_snr, m_rmse = [], []

print("Extracting Baseline Depth Maps...")
baseline_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    roi_h = int(img_rgb.shape[0] * 0.8)
    base_depth = get_normalized_depth(img_rgb, roi_h)
    baseline_data.append((img_rgb, base_depth, roi_h))

print("Running Model on Darkened Images...")
for factor in darkness_levels:
    l_snr, l_rmse = [], []
    for img_rgb, base_depth, roi_h in baseline_data:
        # Apply darkness distortion
        dark_img = apply_realistic_darkness(img_rgb, factor)
        # Execute model inference
        dark_depth = get_normalized_depth(dark_img, roi_h)

        # Calculate metrics
        l_snr.append(compute_snr(img_rgb, dark_img))
        l_rmse.append(compute_roi_rmse(base_depth, dark_depth, roi_h))

    m_snr.append(np.mean(l_snr))
    m_rmse.append(np.mean(l_rmse))

# --- 4. Plotting the Graph (RMSE vs SNR) ---
print("Plotting Degradation Graph...")
fig, ax = plt.subplots(figsize=(10, 6))

color = 'tab:red'
ax.set_xlabel(r'SNR (dB) $\rightarrow$ More Darkness', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean RMSE (Error)', color=color, fontsize=12, fontweight='bold')
ax.plot(m_snr, m_rmse, marker='X', markersize=10, color=color, linewidth=3, label='Depth Error (RMSE)')
ax.tick_params(axis='y', labelcolor=color)

# Invert the X-axis so that increased degradation (lower SNR) appears on the right
ax.invert_xaxis()

for i, factor in enumerate(darkness_levels):
    ax.annotate(f"Factor: {factor}", (m_snr[i], m_rmse[i]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=10, fontweight='bold')

plt.title("MiDaS Depth Perception Error vs SNR under Darkness\nProject by: Ayelet & Noa", fontsize=15, fontweight='bold', pad=15)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
#@title Classical Enhancement (CLAHE) vs Extreme Darkness

import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import structural_similarity as ssim

print("Classical Enhancement (CLAHE) vs Extreme Darkness...")

# --- 1. Environment and Model Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'midas' not in locals():
    print("Loading MiDaS_small...")
    midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()
    midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
    transform = midas_transforms.small_transform

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing!")

# --- 2. Image Processing and Metrics Functions ---
def apply_realistic_darkness(img_rgb, darkness_factor):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * darkness_factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

def apply_enhancement_darkness(dark_img):
    # Classical enhancement for darkness: CLAHE (Contrast Limited Adaptive Histogram Equalization)
    # Convert to LAB color space to process only the Lightness (L) channel, preventing color distortion
    lab = cv2.cvtColor(dark_img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l)

    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

def compute_snr(clean_rgb, dist_rgb):
    clean, dist = clean_rgb.astype(np.float64), dist_rgb.astype(np.float64)
    noise_power = np.mean((clean - dist) ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)

def get_normalized_depth(model, img_rgb, roi_h):
    model.eval()
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = model(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

def compute_metrics(depth1, depth2, roi_h):
    roi_d1 = depth1[:roi_h, :]
    roi_d2 = depth2[:roi_h, :]
    rmse_val = np.sqrt(np.mean((roi_d1 - roi_d2) ** 2))
    ssim_val = ssim(roi_d1, roi_d2, data_range=1.0)
    return rmse_val, ssim_val

# --- 3. Run Comparative Experiment ---
# Added extreme factor 0.05 to observe model failure and subsequent recovery via enhancement
darkness_levels = [0.9, 0.7, 0.5, 0.15, 0.05]
m_snr, m_rmse_dist, m_ssim_dist = [], [], []
m_rmse_enh, m_ssim_enh = [], []

print("Extracting Baseline Depth Maps...")
baseline_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    roi_h = int(img_rgb.shape[0] * 0.8)
    base_depth = get_normalized_depth(midas, img_rgb, roi_h)
    baseline_data.append((img_rgb, base_depth, roi_h))

print("Running Inference: Distorted (Dark) vs. Enhanced (CLAHE)...")
for factor in darkness_levels:
    l_snr, l_r_dist, l_s_dist, l_r_enh, l_s_enh = [], [], [], [], []

    for img_rgb, base_depth, roi_h in baseline_data:
        # 1. Apply distortion (Darkness)
        dark_img = apply_realistic_darkness(img_rgb, factor)
        # 2. Apply classical correction (CLAHE)
        enhanced_img = apply_enhancement_darkness(dark_img)

        # 3. Execute model inference
        depth_dist = get_normalized_depth(midas, dark_img, roi_h)
        depth_enh = get_normalized_depth(midas, enhanced_img, roi_h)

        # 4. Calculate metrics
        l_snr.append(compute_snr(img_rgb, dark_img))

        r_d, s_d = compute_metrics(base_depth, depth_dist, roi_h)
        r_e, s_e = compute_metrics(base_depth, depth_enh, roi_h)

        l_r_dist.append(r_d); l_s_dist.append(s_d)
        l_r_enh.append(r_e); l_s_enh.append(s_e)

    m_snr.append(np.mean(l_snr))
    m_rmse_dist.append(np.mean(l_r_dist)); m_ssim_dist.append(np.mean(l_s_dist))
    m_rmse_enh.append(np.mean(l_r_enh)); m_ssim_enh.append(np.mean(l_s_enh))

# --- 4. Plot Head-to-Head Comparative Graphs ---
print("Plotting Head-to-Head Comparison Graphs...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Classical Enhancement (CLAHE) vs Extreme Darkness\nProject by: Ayelet & Noa", fontsize=18, fontweight='bold', y=1.05)

# RMSE Graph
ax1.set_xlabel(r'SNR (dB) $\rightarrow$ More Darkness', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean RMSE (Lower is Better)', fontsize=12, fontweight='bold')
ax1.plot(m_snr, m_rmse_dist, marker='X', markersize=10, color='tab:red', linewidth=3, label='Distorted (Darkness)')
ax1.plot(m_snr, m_rmse_enh, marker='o', markersize=10, color='tab:green', linewidth=3, label='Enhanced (CLAHE)')
ax1.invert_xaxis()
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(fontsize=11)
for i, factor in enumerate(darkness_levels):
    ax1.annotate(f"F:{factor}", (m_snr[i], m_rmse_dist[i]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, fontweight='bold')
ax1.set_title("Mathematical Error (RMSE)", fontsize=14, fontweight='bold')

# SSIM Graph
ax2.set_xlabel(r'SNR (dB) $\rightarrow$ More Darkness', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean SSIM (Higher is Better)', fontsize=12, fontweight='bold')
ax2.plot(m_snr, m_ssim_dist, marker='X', markersize=10, color='tab:blue', linewidth=3, label='Distorted (Darkness)')
ax2.plot(m_snr, m_ssim_enh, marker='o', markersize=10, color='tab:green', linewidth=3, label='Enhanced (CLAHE)')
ax2.invert_xaxis()
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(fontsize=11)
for i, factor in enumerate(darkness_levels):
    ax2.annotate(f"F:{factor}", (m_snr[i], m_ssim_dist[i]), textcoords="offset points", xytext=(0,-15), ha='center', fontsize=9, fontweight='bold')
ax2.set_title("Structural Similarity Index (SSIM)", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 3: MiDaS Fine-Tuning - Data Mixing (50% Clean / 50% Dark, Full Range)

import random
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

print("Preparing Dataset (3000 images) with strict 50/50 Mix and Full Darkness Range...")

def apply_realistic_darkness(img_rgb, darkness_factor):
    # Scale the Value (V) channel in the HSV color space to simulate realistic lighting degradation
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * darkness_factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

class MiDaSMixedDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = str(self.image_paths[idx])
        img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

        # Resize while preserving the standard 16:9 dash camera aspect ratio
        img_rgb = cv2.resize(img_rgb, (512, 288))

        # Ground truth is strictly derived from the pristine, unaugmented image
        clean_input = self.transform(img_rgb).squeeze(0)

        # Enforce a strict 50/50 probability split between clean and darkened inputs to prevent mode collapse
        if random.random() < 0.50:
            # Apply a full dynamic range of darkness, from extreme (0.05) to subtle dimming (0.95)
            dark_factor = random.uniform(0.05, 0.95)
            mixed_rgb = apply_realistic_darkness(img_rgb, dark_factor)
        else:
            mixed_rgb = img_rgb

        mixed_input = self.transform(mixed_rgb).squeeze(0)

        return mixed_input, clean_input

# Filter out test images to prevent data leakage during training
test_image_paths = [str(p) for p, _ in selected_data]
train_images = [p for p in all_images if str(p) not in test_image_paths][:3000]

# Initialize the Dataset and DataLoader
train_dataset = MiDaSMixedDataset(train_images, transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)

print(f"Dataset ready! Total training images: {len(train_dataset)}")

In [ ]:
#@title Task 3: MiDaS Fine-Tuning - Smart Denoising for Darkness

import copy
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch

print("Initializing MiDaS Student Model (Smart Unfrozen Training)...")

# Initialize student model via deep copy for the Teacher-Student architecture
student_midas = copy.deepcopy(midas).to(device)
student_midas.train()
midas.eval()

# Unfreeze the entire network to enable deep feature adaptation under low-light conditions
for param in student_midas.parameters():
    param.requires_grad = True

# Smooth L1 Loss (Huber Loss) prevents mode collapse (averaging) caused by extreme MSE penalties in dark regions
criterion = nn.SmoothL1Loss(beta=0.1)

# AdamW optimizer with weight decay for stable regularization
optimizer = optim.AdamW(student_midas.parameters(), lr=1e-4, weight_decay=1e-4)

# Cosine Annealing scheduler ensures smooth convergence over the training cycle
epochs = 15
scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

print(f"Starting Training for {epochs} epochs...")

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_idx, (mixed_inputs, clean_inputs) in enumerate(train_loader):
        mixed_inputs = mixed_inputs.to(device)
        clean_inputs = clean_inputs.to(device)

        # Generate ground-truth depth targets from pristine images using the frozen teacher
        with torch.no_grad():
            target_depths = midas(clean_inputs)

        optimizer.zero_grad()

        # Forward pass: Predict depth from darkened inputs
        predictions = student_midas(mixed_inputs)

        # Calculate loss using the robust Smooth L1 objective
        loss = criterion(predictions, target_depths)

        # Backward pass and optimization step
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        if batch_idx % 25 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    # Step the learning rate scheduler per epoch
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"===> Epoch {epoch+1} Average Loss: {epoch_loss / len(train_loader):.4f} | Current LR: {current_lr:.6f}")

# Persist the optimized weights for evaluation
torch.save(student_midas.state_dict(), '/content/midas_finetuned_dark_smart.pth')
print("Training Complete! Model saved to '/content/midas_finetuned_dark_smart.pth'")

In [ ]:
#@title Task 3: The Grand Finale Evaluation (FIXED SCALE)

import gc
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

print("Initializing final comprehensive evaluation with NORMALIZED metrics...")

torch.cuda.empty_cache()
gc.collect()

device = torch.device('cuda')

# Load Models
original_midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()

finetuned_midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small")
finetuned_midas.load_state_dict(torch.load('/content/midas_finetuned_dark_smart.pth'))
finetuned_midas.to(device).eval()

midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
transform = midas_transforms.small_transform

# Utility Functions
def compute_snr(clean_rgb, dist_rgb):
    clean, dist = clean_rgb.astype(np.float64), dist_rgb.astype(np.float64)
    noise_power = np.mean((clean - dist) ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)

def apply_realistic_darkness(img_rgb, darkness_factor):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * darkness_factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

def apply_enhancement_darkness(dark_img):
    lab = cv2.cvtColor(dark_img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

# Critical Adjustment: Normalization and ROI Cropping Function
def get_normalized_depth(model, img_rgb, roi_h):
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = model(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()

    # Normalize values to [0, 1] range
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0  # Black out the dashboard area

    return depth_norm

darkness_levels = [0.9, 0.7, 0.5, 0.15, 0.05]
m_snr, m_rmse_orig, m_rmse_enh, m_rmse_ft = [], [], [], []

print("Extracting depths and measuring performance...")

for factor in darkness_levels:
    l_snr, l_orig, l_enh, l_ft = [], [], [], []

    for img_path, _ in selected_data:
        img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        roi_h = int(img_rgb.shape[0] * 0.8)

        # 1. Clean Baseline (Normalized)
        base_depth = get_normalized_depth(original_midas, img_rgb, roi_h)

        # 2. Apply Darkness Distortion
        dark_img = apply_realistic_darkness(img_rgb, factor)
        l_snr.append(compute_snr(img_rgb, dark_img))

        # 3. Apply Classical Enhancement
        enhanced_img = apply_enhancement_darkness(dark_img)

        # 4. Execute all three scenarios (Ensuring all outputs are normalized)
        depth_orig = get_normalized_depth(original_midas, dark_img, roi_h)
        depth_enh = get_normalized_depth(original_midas, enhanced_img, roi_h)
        depth_ft = get_normalized_depth(finetuned_midas, dark_img, roi_h)

        # 5. Calculate RMSE (Strictly on the Region of Interest)
        roi_base = base_depth[:roi_h, :]
        l_orig.append(np.sqrt(np.mean((roi_base - depth_orig[:roi_h, :]) ** 2)))
        l_enh.append(np.sqrt(np.mean((roi_base - depth_enh[:roi_h, :]) ** 2)))
        l_ft.append(np.sqrt(np.mean((roi_base - depth_ft[:roi_h, :]) ** 2)))

    m_snr.append(np.mean(l_snr))
    m_rmse_orig.append(np.mean(l_orig))
    m_rmse_enh.append(np.mean(l_enh))
    m_rmse_ft.append(np.mean(l_ft))

print("Plotting the Ultimate Graph...")
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(m_snr, m_rmse_orig, marker='X', markersize=10, color='#d62728', linewidth=3, label='Original MiDaS (Distorted)')
ax.plot(m_snr, m_rmse_enh, marker='o', markersize=10, color='#2ca02c', linewidth=3, label='Enhanced MiDaS (CLAHE)')
ax.plot(m_snr, m_rmse_ft, marker='^', markersize=10, color='#9467bd', linewidth=3, label='Fine-Tuned MiDaS (Distorted)')

ax.set_xlabel(r'SNR (dB) $\rightarrow$ More Darkness', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean RMSE (Lower is Better)', fontsize=12, fontweight='bold')
ax.invert_xaxis()
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(fontsize=12)

for i, factor in enumerate(darkness_levels):
    ax.annotate(f"F:{factor}", (m_snr[i], m_rmse_ft[i]), textcoords="offset points", xytext=(0,-15), ha='center', fontsize=10, fontweight='bold', color='#9467bd')

plt.title("MiDaS Depth Perception: Original vs. Enhanced vs. Fine-Tuned\nProject by: Ayelet & Noa", fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
#@title Task 3: The Grand Finale - SSIM Evaluation Graph

import gc
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

print("Initializing final SSIM evaluation with NORMALIZED metrics...")

torch.cuda.empty_cache()
gc.collect()

device = torch.device('cuda')

# Load Models
original_midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()

finetuned_midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small")
# Ensure the path matches your trained model (midas_finetuned.pth or midas_finetuned_v2.pth)
finetuned_midas.load_state_dict(torch.load('/content/midas_finetuned_dark_smart.pth'))
finetuned_midas.to(device).eval()

midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
transform = midas_transforms.small_transform

# Utility Functions
def compute_snr(clean_rgb, dist_rgb):
    clean, dist = clean_rgb.astype(np.float64), dist_rgb.astype(np.float64)
    noise_power = np.mean((clean - dist) ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)

def apply_realistic_darkness(img_rgb, darkness_factor):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * darkness_factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

def apply_enhancement_darkness(dark_img):
    lab = cv2.cvtColor(dark_img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

def get_normalized_depth(model, img_rgb, roi_h):
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = model(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()

    # Normalize values to [0, 1] range
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

def compute_ssim(depth1, depth2, roi_h):
    # Calculate the Structural Similarity Index strictly on the Region of Interest (ROI)
    roi_d1 = depth1[:roi_h, :]
    roi_d2 = depth2[:roi_h, :]
    return ssim(roi_d1, roi_d2, data_range=1.0)

darkness_levels = [0.9, 0.7, 0.5, 0.15, 0.05]
m_snr, m_ssim_orig, m_ssim_enh, m_ssim_ft = [], [], [], []

print("Extracting depths and measuring SSIM performance...")

for factor in darkness_levels:
    l_snr, l_orig, l_enh, l_ft = [], [], [], []

    for img_path, _ in selected_data:
        img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        roi_h = int(img_rgb.shape[0] * 0.8)

        # 1. Clean Baseline
        base_depth = get_normalized_depth(original_midas, img_rgb, roi_h)

        # 2. Apply Darkness Distortion
        dark_img = apply_realistic_darkness(img_rgb, factor)
        l_snr.append(compute_snr(img_rgb, dark_img))

        # 3. Apply Classical Enhancement
        enhanced_img = apply_enhancement_darkness(dark_img)

        # 4. Extract Depth Maps
        depth_orig = get_normalized_depth(original_midas, dark_img, roi_h)
        depth_enh = get_normalized_depth(original_midas, enhanced_img, roi_h)
        depth_ft = get_normalized_depth(finetuned_midas, dark_img, roi_h)

        # 5. Calculate SSIM
        l_orig.append(compute_ssim(base_depth, depth_orig, roi_h))
        l_enh.append(compute_ssim(base_depth, depth_enh, roi_h))
        l_ft.append(compute_ssim(base_depth, depth_ft, roi_h))

    m_snr.append(np.mean(l_snr))
    m_ssim_orig.append(np.mean(l_orig))
    m_ssim_enh.append(np.mean(l_enh))
    m_ssim_ft.append(np.mean(l_ft))

print("Plotting the SSIM Graph...")
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(m_snr, m_ssim_orig, marker='X', markersize=10, color='#d62728', linewidth=3, label='Original MiDaS (Distorted)')
ax.plot(m_snr, m_ssim_enh, marker='o', markersize=10, color='#2ca02c', linewidth=3, label='Enhanced MiDaS (CLAHE)')
ax.plot(m_snr, m_ssim_ft, marker='^', markersize=10, color='#9467bd', linewidth=3, label='Fine-Tuned MiDaS (Distorted)')

ax.set_xlabel(r'SNR (dB) $\rightarrow$ More Darkness', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean SSIM (Higher is Better)', fontsize=12, fontweight='bold')
ax.invert_xaxis()
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(fontsize=12)

for i, factor in enumerate(darkness_levels):
    ax.annotate(f"F:{factor}", (m_snr[i], m_ssim_ft[i]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=10, fontweight='bold', color='#9467bd')

plt.title("Structural Similarity Index (SSIM) under Low Light\nProject by: Ayelet & Noa", fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
#@title Task 3: Grand Finale - MiDaS Bar Chart Comparison (Extreme Darkness F:0.05)

import gc
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

print("Initializing Bar Chart Evaluation for Extreme Low Light (Factor: 0.05)...")

# Clear GPU runtime memory to ensure a fresh execution environment
torch.cuda.empty_cache()
gc.collect()

device = torch.device('cuda')

# Load models for benchmarking against a clean baseline environment
original_midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()

finetuned_midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small")
# Ensure the path matches the saved model filename (e.g., v2 or standard)
finetuned_midas.load_state_dict(torch.load('/content/midas_finetuned_dark_smart.pth'))
finetuned_midas.to(device).eval()

midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
transform = midas_transforms.small_transform

# --- Utility Functions ---
def apply_realistic_darkness(img_rgb, darkness_factor):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * darkness_factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

def apply_enhancement_darkness(dark_img):
    lab = cv2.cvtColor(dark_img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

def get_normalized_depth(model, img_rgb, roi_h):
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = model(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()

    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

def compute_metrics(depth1, depth2, roi_h):
    roi_d1 = depth1[:roi_h, :]
    roi_d2 = depth2[:roi_h, :]
    rmse_val = np.sqrt(np.mean((roi_d1 - roi_d2) ** 2))
    ssim_val = ssim(roi_d1, roi_d2, data_range=1.0)
    return rmse_val, ssim_val

# Define the extreme degradation level for testing
factor_hard = 0.05
l_r_orig, l_s_orig = [], []
l_r_enh, l_s_enh = [], []
l_r_ft, l_s_ft = [], []

print(f"Running Final Stand-off at darkness level {factor_hard}...")

for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    roi_h = int(img_rgb.shape[0] * 0.8)

    # Clean Baseline
    base_depth = get_normalized_depth(original_midas, img_rgb, roi_h)

    # Apply Degradation
    dark_img = apply_realistic_darkness(img_rgb, factor_hard)
    enhanced_img = apply_enhancement_darkness(dark_img)

    # Execute Models
    depth_orig = get_normalized_depth(original_midas, dark_img, roi_h)
    depth_enh = get_normalized_depth(original_midas, enhanced_img, roi_h)
    depth_ft = get_normalized_depth(finetuned_midas, dark_img, roi_h)

    # Calculate Metrics
    r_orig, s_orig = compute_metrics(base_depth, depth_orig, roi_h)
    r_enh, s_enh = compute_metrics(base_depth, depth_enh, roi_h)
    r_ft, s_ft = compute_metrics(base_depth, depth_ft, roi_h)

    l_r_orig.append(r_orig); l_s_orig.append(s_orig)
    l_r_enh.append(r_enh); l_s_enh.append(s_enh)
    l_r_ft.append(r_ft); l_s_ft.append(s_ft)

# Calculate averages for visualization
mean_rmse = [np.mean(l_r_orig), np.mean(l_r_enh), np.mean(l_r_ft)]
mean_ssim = [np.mean(l_s_orig), np.mean(l_s_enh), np.mean(l_s_ft)]

labels = ['Original\n(Distorted)', 'Enhanced\n(CLAHE)', 'Fine-Tuned\n(Distorted)']
colors = ['#d62728', '#2ca02c', '#9467bd']

print("Plotting the Grand Finale Bar Charts...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f"MiDaS Performance Under Extreme Low Light (Factor: {factor_hard})\nProject by: Ayelet & Noa", fontsize=18, fontweight='bold', y=1.05)

# --- Chart 1: RMSE ---
bars1 = ax1.bar(labels, mean_rmse, color=colors, width=0.5, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Mean RMSE (Lower is Better)', fontsize=12, fontweight='bold')
ax1.set_title("Mathematical Error (RMSE)", fontsize=14, fontweight='bold')
ax1.set_ylim(0, max(mean_rmse) * 1.2)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars1:
    height = bar.get_height()
    ax1.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 5), textcoords="offset points", ha='center', va='bottom', fontsize=12, fontweight='bold')

# --- Chart 2: SSIM ---
bars2 = ax2.bar(labels, mean_ssim, color=colors, width=0.5, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Mean SSIM (Higher is Better)', fontsize=12, fontweight='bold')
ax2.set_title("Structural Similarity Index (SSIM)", fontsize=14, fontweight='bold')
ax2.set_ylim(0, 1.0)
ax2.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars2:
    height = bar.get_height()
    ax2.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 5), textcoords="offset points", ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 3: Generating Gaussian Noise Intensity Sweep

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("Task 3: Generating Gaussian Noise Intensity Sweep (No Model)...")

# --- 1. Distortion Function: Gaussian Noise ---
def apply_gaussian_noise(img_rgb, sigma):
    # Generate a noise matrix matching the image dimensions
    noise = np.random.normal(0, sigma, img_rgb.shape)

    # Add noise to the original image (cast to float to prevent value overflow)
    noisy_img = img_rgb.astype(np.float32) + noise

    # Clip values to the valid pixel range [0, 255] and cast back to uint8
    noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)
    return noisy_img

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing! Please re-run the clean image selection block.")

# --- 2. Experimental Noise Levels (Sigma - Standard Deviation) ---
# Utilizing progressive levels, from light noise to heavy visual degradation (snow-like artifact)
noise_sigmas = [15, 30, 50, 75, 100]

# --- 3. Generate Visual Grid ---
fig, axes = plt.subplots(len(selected_data), 6, figsize=(24, 15))
fig.suptitle('Task 3: Distortion Intensity Sweep (Gaussian Noise)\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.98)

for row_idx, (img_path, _) in enumerate(selected_data):
    # Read and convert the original image
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Column 0: Original Image (Reference)
    # Mask (black out) the dashboard in the reference image to maintain consistency
    h, w = img_rgb.shape[:2]
    roi_h = int(h * 0.8)
    base_img = img_rgb.copy()
    base_img[roi_h:, :] = 0

    axes[row_idx, 0].imshow(base_img)
    if row_idx == 0:
        axes[row_idx, 0].set_title("Original\n(Sigma: 0)", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Columns 1-5: Apply incremental noise levels
    for col_idx, sigma in enumerate(noise_sigmas, start=1):
        # Apply Gaussian noise to the entire image
        noisy_img = apply_gaussian_noise(img_rgb, sigma)

        # Mask (black out) the dashboard ROI after noise application
        noisy_img[roi_h:, :] = 0

        axes[row_idx, col_idx].imshow(noisy_img)

        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"Sigma: {sigma}", fontsize=14)
        axes[row_idx, col_idx].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Visualizing MiDaS Depth Maps under Gaussian Noise

import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np

print("Visualizing MiDaS Depth Maps under Gaussian Noise...")

# --- 1. Environment and Model Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'midas' not in locals():
    print("Loading MiDaS_small...")
    midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()
    midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
    transform = midas_transforms.small_transform

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing!")

# --- 2. Utility and Distortion Functions ---
def apply_gaussian_noise(img_rgb, sigma):
    noise = np.random.normal(0, sigma, img_rgb.shape)
    noisy_img = img_rgb.astype(np.float32) + noise
    return np.clip(noisy_img, 0, 255).astype(np.uint8)

def get_normalized_depth(img_rgb, roi_h):
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = midas(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

# --- 3. Generate Visual Grid of Depth Maps ---
noise_sigmas = [15, 30, 50, 75, 100]

fig, axes = plt.subplots(len(selected_data), 6, figsize=(24, 15))
fig.suptitle('MiDaS Depth Maps Degradation under Gaussian Noise\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.98)

for row_idx, (img_path, _) in enumerate(selected_data):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    roi_h = int(img_rgb.shape[0] * 0.8)

    # Mask (black out) the dashboard in the reference image prior to depth calculation
    base_img = img_rgb.copy()
    base_img[roi_h:, :] = 0

    # Column 0: Reference (Clean baseline depth map)
    base_depth = get_normalized_depth(base_img, roi_h)
    axes[row_idx, 0].imshow(base_depth, cmap='magma')
    if row_idx == 0:
        axes[row_idx, 0].set_title("Baseline Depth\n(Sigma: 0)", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Columns 1-5: Depth maps under noise degradation
    for col_idx, sigma in enumerate(noise_sigmas, start=1):
        # 1. Apply noise to the entire image
        noisy_img = apply_gaussian_noise(img_rgb, sigma)
        # 2. Mask out the dashboard area
        noisy_img[roi_h:, :] = 0
        # 3. Execute depth model inference
        noisy_depth = get_normalized_depth(noisy_img, roi_h)

        axes[row_idx, col_idx].imshow(noisy_depth, cmap='magma')

        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"Depth at\nSigma: {sigma}", fontsize=14)
        axes[row_idx, col_idx].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title MiDaS Inference, RMSE & SSIM Evaluation on Gaussian Noise

import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import structural_similarity as ssim

print("MiDaS Inference, RMSE & SSIM Evaluation on Gaussian Noise...")

# --- 1. Environment and Model Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'midas' not in locals():
    print("Loading MiDaS_small...")
    midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()
    midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
    transform = midas_transforms.small_transform

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing!")

# --- 2. Mathematical Functions ---
def apply_gaussian_noise(img_rgb, sigma):
    noise = np.random.normal(0, sigma, img_rgb.shape)
    noisy_img = img_rgb.astype(np.float32) + noise
    return np.clip(noisy_img, 0, 255).astype(np.uint8)

def compute_snr(clean_rgb, dist_rgb):
    clean, dist = clean_rgb.astype(np.float64), dist_rgb.astype(np.float64)
    noise_power = np.mean((clean - dist) ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)

def get_normalized_depth(img_rgb, roi_h):
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = midas(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

def compute_metrics(depth1, depth2, roi_h):
    # Extract the Region of Interest (ROI)
    roi_d1 = depth1[:roi_h, :]
    roi_d2 = depth2[:roi_h, :]

    # Compute Root Mean Square Error (RMSE)
    mse = np.mean((roi_d1 - roi_d2) ** 2)
    rmse_val = np.sqrt(mse)

    # Compute Structural Similarity Index (SSIM) over the [0, 1] range
    ssim_val = ssim(roi_d1, roi_d2, data_range=1.0)

    return rmse_val, ssim_val

# --- 3. Experiment Execution ---
noise_sigmas = [15, 30, 50, 75, 100]
m_snr, m_rmse, m_ssim = [], [], []

print("Extracting Baseline Depth Maps...")
baseline_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    roi_h = int(img_rgb.shape[0] * 0.8)
    base_depth = get_normalized_depth(img_rgb, roi_h)
    baseline_data.append((img_rgb, base_depth, roi_h))

print("Running Model on Noisy Images & Calculating Metrics...")
for sigma in noise_sigmas:
    l_snr, l_rmse, l_ssim = [], [], []
    for img_rgb, base_depth, roi_h in baseline_data:
        noisy_img = apply_gaussian_noise(img_rgb, sigma)
        noisy_depth = get_normalized_depth(noisy_img, roi_h)

        l_snr.append(compute_snr(img_rgb, noisy_img))
        r_val, s_val = compute_metrics(base_depth, noisy_depth, roi_h)

        l_rmse.append(r_val)
        l_ssim.append(s_val)

    m_snr.append(np.mean(l_snr))
    m_rmse.append(np.mean(l_rmse))
    m_ssim.append(np.mean(l_ssim))

# --- 4. Plotting Subplots ---
print("Plotting Degradation Graphs...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("MiDaS Depth Perception Degradation under Gaussian Noise\nProject by: Ayelet & Noa", fontsize=18, fontweight='bold', y=1.05)

# Graph 1: RMSE (Increasing Error)
color1 = 'tab:red'
ax1.set_xlabel(r'SNR (dB) $\rightarrow$ More Noise', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean RMSE (Error - Lower is Better)', color=color1, fontsize=12, fontweight='bold')
ax1.plot(m_snr, m_rmse, marker='X', markersize=10, color=color1, linewidth=3, label='RMSE')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.invert_xaxis()
ax1.grid(True, linestyle='--', alpha=0.6)
for i, sigma in enumerate(noise_sigmas):
    ax1.annotate(f"$\sigma$:{sigma}", (m_snr[i], m_rmse[i]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, fontweight='bold')
ax1.set_title("Mathematical Error (RMSE)", fontsize=14, fontweight='bold')

# Graph 2: SSIM (Decreasing Structural Similarity)
color2 = 'tab:blue'
ax2.set_xlabel(r'SNR (dB) $\rightarrow$ More Noise', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean SSIM (Similarity - Higher is Better)', color=color2, fontsize=12, fontweight='bold')
ax2.plot(m_snr, m_ssim, marker='o', markersize=10, color=color2, linewidth=3, label='SSIM')
ax2.tick_params(axis='y', labelcolor=color2)
ax2.invert_xaxis()
ax2.grid(True, linestyle='--', alpha=0.6)
for i, sigma in enumerate(noise_sigmas):
    ax2.annotate(f"$\sigma$:{sigma}", (m_snr[i], m_ssim[i]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, fontweight='bold')
ax2.set_title("Structural Similarity Index (SSIM)", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 3: Classical Enhancement (Gaussian Blur) vs Gaussian Noise

import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import structural_similarity as ssim

print("Task 3: Classical Enhancement (Gaussian Blur) vs Gaussian Noise...")

# --- 1. Environment and Model Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'midas' not in locals():
    print("Loading MiDaS_small...")
    midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()
    midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
    transform = midas_transforms.small_transform

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing!")

np.random.seed(42)  # Set random seed for reproducibility, ensuring stable and report-ready graphs

# --- 2. Image Processing Functions ---
def apply_gaussian_noise(img_rgb, sigma):
    noise = np.random.normal(0, sigma, img_rgb.shape)
    noisy_img = img_rgb.astype(np.float32) + noise
    return np.clip(noisy_img, 0, 255).astype(np.uint8)

def apply_enhancement(noisy_img):
    # Gaussian blur applied for noise attenuation (Low-Pass Filter)
    return cv2.GaussianBlur(noisy_img, (7, 7), 1.5)

def compute_snr(clean_rgb, dist_rgb):
    clean, dist = clean_rgb.astype(np.float64), dist_rgb.astype(np.float64)
    noise_power = np.mean((clean - dist) ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)

def get_normalized_depth(img_rgb, roi_h):
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = midas(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

def compute_metrics(depth1, depth2, roi_h):
    roi_d1 = depth1[:roi_h, :]
    roi_d2 = depth2[:roi_h, :]
    rmse_val = np.sqrt(np.mean((roi_d1 - roi_d2) ** 2))
    ssim_val = ssim(roi_d1, roi_d2, data_range=1.0)
    return rmse_val, ssim_val

# --- 3. Run Comparative Experiment ---
noise_sigmas = [15, 30, 50, 75, 100]
m_snr, m_rmse_noisy, m_ssim_noisy = [], [], []
m_rmse_enh, m_ssim_enh = [], []

print("Extracting Baseline Depth Maps...")
baseline_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    roi_h = int(img_rgb.shape[0] * 0.8)
    base_depth = get_normalized_depth(img_rgb, roi_h)
    baseline_data.append((img_rgb, base_depth, roi_h))

print("Running Inference: Noisy vs. Enhanced...")
for sigma in noise_sigmas:
    l_snr, l_r_noisy, l_s_noisy, l_r_enh, l_s_enh = [], [], [], [], []
    for img_rgb, base_depth, roi_h in baseline_data:
        # 1. Apply distortion
        noisy_img = apply_gaussian_noise(img_rgb, sigma)
        # 2. Apply classical correction
        enhanced_img = apply_enhancement(noisy_img)

        # 3. Execute model on both versions
        noisy_depth = get_normalized_depth(noisy_img, roi_h)
        enhanced_depth = get_normalized_depth(enhanced_img, roi_h)

        # 4. Calculate metrics
        l_snr.append(compute_snr(img_rgb, noisy_img))  # SNR is consistently calculated against the original noise for unbiased evaluation

        r_n, s_n = compute_metrics(base_depth, noisy_depth, roi_h)
        r_e, s_e = compute_metrics(base_depth, enhanced_depth, roi_h)

        l_r_noisy.append(r_n); l_s_noisy.append(s_n)
        l_r_enh.append(r_e); l_s_enh.append(s_e)

    m_snr.append(np.mean(l_snr))
    m_rmse_noisy.append(np.mean(l_r_noisy)); m_ssim_noisy.append(np.mean(l_s_noisy))
    m_rmse_enh.append(np.mean(l_r_enh)); m_ssim_enh.append(np.mean(l_s_enh))

# --- 4. Plot Head-to-Head Comparative Graphs ---
print("Plotting Head-to-Head Comparison Graphs...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Classical Enhancement (Gaussian Blur) vs Gaussian Noise\nProject by: Ayelet & Noa", fontsize=18, fontweight='bold', y=1.05)

# RMSE Graph
ax1.set_xlabel(r'SNR (dB) $\rightarrow$ More Noise', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean RMSE (Lower is Better)', fontsize=12, fontweight='bold')
ax1.plot(m_snr, m_rmse_noisy, marker='X', markersize=10, color='tab:red', linewidth=3, label='Distorted (Noisy)')
ax1.plot(m_snr, m_rmse_enh, marker='o', markersize=10, color='tab:green', linewidth=3, label='Enhanced (Gaussian Blur)')
ax1.invert_xaxis()
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(fontsize=11)
ax1.set_title("Mathematical Error (RMSE)", fontsize=14, fontweight='bold')

# SSIM Graph
ax2.set_xlabel(r'SNR (dB) $\rightarrow$ More Noise', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean SSIM (Higher is Better)', fontsize=12, fontweight='bold')
ax2.plot(m_snr, m_ssim_noisy, marker='X', markersize=10, color='tab:blue', linewidth=3, label='Distorted (Noisy)')
ax2.plot(m_snr, m_ssim_enh, marker='o', markersize=10, color='tab:green', linewidth=3, label='Enhanced (Gaussian Blur)')
ax2.invert_xaxis()
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(fontsize=11)
ax2.set_title("Structural Similarity Index (SSIM)", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 3: MiDaS Fine-Tuning - Data Preparation (50/50 Split)

import random
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

print("Preparing Dataset with absolute 50/50 Mix...")

def apply_gaussian_noise(img_rgb, sigma):
    """Applies Gaussian noise to the image based on the specified sigma value."""
    noise = np.random.normal(0, sigma, img_rgb.shape)
    noisy_img = img_rgb.astype(np.float32) + noise
    return np.clip(noisy_img, 0, 255).astype(np.uint8)

class MiDaSGaussianDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = str(self.image_paths[idx])
        img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

        # Maintain aspect ratio (16:9) by resizing
        img_rgb = cv2.resize(img_rgb, (512, 288))

        # Ground truth derived from pristine images
        clean_input = self.transform(img_rgb).squeeze(0)

        # Strict 50% / 50% split between clean and distorted images
        if random.random() < 0.50:
            # Broad noise range to handle extreme tests at low SNR
            sigma = random.uniform(15, 120)
            mixed_rgb = apply_gaussian_noise(img_rgb, sigma)
        else:
            mixed_rgb = img_rgb

        mixed_input = self.transform(mixed_rgb).squeeze(0)
        return mixed_input, clean_input

# Dataset preparation
test_image_paths = [str(p) for p, _ in selected_data]
train_images = [p for p in all_images if str(p) not in test_image_paths][:3000]

train_dataset = MiDaSGaussianDataset(train_images, transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)

print(f"Dataset ready! Total training images: {len(train_dataset)}")

In [ ]:
#@title Task 3: MiDaS Fine-Tuning - RMSE Domination (Pure MSE, 50/50 Split)

import copy
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch

print("Initializing MiDaS: Pure MSE Optimization for Lowest RMSE...")

student_midas_gauss = copy.deepcopy(midas).to(device)
student_midas_gauss.train()
midas.eval()

# Unfreeze the entire network to allow it to learn the complete noise pattern
for param in student_midas_gauss.parameters():
    param.requires_grad = True

# Optimize RMSE directly using a pure MSE loss function
criterion_mse = nn.MSELoss()

# Use a stronger initial learning rate to reduce large errors, followed by gradual decay
optimizer = optim.AdamW(student_midas_gauss.parameters(), lr=1e-4, weight_decay=1e-4)

epochs = 15
scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

print(f"Starting Training for {epochs} epochs...")

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_idx, (mixed_inputs, clean_inputs) in enumerate(train_loader):
        mixed_inputs = mixed_inputs.to(device)
        clean_inputs = clean_inputs.to(device)

        with torch.no_grad():
            target_depths = midas(clean_inputs)

        optimizer.zero_grad()
        predictions = student_midas_gauss(mixed_inputs)

        # Focus exclusively on minimizing MSE
        loss = criterion_mse(predictions, target_depths)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        if batch_idx % 25 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"===> Epoch {epoch+1} Average Loss: {epoch_loss / len(train_loader):.4f} | Current LR: {current_lr:.6f}")

torch.save(student_midas_gauss.state_dict(), '/content/midas_finetuned_gaussian_rmse.pth')
print("Training Complete! Model saved to '/content/midas_finetuned_gaussian_rmse.pth'")

In [ ]:
#@title Task 3: The Ultimate Evaluation - Gaussian Noise (RMSE & SSIM)

import gc
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

print("Initializing comprehensive evaluation pipeline for Gaussian Noise...")

# Clear GPU memory to prevent Out-Of-Memory (OOM) errors during inference
torch.cuda.empty_cache()
gc.collect()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load the original pre-trained MiDaS model (Baseline)
original_midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()

# Load the fine-tuned MiDaS model specifically trained for Gaussian Noise robustness
finetuned_midas_gauss = torch.hub.load("isl-org/MiDaS", "MiDaS_small")
finetuned_midas_gauss.load_state_dict(torch.load('/content/midas_finetuned_gaussian_rmse.pth'))
finetuned_midas_gauss.to(device).eval()

# Load MiDaS transformation pipeline
midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
transform = midas_transforms.small_transform

# --- Helper Functions ---

def compute_snr(clean_rgb, dist_rgb):
    """Calculates the Signal-to-Noise Ratio (SNR) between clean and distorted images."""
    clean, dist = clean_rgb.astype(np.float64), dist_rgb.astype(np.float64)
    noise_power = np.mean((clean - dist) ** 2)
    if noise_power == 0: return np.inf
    return 10.0 * np.log10(np.mean(clean ** 2) / noise_power)

def apply_gaussian_noise(img_rgb, sigma):
    """Applies Gaussian noise to the image based on the specified sigma value."""
    noise = np.random.normal(0, sigma, img_rgb.shape)
    noisy_img = img_rgb.astype(np.float32) + noise
    return np.clip(noisy_img, 0, 255).astype(np.uint8)

def apply_enhancement(noisy_img):
    """Applies a classical Gaussian Blur filter (Low-Pass Filter) to mitigate noise."""
    return cv2.GaussianBlur(noisy_img, (7, 7), 1.5)

def get_normalized_depth(model, img_rgb, roi_h):
    """Extracts and normalizes the depth map, applying a mask to the car dashboard (ROI)."""
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = model(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()

    # Min-Max normalization (0 to 1 scale)
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)

    # Mask the dashboard region (set to 0) to avoid skewed metrics
    depth_norm[roi_h:, :] = 0
    return depth_norm

def compute_metrics(depth1, depth2, roi_h):
    """Computes RMSE and SSIM metrics specifically for the Region of Interest (ROI)."""
    roi_d1 = depth1[:roi_h, :]
    roi_d2 = depth2[:roi_h, :]
    rmse_val = np.sqrt(np.mean((roi_d1 - roi_d2) ** 2))
    ssim_val = ssim(roi_d1, roi_d2, data_range=1.0)
    return rmse_val, ssim_val

# --- Evaluation Execution ---

noise_sigmas = [15, 30, 50, 75, 100]
m_snr = []
m_rmse_orig, m_ssim_orig = [], []
m_rmse_enh, m_ssim_enh = [], []
m_rmse_ft, m_ssim_ft = [], []

print("Extracting depth maps and computing evaluation metrics...")

for sigma in noise_sigmas:
    l_snr = []
    l_r_orig, l_s_orig = [], []
    l_r_enh, l_s_enh = [], []
    l_r_ft, l_s_ft = [], []

    for img_path, _ in selected_data:
        img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        roi_h = int(img_rgb.shape[0] * 0.8)

        # Extract pristine baseline depth map
        base_depth = get_normalized_depth(original_midas, img_rgb, roi_h)

        # Apply noise distortion and compute SNR
        noisy_img = apply_gaussian_noise(img_rgb, sigma)
        l_snr.append(compute_snr(img_rgb, noisy_img))

        # Apply classical image enhancement
        enhanced_img = apply_enhancement(noisy_img)

        # Extract depth maps for all three evaluated scenarios
        depth_orig = get_normalized_depth(original_midas, noisy_img, roi_h)
        depth_enh = get_normalized_depth(original_midas, enhanced_img, roi_h)
        depth_ft = get_normalized_depth(finetuned_midas_gauss, noisy_img, roi_h)

        # Compute RMSE and SSIM against the pristine baseline
        r_orig, s_orig = compute_metrics(base_depth, depth_orig, roi_h)
        r_enh, s_enh = compute_metrics(base_depth, depth_enh, roi_h)
        r_ft, s_ft = compute_metrics(base_depth, depth_ft, roi_h)

        l_r_orig.append(r_orig); l_s_orig.append(s_orig)
        l_r_enh.append(r_enh); l_s_enh.append(s_enh)
        l_r_ft.append(r_ft); l_s_ft.append(s_ft)

    m_snr.append(np.mean(l_snr))
    m_rmse_orig.append(np.mean(l_r_orig)); m_ssim_orig.append(np.mean(l_s_orig))
    m_rmse_enh.append(np.mean(l_r_enh)); m_ssim_enh.append(np.mean(l_s_enh))
    m_rmse_ft.append(np.mean(l_r_ft)); m_ssim_ft.append(np.mean(l_s_ft))

# --- Plotting Results ---
print("Generating comprehensive head-to-head comparison graphs...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle("MiDaS Depth Perception: Original vs. Enhanced vs. Fine-Tuned (Gaussian Noise)\nProject by: Ayelet & Noa", fontsize=20, fontweight='bold', y=1.02)

# Define consistent colors for the graph traces
c_orig, c_enh, c_ft = '#d62728', '#2ca02c', '#9467bd'

# --- 1. Normalized RMSE Plot ---
ax1.plot(m_snr, m_rmse_orig, marker='X', markersize=10, color=c_orig, linewidth=3, label='Original MiDaS (Distorted)')
ax1.plot(m_snr, m_rmse_enh, marker='o', markersize=10, color=c_enh, linewidth=3, label='Enhanced MiDaS (Gaussian Blur)')
ax1.plot(m_snr, m_rmse_ft, marker='^', markersize=10, color=c_ft, linewidth=3, label='Fine-Tuned MiDaS (Distorted)')

ax1.set_xlabel(r'SNR (dB) $\rightarrow$ More Noise', fontsize=14, fontweight='bold')
ax1.set_ylabel('Mean RMSE (Lower is Better)', fontsize=14, fontweight='bold')
ax1.invert_xaxis()
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(fontsize=12)
ax1.set_title("Mathematical Error (Normalized RMSE)", fontsize=16, fontweight='bold')

# Annotate data points with their respective Sigma values
for i, sigma in enumerate(noise_sigmas):
    ax1.annotate(f"S:{sigma}", (m_snr[i], m_rmse_ft[i]), textcoords="offset points", xytext=(0,-18), ha='center', fontsize=10, fontweight='bold', color=c_ft)

# --- 2. SSIM Plot ---
ax2.plot(m_snr, m_ssim_orig, marker='X', markersize=10, color=c_orig, linewidth=3, label='Original MiDaS (Distorted)')
ax2.plot(m_snr, m_ssim_enh, marker='o', markersize=10, color=c_enh, linewidth=3, label='Enhanced MiDaS (Gaussian Blur)')
ax2.plot(m_snr, m_ssim_ft, marker='^', markersize=10, color=c_ft, linewidth=3, label='Fine-Tuned MiDaS (Distorted)')

ax2.set_xlabel(r'SNR (dB) $\rightarrow$ More Noise', fontsize=14, fontweight='bold')
ax2.set_ylabel('Mean SSIM (Higher is Better)', fontsize=14, fontweight='bold')
ax2.invert_xaxis()
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(fontsize=12)
ax2.set_title("Structural Similarity Index (SSIM)", fontsize=16, fontweight='bold')

# Annotate data points with their respective Sigma values
for i, sigma in enumerate(noise_sigmas):
    ax2.annotate(f"S:{sigma}", (m_snr[i], m_ssim_ft[i]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=10, fontweight='bold', color=c_ft)

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 3: The Ultimate Evaluation - Gaussian Noise (Bar Charts)

import gc
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

print("Initializing evaluation pipeline and Bar Chart generation...")

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load Baseline
original_midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()

# Load our winning Fine-Tuned model
finetuned_midas_gauss = torch.hub.load("isl-org/MiDaS", "MiDaS_small")
finetuned_midas_gauss.load_state_dict(torch.load('/content/midas_finetuned_gaussian_rmse.pth'))
finetuned_midas_gauss.to(device).eval()

midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
transform = midas_transforms.small_transform

# --- Helper Functions ---
def apply_gaussian_noise(img_rgb, sigma):
    noise = np.random.normal(0, sigma, img_rgb.shape)
    noisy_img = img_rgb.astype(np.float32) + noise
    return np.clip(noisy_img, 0, 255).astype(np.uint8)

def apply_enhancement(noisy_img):
    return cv2.GaussianBlur(noisy_img, (7, 7), 1.5)

def get_normalized_depth(model, img_rgb, roi_h):
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = model(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

def compute_metrics(depth1, depth2, roi_h):
    roi_d1 = depth1[:roi_h, :]
    roi_d2 = depth2[:roi_h, :]
    rmse_val = np.sqrt(np.mean((roi_d1 - roi_d2) ** 2))
    ssim_val = ssim(roi_d1, roi_d2, data_range=1.0)
    return rmse_val, ssim_val

# --- Evaluation Execution ---
noise_sigmas = [15, 30, 50, 75, 100]
m_rmse_orig, m_ssim_orig = [], []
m_rmse_enh, m_ssim_enh = [], []
m_rmse_ft, m_ssim_ft = [], []

print("Extracting depth maps and computing evaluation metrics...")

for sigma in noise_sigmas:
    l_r_orig, l_s_orig = [], []
    l_r_enh, l_s_enh = [], []
    l_r_ft, l_s_ft = [], []

    for img_path, _ in selected_data:
        img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        roi_h = int(img_rgb.shape[0] * 0.8)

        base_depth = get_normalized_depth(original_midas, img_rgb, roi_h)
        noisy_img = apply_gaussian_noise(img_rgb, sigma)
        enhanced_img = apply_enhancement(noisy_img)

        depth_orig = get_normalized_depth(original_midas, noisy_img, roi_h)
        depth_enh = get_normalized_depth(original_midas, enhanced_img, roi_h)
        depth_ft = get_normalized_depth(finetuned_midas_gauss, noisy_img, roi_h)

        r_orig, s_orig = compute_metrics(base_depth, depth_orig, roi_h)
        r_enh, s_enh = compute_metrics(base_depth, depth_enh, roi_h)
        r_ft, s_ft = compute_metrics(base_depth, depth_ft, roi_h)

        l_r_orig.append(r_orig); l_s_orig.append(s_orig)
        l_r_enh.append(r_enh); l_s_enh.append(s_enh)
        l_r_ft.append(r_ft); l_s_ft.append(s_ft)

    m_rmse_orig.append(np.mean(l_r_orig)); m_ssim_orig.append(np.mean(l_s_orig))
    m_rmse_enh.append(np.mean(l_r_enh)); m_ssim_enh.append(np.mean(l_s_enh))
    m_rmse_ft.append(np.mean(l_r_ft)); m_ssim_ft.append(np.mean(l_s_ft))

# --- Plotting Grouped Bar Charts ---
print("Generating comprehensive head-to-head Bar Charts...")

labels = [f'Sigma {s}' for s in noise_sigmas]
x = np.arange(len(labels))  # The label locations
width = 0.25  # The width of the bars

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle("MiDaS Depth Perception: Original vs. Enhanced vs. Fine-Tuned (Gaussian Noise)\nProject by: Ayelet & Noa", fontsize=20, fontweight='bold', y=1.02)

c_orig, c_enh, c_ft = '#d62728', '#2ca02c', '#9467bd'

# --- 1. Normalized RMSE Bar Chart ---
rects1_rmse = ax1.bar(x - width, m_rmse_orig, width, label='Original MiDaS (Distorted)', color=c_orig, edgecolor='black', alpha=0.9)
rects2_rmse = ax1.bar(x, m_rmse_enh, width, label='Enhanced MiDaS (Gaussian Blur)', color=c_enh, edgecolor='black', alpha=0.9)
rects3_rmse = ax1.bar(x + width, m_rmse_ft, width, label='Fine-Tuned MiDaS (Distorted)', color=c_ft, edgecolor='black', alpha=0.9)

ax1.set_ylabel('Mean RMSE (Lower is Better)', fontsize=14, fontweight='bold')
ax1.set_title("Mathematical Error (Normalized RMSE)", fontsize=16, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=12, fontweight='bold')
ax1.legend(fontsize=12, loc='upper left')
ax1.grid(axis='y', linestyle='--', alpha=0.6)

# --- 2. SSIM Bar Chart ---
rects1_ssim = ax2.bar(x - width, m_ssim_orig, width, label='Original MiDaS (Distorted)', color=c_orig, edgecolor='black', alpha=0.9)
rects2_ssim = ax2.bar(x, m_ssim_enh, width, label='Enhanced MiDaS (Gaussian Blur)', color=c_enh, edgecolor='black', alpha=0.9)
rects3_ssim = ax2.bar(x + width, m_ssim_ft, width, label='Fine-Tuned MiDaS (Distorted)', color=c_ft, edgecolor='black', alpha=0.9)

ax2.set_ylabel('Mean SSIM (Higher is Better)', fontsize=14, fontweight='bold')
ax2.set_title("Structural Similarity Index (SSIM)", fontsize=16, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, fontsize=12, fontweight='bold')
ax2.legend(fontsize=12, loc='lower left')
ax2.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 3: Generating JPEG Compression Sweep

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("Task 3: Generating JPEG Compression Sweep (No Model)...")

# --- 1. Distortion Function: JPEG Compression ---
def apply_jpeg_compression(img_rgb, quality):
    # Convert from RGB to BGR since OpenCV's encoding function expects this format
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

    # Set the compression quality parameter
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]

    # Encode (virtual compression into memory) and decode back to apply the distortion artifacts
    _, encimg = cv2.imencode('.jpg', img_bgr, encode_param)
    decimg = cv2.imdecode(encimg, 1)

    # Convert back to RGB for correct visualization in the plot
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing! Please re-run the clean image selection block.")

# --- 2. Experimental Compression Levels ---
# Scale down from a quality of 30 (poor) to 2 (barely recognizable)
jpeg_qualities = [30, 20, 10, 5, 2]

# --- 3. Generate Visual Grid ---
fig, axes = plt.subplots(len(selected_data), 6, figsize=(24, 15))
fig.suptitle('Distortion Intensity Sweep (JPEG Compression)\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.98)

for row_idx, (img_path, _) in enumerate(selected_data):
    # Read and convert the original image
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    roi_h = int(h * 0.8)

    # Column 0: Reference (pre-masking the dashboard)
    base_img = img_rgb.copy()
    base_img[roi_h:, :] = 0
    axes[row_idx, 0].imshow(base_img)
    if row_idx == 0:
        axes[row_idx, 0].set_title("Original\n(Quality: 100)", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Columns 1-5: Apply incremental JPEG compression levels
    for col_idx, quality in enumerate(jpeg_qualities, start=1):
        comp_img = apply_jpeg_compression(img_rgb, quality)
        comp_img[roi_h:, :] = 0

        axes[row_idx, col_idx].imshow(comp_img)
        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"Quality: {quality}", fontsize=14)
        axes[row_idx, col_idx].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Visualizing MiDaS Depth Maps under JPEG Compression

import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np

print("Visualizing MiDaS Depth Maps under JPEG Compression...")

# --- 1. Environment and Model Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'midas' not in locals():
    print("Loading MiDaS_small...")
    midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()
    midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
    transform = midas_transforms.small_transform

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing!")

# --- 2. Utility and Distortion Functions ---
def apply_jpeg_compression(img_rgb, quality):
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encimg = cv2.imencode('.jpg', img_bgr, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

def get_normalized_depth(img_rgb, roi_h):
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = midas(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

# --- 3. Generate Visual Grid of Depth Maps ---
jpeg_qualities = [30, 20, 10, 5, 2]

fig, axes = plt.subplots(len(selected_data), 6, figsize=(24, 15))
fig.suptitle('MiDaS Depth Maps Degradation under JPEG Compression\nProject by: Ayelet & Noa', fontsize=20, fontweight='bold', y=0.98)

for row_idx, (img_path, _) in enumerate(selected_data):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    roi_h = int(img_rgb.shape[0] * 0.8)

    # Mask (black out) the dashboard in the reference image prior to depth calculation
    base_img = img_rgb.copy()
    base_img[roi_h:, :] = 0

    # Column 0: Reference (Clean baseline depth map)
    base_depth = get_normalized_depth(base_img, roi_h)
    axes[row_idx, 0].imshow(base_depth, cmap='magma')
    if row_idx == 0:
        axes[row_idx, 0].set_title("Baseline Depth\n(Quality: 100)", fontsize=14, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Columns 1-5: Depth maps under compression degradation
    for col_idx, quality in enumerate(jpeg_qualities, start=1):
        # 1. Apply compression to the entire image
        comp_img = apply_jpeg_compression(img_rgb, quality)
        # 2. Mask out the dashboard area
        comp_img[roi_h:, :] = 0
        # 3. Execute depth model inference
        comp_depth = get_normalized_depth(comp_img, roi_h)

        axes[row_idx, col_idx].imshow(comp_depth, cmap='magma')

        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"Depth at\nQuality: {quality}", fontsize=14)
        axes[row_idx, col_idx].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title Task 3: Side-by-Side JPEG Evaluation Sweep (Line Charts)

import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import structural_similarity as ssim

print("MiDaS Inference & Evaluation on JPEG Compression...")

# --- 1. Environment and Model Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'midas' not in locals():
    print("Loading MiDaS_small...")
    midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()
    midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
    transform = midas_transforms.small_transform

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing!")

# --- 2. Image Processing and Metrics Functions ---
def apply_jpeg_compression(img_rgb, quality):
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encimg = cv2.imencode('.jpg', img_bgr, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

def get_normalized_depth(model, img_rgb, roi_h):
    model.eval()
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = model(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

def compute_metrics(depth1, depth2, roi_h):
    roi_d1 = depth1[:roi_h, :]
    roi_d2 = depth2[:roi_h, :]
    rmse_val = np.sqrt(np.mean((roi_d1 - roi_d2) ** 2))
    ssim_val = ssim(roi_d1, roi_d2, data_range=1.0)
    return rmse_val, ssim_val

# --- 3. Experiment Execution ---
jpeg_qualities = [30, 20, 10, 5, 2]  # From reasonable quality to severe degradation
m_rmse, m_ssim = [], []

print("Extracting Baseline Depth Maps...")
baseline_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    roi_h = int(img_rgb.shape[0] * 0.8)
    base_depth = get_normalized_depth(midas, img_rgb, roi_h)
    baseline_data.append((img_rgb, base_depth, roi_h))

print("Running Model on JPEG Compressed Images...")
for quality in jpeg_qualities:
    l_rmse, l_ssim = [], []
    for img_rgb, base_depth, roi_h in baseline_data:
        comp_img = apply_jpeg_compression(img_rgb, quality)
        comp_depth = get_normalized_depth(midas, comp_img, roi_h)

        r_val, s_val = compute_metrics(base_depth, comp_depth, roi_h)
        l_rmse.append(r_val)
        l_ssim.append(s_val)

    m_rmse.append(np.mean(l_rmse))
    m_ssim.append(np.mean(l_ssim))

# --- 4. Plotting Subplots ---
print("Plotting Degradation Graphs...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("MiDaS Depth Perception Degradation under JPEG Compression\nProject by: Ayelet & Noa", fontsize=18, fontweight='bold', y=1.05)

# Graph 1: RMSE (Increasing Error)
color1 = 'tab:red'
ax1.set_xlabel(r'JPEG Quality $\rightarrow$ Worse Quality', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean RMSE (Error - Lower is Better)', color=color1, fontsize=12, fontweight='bold')
ax1.plot(jpeg_qualities, m_rmse, marker='X', markersize=10, color=color1, linewidth=3, label='RMSE')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.invert_xaxis()  # Lower quality (worse) will be on the right side
ax1.grid(True, linestyle='--', alpha=0.6)
for i, q in enumerate(jpeg_qualities):
    ax1.annotate(f"Q:{q}", (jpeg_qualities[i], m_rmse[i]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, fontweight='bold')
ax1.set_title("Mathematical Error (RMSE)", fontsize=14, fontweight='bold')

# Graph 2: SSIM (Decreasing Structural Similarity)
color2 = 'tab:blue'
ax2.set_xlabel(r'JPEG Quality $\rightarrow$ Worse Quality', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean SSIM (Similarity - Higher is Better)', color=color2, fontsize=12, fontweight='bold')
ax2.plot(jpeg_qualities, m_ssim, marker='o', markersize=10, color=color2, linewidth=3, label='SSIM')
ax2.tick_params(axis='y', labelcolor=color2)
ax2.invert_xaxis()  # Lower quality will be on the right side
ax2.grid(True, linestyle='--', alpha=0.6)
for i, q in enumerate(jpeg_qualities):
    ax2.annotate(f"Q:{q}", (jpeg_qualities[i], m_ssim[i]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, fontweight='bold')
ax2.set_title("Structural Similarity Index (SSIM)", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
#@title Classical Enhancement (Gaussian Blur) vs JPEG Compression

import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import structural_similarity as ssim

print("Classical Enhancement (Gaussian Blur) vs JPEG Compression...")

# --- 1. Environment and Model Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'midas' not in locals():
    print("Loading MiDaS_small...")
    midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()
    midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
    transform = midas_transforms.small_transform

if 'selected_data' not in locals() or len(selected_data) == 0:
    raise ValueError("ERROR: 'selected_data' is missing!")

# --- 2. Image Processing and Metrics Functions ---
def apply_jpeg_compression(img_rgb, quality):
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encimg = cv2.imencode('.jpg', img_bgr, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

def apply_enhancement(distorted_img):
    # Classical filter applied to mitigate degradation: Gaussian Blur
    return cv2.GaussianBlur(distorted_img, (7, 7), 1.5)

def get_normalized_depth(model, img_rgb, roi_h):
    model.eval()
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = model(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

def compute_metrics(depth1, depth2, roi_h):
    roi_d1 = depth1[:roi_h, :]
    roi_d2 = depth2[:roi_h, :]
    rmse_val = np.sqrt(np.mean((roi_d1 - roi_d2) ** 2))
    ssim_val = ssim(roi_d1, roi_d2, data_range=1.0)
    return rmse_val, ssim_val

# --- 3. Run Comparative Experiment ---
jpeg_qualities = [30, 20, 10, 5, 2]
m_rmse_dist, m_ssim_dist = [], []
m_rmse_enh, m_ssim_enh = [], []

print("Extracting Baseline Depth Maps...")
baseline_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    roi_h = int(img_rgb.shape[0] * 0.8)
    base_depth = get_normalized_depth(midas, img_rgb, roi_h)
    baseline_data.append((img_rgb, base_depth, roi_h))

print("Running Inference: Distorted (JPEG) vs. Enhanced (Blur)...")
for quality in jpeg_qualities:
    l_r_dist, l_s_dist, l_r_enh, l_s_enh = [], [], [], []

    for img_rgb, base_depth, roi_h in baseline_data:
        # 1. Apply Distortion (JPEG)
        distorted_img = apply_jpeg_compression(img_rgb, quality)
        # 2. Apply Classical Enhancement
        enhanced_img = apply_enhancement(distorted_img)

        # 3. Execute Model Inference on Both Versions
        depth_dist = get_normalized_depth(midas, distorted_img, roi_h)
        depth_enh = get_normalized_depth(midas, enhanced_img, roi_h)

        # 4. Calculate Metrics
        r_d, s_d = compute_metrics(base_depth, depth_dist, roi_h)
        r_e, s_e = compute_metrics(base_depth, depth_enh, roi_h)

        l_r_dist.append(r_d); l_s_dist.append(s_d)
        l_r_enh.append(r_e); l_s_enh.append(s_e)

    m_rmse_dist.append(np.mean(l_r_dist)); m_ssim_dist.append(np.mean(l_s_dist))
    m_rmse_enh.append(np.mean(l_r_enh)); m_ssim_enh.append(np.mean(l_s_enh))

# --- 4. Plot Head-to-Head Comparative Graphs ---
print("Plotting Head-to-Head Comparison Graphs...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Classical Enhancement (Gaussian Blur) vs JPEG Compression\nProject by: Ayelet & Noa", fontsize=18, fontweight='bold', y=1.05)

# RMSE Graph
ax1.set_xlabel(r'JPEG Quality $\rightarrow$ Worse Quality', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean RMSE (Lower is Better)', fontsize=12, fontweight='bold')
ax1.plot(jpeg_qualities, m_rmse_dist, marker='X', markersize=10, color='tab:red', linewidth=3, label='Distorted (JPEG)')
ax1.plot(jpeg_qualities, m_rmse_enh, marker='o', markersize=10, color='tab:green', linewidth=3, label='Enhanced (Gaussian Blur)')
ax1.invert_xaxis()  # Lower (worse) quality levels are positioned on the right
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(fontsize=11)
ax1.set_title("Mathematical Error (RMSE)", fontsize=14, fontweight='bold')

# SSIM Graph
ax2.set_xlabel(r'JPEG Quality $\rightarrow$ Worse Quality', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean SSIM (Higher is Better)', fontsize=12, fontweight='bold')
ax2.plot(jpeg_qualities, m_ssim_dist, marker='X', markersize=10, color='tab:blue', linewidth=3, label='Distorted (JPEG)')
ax2.plot(jpeg_qualities, m_ssim_enh, marker='o', markersize=10, color='tab:green', linewidth=3, label='Enhanced (Gaussian Blur)')
ax2.invert_xaxis()  # Lower quality levels are positioned on the right
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(fontsize=11)
ax2.set_title("Structural Similarity Index (SSIM)", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 3: MiDaS Dataset Prep - JPEG Compression (50/50 Mix, Fixed Range 2-100)

import random
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

print("Preparing Dataset with absolute 50/50 Mix and FULL Quality Range...")

def apply_jpeg_compression(img_rgb, quality):
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encimg = cv2.imencode('.jpg', img_bgr, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

class MiDaSJPEGDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = str(self.image_paths[idx])
        img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

        # Maintain aspect ratio (16:9) by resizing
        img_rgb = cv2.resize(img_rgb, (512, 288))
        clean_input = self.transform(img_rgb).squeeze(0)

        # Strict 50% / 50% split for robust learning against block artifacts
        if random.random() < 0.50:
            # THE FIX: Broad range 2 to 100 so it learns not to blur good images!
            quality = int(random.uniform(2, 100))
            mixed_rgb = apply_jpeg_compression(img_rgb, quality)
        else:
            mixed_rgb = img_rgb

        mixed_input = self.transform(mixed_rgb).squeeze(0)
        return mixed_input, clean_input

test_image_paths = [str(p) for p, _ in selected_data]
train_images = [p for p in all_images if str(p) not in test_image_paths][:3000]

train_dataset = MiDaSJPEGDataset(train_images, transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)

print(f"Dataset ready! Total training images: {len(train_dataset)}")

In [ ]:
#@title Task 3: MiDaS Fine-Tuning - JPEG Smart Denoising (SmoothL1, Full Range)

import copy
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch

print("Initializing MiDaS: Smart Denoising for JPEG...")

# Create a deep copy of the pre-trained MiDaS model to serve as the student model
student_midas_jpeg = copy.deepcopy(midas).to(device)
student_midas_jpeg.train()
midas.eval()

# Unfreeze the entire network to allow comprehensive feature adaptation against JPEG block artifacts
for param in student_midas_jpeg.parameters():
    param.requires_grad = True

# Implement SmoothL1Loss (Huber Loss) as the primary optimization objective.
# This objective behaves like L1 for small errors (preserving sharp structural edges at high quality, e.g., Q=30),
# and like MSE for large errors (effectively suppressing severe block artifacts at low quality, e.g., Q=2).
criterion_smart = nn.SmoothL1Loss(beta=0.1)

# Configure the AdamW optimizer with a strong initial learning rate and weight decay for regularization
optimizer = optim.AdamW(student_midas_jpeg.parameters(), lr=1e-4, weight_decay=1e-4)

# Set up a Cosine Annealing learning rate scheduler for smooth convergence
epochs = 15
scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

print(f"Starting Training for {epochs} epochs...")

for epoch in range(epochs):
    epoch_loss = 0.0

    # Iterate through the DataLoader containing the 50/50 mixed dataset (clean vs. compressed)
    for batch_idx, (mixed_inputs, clean_inputs) in enumerate(train_loader):
        mixed_inputs = mixed_inputs.to(device)
        clean_inputs = clean_inputs.to(device)

        # Generate ground-truth depth maps using the frozen teacher model on pristine inputs
        with torch.no_grad():
            target_depths = midas(clean_inputs)

        optimizer.zero_grad()

        # Forward pass: Generate depth predictions from the compressed/mixed inputs
        predictions = student_midas_jpeg(mixed_inputs)

        # Compute the loss using the robust Smooth L1 objective
        loss = criterion_smart(predictions, target_depths)

        # Backward pass: Compute gradients and update model weights
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # Log training progress at regular intervals
        if batch_idx % 25 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    # Step the learning rate scheduler at the end of each epoch
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"===> Epoch {epoch+1} Average Loss: {epoch_loss / len(train_loader):.4f} | Current LR: {current_lr:.6f}")

# Persist the fine-tuned model weights to disk for subsequent evaluation
torch.save(student_midas_jpeg.state_dict(), '/content/midas_finetuned_jpeg_smart.pth')
print("Training Complete! Model saved to '/content/midas_finetuned_jpeg_smart.pth'")

In [ ]:
#@title Task 3: Evaluation - Original vs. Enhanced vs. Fine-Tuned (JPEG Compression)

import gc
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

print("Initializing full evaluation pipeline for JPEG Compression...")

# Prevent GPU memory leaks
torch.cuda.empty_cache()
gc.collect()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

original_midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()

finetuned_midas_jpeg = torch.hub.load("isl-org/MiDaS", "MiDaS_small")
finetuned_midas_jpeg.load_state_dict(torch.load('/content/midas_finetuned_jpeg_smart.pth'))
finetuned_midas_jpeg.to(device).eval()

midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
transform = midas_transforms.small_transform

# --- Helper Functions ---
def apply_jpeg_compression(img_rgb, quality):
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encimg = cv2.imencode('.jpg', img_bgr, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

def apply_enhancement(distorted_img):
    # Gaussian Blur as the classical baseline for artifact smoothing
    return cv2.GaussianBlur(distorted_img, (7, 7), 1.5)

def get_normalized_depth(model, img_rgb, roi_h):
    model.eval()
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = model(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

def compute_metrics(depth1, depth2, roi_h):
    roi_d1 = depth1[:roi_h, :]
    roi_d2 = depth2[:roi_h, :]
    rmse_val = np.sqrt(np.mean((roi_d1 - roi_d2) ** 2))
    ssim_val = ssim(roi_d1, roi_d2, data_range=1.0)
    return rmse_val, ssim_val

# --- Execution ---
jpeg_qualities = [30, 20, 10, 5, 2]
m_rmse_orig, m_ssim_orig = [], []
m_rmse_enh, m_ssim_enh = [], []
m_rmse_ft, m_ssim_ft = [], []

print("Extracting Baseline Depth Maps...")
baseline_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    roi_h = int(img_rgb.shape[0] * 0.8)
    base_depth = get_normalized_depth(original_midas, img_rgb, roi_h)
    baseline_data.append((img_rgb, base_depth, roi_h))

print("Running Head-to-Head Inference...")
for quality in jpeg_qualities:
    l_r_orig, l_s_orig, l_r_enh, l_s_enh, l_r_ft, l_s_ft = [], [], [], [], [], []

    for img_rgb, base_depth, roi_h in baseline_data:
        comp_img = apply_jpeg_compression(img_rgb, quality)
        enhanced_img = apply_enhancement(comp_img)

        depth_orig = get_normalized_depth(original_midas, comp_img, roi_h)
        depth_enh = get_normalized_depth(original_midas, enhanced_img, roi_h)
        depth_ft = get_normalized_depth(finetuned_midas_jpeg, comp_img, roi_h)

        r_orig, s_orig = compute_metrics(base_depth, depth_orig, roi_h)
        r_enh, s_enh = compute_metrics(base_depth, depth_enh, roi_h)
        r_ft, s_ft = compute_metrics(base_depth, depth_ft, roi_h)

        l_r_orig.append(r_orig); l_s_orig.append(s_orig)
        l_r_enh.append(r_enh); l_s_enh.append(s_enh)
        l_r_ft.append(r_ft); l_s_ft.append(s_ft)

    m_rmse_orig.append(np.mean(l_r_orig)); m_ssim_orig.append(np.mean(l_s_orig))
    m_rmse_enh.append(np.mean(l_r_enh)); m_ssim_enh.append(np.mean(l_s_enh))
    m_rmse_ft.append(np.mean(l_r_ft)); m_ssim_ft.append(np.mean(l_s_ft))

# --- Plotting ---
print("Generating comprehensive head-to-head comparison graphs...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle("MiDaS Depth Perception: Original vs. Enhanced vs. Fine-Tuned (JPEG Compression)\nProject by: Ayelet & Noa", fontsize=20, fontweight='bold', y=1.02)

c_orig, c_enh, c_ft = '#d62728', '#2ca02c', '#9467bd'

# Graph 1: RMSE
ax1.plot(jpeg_qualities, m_rmse_orig, marker='X', markersize=10, color=c_orig, linewidth=3, label='Original MiDaS (JPEG)')
ax1.plot(jpeg_qualities, m_rmse_enh, marker='o', markersize=10, color=c_enh, linewidth=3, label='Enhanced MiDaS (Gaussian Blur)')
ax1.plot(jpeg_qualities, m_rmse_ft, marker='^', markersize=10, color=c_ft, linewidth=3, label='Fine-Tuned MiDaS (JPEG)')

ax1.set_xlabel(r'JPEG Quality $\rightarrow$ Worse Quality', fontsize=14, fontweight='bold')
ax1.set_ylabel('Mean RMSE (Lower is Better)', fontsize=14, fontweight='bold')
ax1.invert_xaxis()
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(fontsize=12)
ax1.set_title("Mathematical Error (Normalized RMSE)", fontsize=16, fontweight='bold')

for i, q in enumerate(jpeg_qualities):
    ax1.annotate(f"Q:{q}", (jpeg_qualities[i], m_rmse_ft[i]), textcoords="offset points", xytext=(0,-18), ha='center', fontsize=10, fontweight='bold', color=c_ft)

# Graph 2: SSIM
ax2.plot(jpeg_qualities, m_ssim_orig, marker='X', markersize=10, color=c_orig, linewidth=3, label='Original MiDaS (JPEG)')
ax2.plot(jpeg_qualities, m_ssim_enh, marker='o', markersize=10, color=c_enh, linewidth=3, label='Enhanced MiDaS (Gaussian Blur)')
ax2.plot(jpeg_qualities, m_ssim_ft, marker='^', markersize=10, color=c_ft, linewidth=3, label='Fine-Tuned MiDaS (JPEG)')

ax2.set_xlabel(r'JPEG Quality $\rightarrow$ Worse Quality', fontsize=14, fontweight='bold')
ax2.set_ylabel('Mean SSIM (Higher is Better)', fontsize=14, fontweight='bold')
ax2.invert_xaxis()
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(fontsize=12)
ax2.set_title("Structural Similarity Index (SSIM)", fontsize=16, fontweight='bold')

for i, q in enumerate(jpeg_qualities):
    ax2.annotate(f"Q:{q}", (jpeg_qualities[i], m_ssim_ft[i]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=10, fontweight='bold', color=c_ft)

plt.tight_layout()
plt.show()

In [ ]:
#@title Task 3: The Ultimate Evaluation - JPEG Compression (Bar Charts)

import gc
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

print("Initializing Bar Chart evaluation pipeline for JPEG Compression...")

# Prevent GPU memory leaks
torch.cuda.empty_cache()
gc.collect()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

original_midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()

finetuned_midas_jpeg = torch.hub.load("isl-org/MiDaS", "MiDaS_small")
# טוענים את המודל מהאימון החכם האחרון
finetuned_midas_jpeg.load_state_dict(torch.load('/content/midas_finetuned_jpeg_smart.pth'))
finetuned_midas_jpeg.to(device).eval()

midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
transform = midas_transforms.small_transform

# --- Helper Functions ---
def apply_jpeg_compression(img_rgb, quality):
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encimg = cv2.imencode('.jpg', img_bgr, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

def apply_enhancement(distorted_img):
    return cv2.GaussianBlur(distorted_img, (7, 7), 1.5)

def get_normalized_depth(model, img_rgb, roi_h):
    model.eval()
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = model(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

def compute_metrics(depth1, depth2, roi_h):
    roi_d1 = depth1[:roi_h, :]
    roi_d2 = depth2[:roi_h, :]
    rmse_val = np.sqrt(np.mean((roi_d1 - roi_d2) ** 2))
    ssim_val = ssim(roi_d1, roi_d2, data_range=1.0)
    return rmse_val, ssim_val

# --- Execution ---
jpeg_qualities = [30, 20, 10, 5, 2]
m_rmse_orig, m_ssim_orig = [], []
m_rmse_enh, m_ssim_enh = [], []
m_rmse_ft, m_ssim_ft = [], []

print("Extracting Baseline Depth Maps...")
baseline_data = []
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    roi_h = int(img_rgb.shape[0] * 0.8)
    base_depth = get_normalized_depth(original_midas, img_rgb, roi_h)
    baseline_data.append((img_rgb, base_depth, roi_h))

print("Running Head-to-Head Inference...")
for quality in jpeg_qualities:
    l_r_orig, l_s_orig, l_r_enh, l_s_enh, l_r_ft, l_s_ft = [], [], [], [], [], []

    for img_rgb, base_depth, roi_h in baseline_data:
        comp_img = apply_jpeg_compression(img_rgb, quality)
        enhanced_img = apply_enhancement(comp_img)

        depth_orig = get_normalized_depth(original_midas, comp_img, roi_h)
        depth_enh = get_normalized_depth(original_midas, enhanced_img, roi_h)
        depth_ft = get_normalized_depth(finetuned_midas_jpeg, comp_img, roi_h)

        r_orig, s_orig = compute_metrics(base_depth, depth_orig, roi_h)
        r_enh, s_enh = compute_metrics(base_depth, depth_enh, roi_h)
        r_ft, s_ft = compute_metrics(base_depth, depth_ft, roi_h)

        l_r_orig.append(r_orig); l_s_orig.append(s_orig)
        l_r_enh.append(r_enh); l_s_enh.append(s_enh)
        l_r_ft.append(r_ft); l_s_ft.append(s_ft)

    m_rmse_orig.append(np.mean(l_r_orig)); m_ssim_orig.append(np.mean(l_s_orig))
    m_rmse_enh.append(np.mean(l_r_enh)); m_ssim_enh.append(np.mean(l_s_enh))
    m_rmse_ft.append(np.mean(l_r_ft)); m_ssim_ft.append(np.mean(l_s_ft))

# --- Plotting Grouped Bar Charts ---
print("Generating comprehensive head-to-head Bar Charts...")

labels = [f'Q:{q}' for q in jpeg_qualities]
x = np.arange(len(labels))  # The label locations
width = 0.25  # The width of the bars

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle("MiDaS Depth Perception: Original vs. Enhanced vs. Fine-Tuned (JPEG Compression)\nProject by: Ayelet & Noa", fontsize=20, fontweight='bold', y=1.02)

c_orig, c_enh, c_ft = '#d62728', '#2ca02c', '#9467bd'

# --- 1. Normalized RMSE Bar Chart ---
rects1_rmse = ax1.bar(x - width, m_rmse_orig, width, label='Original MiDaS (JPEG)', color=c_orig, edgecolor='black', alpha=0.9)
rects2_rmse = ax1.bar(x, m_rmse_enh, width, label='Enhanced MiDaS (Gaussian Blur)', color=c_enh, edgecolor='black', alpha=0.9)
rects3_rmse = ax1.bar(x + width, m_rmse_ft, width, label='Fine-Tuned MiDaS (JPEG)', color=c_ft, edgecolor='black', alpha=0.9)

ax1.set_ylabel('Mean RMSE (Lower is Better)', fontsize=14, fontweight='bold')
ax1.set_title("Mathematical Error (Normalized RMSE)", fontsize=16, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=12, fontweight='bold')
ax1.legend(fontsize=12, loc='upper left')
ax1.grid(axis='y', linestyle='--', alpha=0.6)

# --- 2. SSIM Bar Chart ---
rects1_ssim = ax2.bar(x - width, m_ssim_orig, width, label='Original MiDaS (JPEG)', color=c_orig, edgecolor='black', alpha=0.9)
rects2_ssim = ax2.bar(x, m_ssim_enh, width, label='Enhanced MiDaS (Gaussian Blur)', color=c_enh, edgecolor='black', alpha=0.9)
rects3_ssim = ax2.bar(x + width, m_ssim_ft, width, label='Fine-Tuned MiDaS (JPEG)', color=c_ft, edgecolor='black', alpha=0.9)

ax2.set_ylabel('Mean SSIM (Higher is Better)', fontsize=14, fontweight='bold')
ax2.set_title("Structural Similarity Index (SSIM)", fontsize=16, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, fontsize=12, fontweight='bold')
ax2.legend(fontsize=12, loc='lower left')
ax2.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
#@title The Grand Finale: Overall Robustness Summary (Gaussian vs. JPEG)

import gc
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

print("Initializing the Ultimate Summary Evaluation Pipeline...")

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Load Original Baseline Model
print("Loading Original MiDaS...")
original_midas = torch.hub.load("isl-org/MiDaS", "MiDaS_small").to(device).eval()

# 2. Load Gaussian Fine-Tuned Model
print("Loading Gaussian Fine-Tuned Model...")
ft_midas_gauss = torch.hub.load("isl-org/MiDaS", "MiDaS_small")
ft_midas_gauss.load_state_dict(torch.load('/content/midas_finetuned_gaussian_rmse.pth'))
ft_midas_gauss.to(device).eval()

# 3. Load JPEG Fine-Tuned Model (The Smart Denoising one)
print("Loading JPEG Fine-Tuned Model...")
ft_midas_jpeg = torch.hub.load("isl-org/MiDaS", "MiDaS_small")
ft_midas_jpeg.load_state_dict(torch.load('/content/midas_finetuned_jpeg_smart.pth'))
ft_midas_jpeg.to(device).eval()

midas_transforms = torch.hub.load("isl-org/MiDaS", "transforms")
transform = midas_transforms.small_transform

# --- Helper Functions ---
def apply_gaussian_noise(img_rgb, sigma):
    noise = np.random.normal(0, sigma, img_rgb.shape)
    noisy_img = img_rgb.astype(np.float32) + noise
    return np.clip(noisy_img, 0, 255).astype(np.uint8)

def apply_jpeg_compression(img_rgb, quality):
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encimg = cv2.imencode('.jpg', img_bgr, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

def apply_enhancement(distorted_img):
    return cv2.GaussianBlur(distorted_img, (7, 7), 1.5)

def get_normalized_depth(model, img_rgb, roi_h):
    model.eval()
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = model(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2], mode="bicubic", align_corners=False
        ).squeeze()

    depth = pred.cpu().numpy()
    roi_depth = depth[:roi_h, :]
    d_min, d_max = roi_depth.min(), roi_depth.max()
    depth_norm = (np.clip(depth, d_min, d_max) - d_min) / (d_max - d_min + 1e-6)
    depth_norm[roi_h:, :] = 0
    return depth_norm

def compute_metrics(depth1, depth2, roi_h):
    roi_d1 = depth1[:roi_h, :]
    roi_d2 = depth2[:roi_h, :]
    rmse_val = np.sqrt(np.mean((roi_d1 - roi_d2) ** 2))
    ssim_val = ssim(roi_d1, roi_d2, data_range=1.0)
    return rmse_val, ssim_val

# --- Evaluation Execution ---
noise_sigmas = [15, 30, 50, 75, 100]
jpeg_qualities = [30, 20, 10, 5, 2]

# Aggregate containers
results = {
    'Gaussian': {'rmse_orig': [], 'ssim_orig': [], 'rmse_enh': [], 'ssim_enh': [], 'rmse_ft': [], 'ssim_ft': []},
    'JPEG': {'rmse_orig': [], 'ssim_orig': [], 'rmse_enh': [], 'ssim_enh': [], 'rmse_ft': [], 'ssim_ft': []}
}

print("Running Global Inference (This may take a minute)...")
for img_path, _ in selected_data:
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    roi_h = int(img_rgb.shape[0] * 0.8)
    base_depth = get_normalized_depth(original_midas, img_rgb, roi_h)

    # 1. Evaluate Gaussian Noise
    for sigma in noise_sigmas:
        dist_img = apply_gaussian_noise(img_rgb, sigma)
        enh_img = apply_enhancement(dist_img)

        d_orig = get_normalized_depth(original_midas, dist_img, roi_h)
        d_enh = get_normalized_depth(original_midas, enh_img, roi_h)
        d_ft = get_normalized_depth(ft_midas_gauss, dist_img, roi_h)

        ro, so = compute_metrics(base_depth, d_orig, roi_h)
        re, se = compute_metrics(base_depth, d_enh, roi_h)
        rf, sf = compute_metrics(base_depth, d_ft, roi_h)

        results['Gaussian']['rmse_orig'].append(ro); results['Gaussian']['ssim_orig'].append(so)
        results['Gaussian']['rmse_enh'].append(re); results['Gaussian']['ssim_enh'].append(se)
        results['Gaussian']['rmse_ft'].append(rf); results['Gaussian']['ssim_ft'].append(sf)

    # 2. Evaluate JPEG Compression
    for quality in jpeg_qualities:
        dist_img = apply_jpeg_compression(img_rgb, quality)
        enh_img = apply_enhancement(dist_img)

        d_orig = get_normalized_depth(original_midas, dist_img, roi_h)
        d_enh = get_normalized_depth(original_midas, enh_img, roi_h)
        d_ft = get_normalized_depth(ft_midas_jpeg, dist_img, roi_h)

        ro, so = compute_metrics(base_depth, d_orig, roi_h)
        re, se = compute_metrics(base_depth, d_enh, roi_h)
        rf, sf = compute_metrics(base_depth, d_ft, roi_h)

        results['JPEG']['rmse_orig'].append(ro); results['JPEG']['ssim_orig'].append(so)
        results['JPEG']['rmse_enh'].append(re); results['JPEG']['ssim_enh'].append(se)
        results['JPEG']['rmse_ft'].append(rf); results['JPEG']['ssim_ft'].append(sf)

# --- Compute Grand Means ---
categories = ['Gaussian Noise\n(All Severities)', 'JPEG Compression\n(All Severities)']
x = np.arange(len(categories))
width = 0.25

mean_rmse_orig = [np.mean(results['Gaussian']['rmse_orig']), np.mean(results['JPEG']['rmse_orig'])]
mean_rmse_enh = [np.mean(results['Gaussian']['rmse_enh']), np.mean(results['JPEG']['rmse_enh'])]
mean_rmse_ft = [np.mean(results['Gaussian']['rmse_ft']), np.mean(results['JPEG']['rmse_ft'])]

mean_ssim_orig = [np.mean(results['Gaussian']['ssim_orig']), np.mean(results['JPEG']['ssim_orig'])]
mean_ssim_enh = [np.mean(results['Gaussian']['ssim_enh']), np.mean(results['JPEG']['ssim_enh'])]
mean_ssim_ft = [np.mean(results['Gaussian']['ssim_ft']), np.mean(results['JPEG']['ssim_ft'])]

# --- Plotting ---
print("Generating Final Summary Bar Charts...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle("Overall Model Robustness Across Degradation Types\nProject by: Ayelet & Noa", fontsize=22, fontweight='bold', y=1.05)

c_orig, c_enh, c_ft = '#d62728', '#2ca02c', '#9467bd'

# --- 1. RMSE Subplot ---
rects1 = ax1.bar(x - width, mean_rmse_orig, width, label='Original MiDaS', color=c_orig, edgecolor='black')
rects2 = ax1.bar(x, mean_rmse_enh, width, label='Enhanced (Gaussian Blur)', color=c_enh, edgecolor='black')
rects3 = ax1.bar(x + width, mean_rmse_ft, width, label='Fine-Tuned (Task-Specific)', color=c_ft, edgecolor='black')

ax1.set_ylabel('Grand Mean RMSE (Lower is Better)', fontsize=14, fontweight='bold')
ax1.set_title("Overall Mathematical Error (RMSE)", fontsize=16, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(categories, fontsize=14, fontweight='bold')
ax1.legend(fontsize=12, loc='upper right')
ax1.grid(axis='y', linestyle='--', alpha=0.6)

# Add values on top of bars
for rects in [rects1, rects2, rects3]:
    for rect in rects:
        height = rect.get_height()
        ax1.annotate(f'{height:.3f}', xy=(rect.get_x() + rect.get_width() / 2, height),
                     xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=11, fontweight='bold')

# --- 2. SSIM Subplot ---
rects4 = ax2.bar(x - width, mean_ssim_orig, width, label='Original MiDaS', color=c_orig, edgecolor='black')
rects5 = ax2.bar(x, mean_ssim_enh, width, label='Enhanced (Gaussian Blur)', color=c_enh, edgecolor='black')
rects6 = ax2.bar(x + width, mean_ssim_ft, width, label='Fine-Tuned (Task-Specific)', color=c_ft, edgecolor='black')

ax2.set_ylabel('Grand Mean SSIM (Higher is Better)', fontsize=14, fontweight='bold')
ax2.set_title("Overall Structural Similarity (SSIM)", fontsize=16, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(categories, fontsize=14, fontweight='bold')
ax2.legend(fontsize=12, loc='lower right')
ax2.grid(axis='y', linestyle='--', alpha=0.6)

# Add values on top of bars
for rects in [rects4, rects5, rects6]:
    for rect in rects:
        height = rect.get_height()
        ax2.annotate(f'{height:.3f}', xy=(rect.get_x() + rect.get_width() / 2, height),
                     xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()